# Dissecting Pre-MZT Cis-Regulatory Grammar via Cross-Species Sequence-to-Function Transfer Learning

[![Python 3.12](https://img.shields.io/badge/python-3.12-blue.svg)](https://www.python.org/downloads/release/python-3120/)
[![PyTorch CUDA](https://img.shields.io/badge/PyTorch-CUDA-ee4c2c.svg)](https://pytorch.org/)
[![Conda](https://img.shields.io/badge/conda-nucleotide-green.svg)](https://docs.conda.io/)

---

### Abstract & Workflow Overview
During early embryonic development, zygotic transcription initiates in two phases: an initial **minor wave (pre-MZT, Mid-Blastula Transition)** followed by the genome-wide **major wave (MZT)**. Two technical barriers complicate minor-wave discovery from standard oligo(dT) RNA-seq alone:
1. **Cytoplasmic polyadenylation:** Maternal transcripts stored in the egg often receive poly(A) tail elongation after fertilization, appearing "upregulated" in oligo(dT) RNA-seq without genuine *de novo* transcription in the nucleus.
2. **Low chromatin input:** Generating high-resolution chromatin accessibility maps from tiny early-cleavage embryos is technically difficult in many non-model species.

This gene- and species-agnostic pipeline addresses both limitations by pairing **4sU nascent-RNA timing filters** (SLAM-seq in a reference embryo) with **cross-species sequence-to-function deep learning** trained solely on public reference embryos and applied **zero-shot** (purely from raw DNA sequence) to any target genome configured in **Block 0**.

### Key Quantitative & Machine Learning Concepts (For Biologists)
* **Leak-Free Chromosome Holdout (~64% Train / 12% Valid / 24% Test):** Whole assembled chromosomes (including the chromosome carrying the reference ortholog of your target locus) are withheld across **both** ATAC-seq and SLAM-seq training so every benchmark tests true generalization across unseen genomic DNA rather than memorized sequences. Target-species RNA-seq is never seen during model training.
* **ChromBPNet (`2,114 bp` CNN) & True Bias Factorization (`nobias`):** A base-resolution convolutional neural network split into two subnetworks. First, a **Tn5 bias subnetwork** is trained on closed background DNA (`nonpeaks`) to learn sequence cutting biases of the Tn5 transposase enzyme. Next, the bias subnetwork is frozen and summed with a trainable transcription-factor (`TF`) network during peak training (`full = tf_model + frozen_bias`); because the frozen bias branch absorbs Tn5 texture, backpropagation forces the **`nobias`** network to learn pure regulatory DNA sequence grammar (verified when residual **Max Tn5 Footprint Response** $< 0.003$ and **Profile Median Jensen–Shannon Divergence [`JSD`]** separates bias texture from true TF footprints).
* **Stage 2 Ridge Regression (`RidgeCV`) & Openness Gate:** Because late-blastula ATAC-seq has much higher signal-to-noise than early-blastula ATAC-seq, we freeze the debiased late-stage CNN encoder, extract sequence embeddings over high-confidence summits (`FE >= 4`, `-log10(q) >= 5`), and fit L2-penalized linear regression probes (`RidgeCV`) instead of full CNN retraining to avoid overfitting sparser early reads. Two accessibility outputs are predicted:
  * **$y_{\text{abs}} = \log_2(\text{early CPM} + 1)$:** Predicted absolute pre-MZT chromatin openness.
  * **$y_{\text{rel}}$ (aliased as `y_resid`):** $\log_2\big((\text{early CPM}+1)/(\text{late CPM}+1)\big)$, measuring whether chromatin opens *earlier* than expected from late blastula. **`y_resid_gated`** masks windows falling below the calibrated **Openness Gate (`late` threshold)** so noisy near-zero ratios in closed heterochromatin never generate false-positive timing calls.
* **`NTv3` (`65,536 bp` Foundation Transformer), `LoRA` & `frozen_head`:** Adapted from a ~650M-parameter DNA transformer over 65 kb inputs (predicting the central 24.6 kb / 37.5% crop across 10 joint early/late ATAC and strand-specific nascent SLAM-seq tracks):
  * **`NTv3_raw_pretrained`:** Out-of-the-box pretrained adult/differentiated tracks prior to any blastula head training.
  * **`NTv3_frozen_head`:** Freezes the 650M-parameter transformer trunk and trains only a small linear readout head—revealing how much generic promoter/CpG grammar exists in pretrained features vs. what needs embryonic adaptation.
  * **`NTv3_multitask_lora` (`LoRA` = Low-Rank Adaptation):** Freezes main trunk matrices and optimizes lightweight low-rank adapter matrices inside attention/embedding layers plus the 10-track head using a track-balanced loss (`3×` weight on pre-MZT stages) within single-GPU VRAM, storing training hyperparameters in `ntv3_train_config.json`.
* **Attribution, Perturbation & Parity Metrics:**
  * **Motif Saliency ($|\nabla_{\text{seq}}|$):** Backpropagation derivatives scoring how strongly every single DNA base (`A/C/G/T`) drives predicted accessibility or nascent transcription; sharp clusters reveal regulatory TF binding sites (*seqlets*).
  * **In Silico Ablation & Evolutionary Swap ($\Delta$):** Mutates, dinucleotide-shuffles, or CpG-ablates (`CG -> TG`) a motif or swaps it with the orthologous reference sequence *in silico*, quantifying exact predicted accessibility/SLAM changes ($\Delta$).
  * **Gene Parity & MAD Robust $z$-Score:** Evaluates whether target genes match the predicted activity range (**`Q25–Q75` interquartile parity**) of *bona fide* strictly zygotic minor-wave genes, calibrated against negative controls via `robust_z = (score - median) / (1.4826 * MAD)` (median absolute deviation, resistant to extreme CpG/housekeeping outliers).

In [ ]:
# ==============================================================================
# Emergency Task Termination Cell (Self-Contained)
# ==============================================================================
# Set TERMINATE_ALL_TASKS = True and run this cell to terminate all running background tasks.
# Guarded with False by default so accidental 'Run All' never terminates active jobs.
TERMINATE_ALL_TASKS = False

import os
import sys
from pathlib import Path
import pandas as pd

if "." not in sys.path:
    sys.path.insert(0, ".")

try:
    from scripts.task_manager import TaskManager

    # Collect all candidate work directories
    cand_paths = []
    if "WORK_DIR" in globals() and globals()["WORK_DIR"]:
        cand_paths.append(Path(globals()["WORK_DIR"]))

    globbed = sorted(p for p in Path(".").glob("work_*") if p.is_dir() and (p / "tasks").exists())
    cand_paths.extend(globbed)

    # Deduplicate while preserving order using .resolve()
    seen = set()
    work_dirs = []
    for p in cand_paths:
        r = p.resolve()
        if r not in seen and p.name != "tasks":
            seen.add(r)
            work_dirs.append(p)

    if not work_dirs:
        work_dirs = [Path("./work_astatotilapia_premzt")]

    print(f"[*] Inspecting work directories: {[str(d) for d in work_dirs]}")

    status_dfs = []
    for wd in work_dirs:
        print(f"[*] Inspecting work_dir: {wd}")
        tm = TaskManager(work_dir=str(wd))
        if TERMINATE_ALL_TASKS:
            print(f"[!] TERMINATE_ALL_TASKS is True: Terminating all active background tasks in {wd}...")
            cancel_df = tm.cancel_all()
            if cancel_df is not None and not cancel_df.empty:
                print(f"[+] Termination summary for {wd}:")
                try:
                    display(cancel_df)
                except NameError:
                    print(cancel_df)
        st_df = tm.status()
        if st_df is not None and not st_df.empty:
            st_df["work_dir"] = str(wd)
            status_dfs.append(st_df)

    if status_dfs:
        combined_df = pd.concat(status_dfs, ignore_index=True)
        print("[*] Combined TaskManager status:")
        try:
            display(combined_df)
        except NameError:
            print(combined_df)
    else:
        print("[*] No tasks found in inspected work directories.")
except Exception as e:
    print(f"[-] TaskManager check notice: {e}")

---
## Block 0: Study Configuration, Compute Profile & Task Manager

All study-specific parameters (`TARGET_SPECIES_NAME`, `TARGET_LOCUS_NAME`, `TARGET_GENES`, `CANONICAL_ZGA_MARKERS`, reference genome/GTF paths, sequencing accessions, TF expression filters, and `COMPUTE_PROFILE`) are configured strictly here and exported to `WORK_DIR/config.json`.
* Downstream scripts read `WORK_DIR/config.json` dynamically so analyzing a different species or locus requires **modifying Block 0 only**.
* Switch `COMPUTE_PROFILE` between `"laptop_16gb"` (65 kb `NTv3` LoRA, single GPU ~7.5 GB VRAM) and `"hpc_1mb"` (1 Mb multi-GPU full fine-tuning via `slurm`).
* Execute the resource audit/downloader below to stage reference genomes, GTF annotations, JASPAR 2024 vertebrate PWMs, and Ensembl 1:1 ortholog mappings before model blocks.

In [ ]:
%load_ext autoreload
%autoreload 1
%aimport scripts.task_manager, scripts.infer_and_attribute, scripts.download_resources

import os
import sys
import json
import time
import shutil
import subprocess
from pathlib import Path
bin_dir = str(Path(sys.prefix) / "bin")
if bin_dir not in os.environ.get("PATH", ""):
    os.environ["PATH"] = f"{bin_dir}:{os.environ.get('PATH', '')}"
import pandas as pd
import numpy as np
import torch
import pysam
from dotenv import load_dotenv
load_dotenv()
assert "HF_TOKEN" in os.environ, "HF_TOKEN not found! You need to request access from https://huggingface.co/InstaDeepAI/NTv3_tracks and set your HF_TOKEN in a .env file."

# ==============================================================================
# 1. Study & Target Locus Settings
# ==============================================================================
STUDY_NAME = "astatotilapia_premzt"
TARGET_SPECIES_NAME = "Astatotilapia_calliptera"
TARGET_LOCUS_NAME = "notch2_slc35a3a"
TARGET_GENES = ["notch2", "slc35a3a"]
LOCUS_FLANK_BP = 75000
CANONICAL_ZGA_MARKERS = ["vox", "klf17", "mxtx2"]


# Multi-species RNA-seq TPM RDS input (optional; auto-detected if path ends in .rds or file exists)
RNASEQ_TPM_RDS = "./RNA-seq/TPM_2C_1K_both_species.rds"

# Target RNA-seq BAMs (relative paths inside pwd)
# RNA-seq BAM inputs for study species pre-MZT analysis:
# RNASEQ_EARLY_BAMS: Stage before zygotic genome activation (e.g., 2-cell or unfertilized egg), representing maternal RNA baseline.
# RNASEQ_PREMZT_BAMS: Pre-MZT stage where minor-wave transcription initiates (e.g., 1000-cell or sphere), capturing earliest zygotic output.
RNASEQ_EARLY_BAMS = ["./RNA-seq/AC_2C_rep1.markdup.sorted.bam"]      # 2-cell maternal baseline
RNASEQ_PREMZT_BAMS = ["./RNA-seq/AC_1K_rep1.markdup.sorted.bam"]    # 1000-cell pre-MZT stage

# Auto-detect RNA-seq input mode: .rds (multi-species TPM) vs .bam
if RNASEQ_TPM_RDS and Path(RNASEQ_TPM_RDS).exists():
    RNA_MODE = "rds"
elif RNASEQ_EARLY_BAMS and str(RNASEQ_EARLY_BAMS[0]).endswith(".rds"):
    RNA_MODE = "rds"
    RNASEQ_TPM_RDS = RNASEQ_EARLY_BAMS[0]
else:
    RNA_MODE = "bam"
print(f"[+] RNA-seq input mode: {RNA_MODE.upper()}" + (f" ({RNASEQ_TPM_RDS})" if RNA_MODE == "rds" else " (BAM mode)"))

# Target genome and GTF (set to local relative path, or None to auto-download)
TARGET_GENOME_FA = f"./work_{STUDY_NAME}/downloads/Astatotilapia_calliptera.fAstCal1.3.dna_sm.toplevel.fa"
TARGET_GTF = f"./work_{STUDY_NAME}/downloads/Astatotilapia_calliptera.fAstCal1.3.112.gtf.gz"

# Zebrafish and Medaka reference genomes & JASPAR motif database (set to path or None to auto-download)
ZEBRAFISH_GENOME_FA = None
ZEBRAFISH_GTF = None
MEDAKA_GENOME_FA = None
MEDAKA_GTF = None
JASPAR_MEME_PATH = None

# Pre-MZT TF expression filter
PREMZT_TF_LIST_PATH = "./TF_gene_names.txt"

# SLAM-seq & Ortholog Mapping options
SLAMSEQ_GENE_TABLE_PATH = None  # None -> auto-generated at WORK_DIR/slamseq/zf_slamseq_gene_classes.tsv
ORTHOLOG_MAP_PATH = None        # None -> auto-generated at WORK_DIR/downloads/orthologs_target_to_zf.tsv
SLAM_EXCLUDE_MITO = True
SLAM_MASK_BASELINE_SNPS = True
SLAM_MIN_BASELINE_FC = 3.0
SLAM_MIN_KINETICS_RETENTION = 0.5
SLAM_MAX_BLEED_RATIO = 0.005

# ==============================================================================
# 2. Epigenomic & Transcriptomic Dataset Accessions
# ==============================================================================
# Zebrafish ATAC-seq (GSE130944, Palfy et al. 2020)
ZEBRAFISH_ATAC_ACCESSIONS = [
    "GSM3756599", "GSM3756600",  # 256-cell early blastula replicates
    "GSM3756603", "GSM3756604", "GSM3756605",  # dome late blastula replicates
    "SRR9032640"  # naked genomic DNA Tn5 control
]

# Medaka ATAC-seq (PRJDB7492, Nakamura et al. 2021)
MEDAKA_ATAC_ACCESSIONS = [
    "DRX143405", "DRX143406", "DRX143407",  # st9 early blastula replicates
    "DRX143412", "DRX143413"               # st11 late blastula replicates
]

# Zebrafish Nascent SLAM-seq (PRJNA768450, Bhat et al. 2023)
SLAM_ACCESSIONS = [
    "GSM5610039", "GSM5610040",  # 0.75 hpf (+4sU) baseline
    "GSM5610043", "GSM5610044",  # 2.5 hpf (+4sU) pre-MZT
    "GSM5610045", "GSM5610046", "GSM5610047", "GSM5610048",  # 3.0-3.5 hpf (+4sU) onset
    "GSM5610051", "GSM5610052",  # 4.5 hpf (+4sU) major wave
    "GSM5610075", "GSM5610076", "GSM5610077"   # -4sU untreated controls
]

USE_PRETRAINED_BIAS = False
BIAS_MODEL_PATH = None

# ==============================================================================
# 3. Environment, Credentials & Compute Profile
# ==============================================================================
CONDA_ENV_NAME = "nucleotide"
COMPUTE_PROFILE = "laptop_16gb"  # Options: "laptop_16gb" (default) or "hpc_1mb"

# Hugging Face token (redacted from logs/saved JSON; sets HF_HOME inside WORK_DIR)
HF_TOKEN = os.environ.get("HF_TOKEN", "")

# Compute & execution parameters
WORK_DIR = f"./work_{STUDY_NAME}"
THREADS = 8
FORCE_RERUN = False
CV_FOLDS = 5
ENABLE_MEDAKA = True
MEDAKA_PASSED_QC = False
_qc_status_path = Path(WORK_DIR) / "models" / "medaka" / "qc_status.json"
if _qc_status_path.exists():
    try:
        with open(_qc_status_path) as _f_qc:
            MEDAKA_PASSED_QC = bool(json.load(_f_qc).get("MEDAKA_PASSED_QC", False))
    except Exception:
        MEDAKA_PASSED_QC = False

# Empirical Openness Gate threshold (calibrated from held-out Zebrafish native Poisson dome peaks)
OPEN_GATE_LATE_THRESH = 5.70
_og_path = Path(WORK_DIR) / "models" / "zebrafish" / "openness_gate.json"
if _og_path.exists():
    try:
        with open(_og_path) as _f_og:
            OPEN_GATE_LATE_THRESH = float(json.load(_f_og).get("OPEN_GATE_LATE_THRESH", 5.70))
    except Exception:
        OPEN_GATE_LATE_THRESH = 5.70
LATE_OPEN_THRESHOLD = OPEN_GATE_LATE_THRESH
STAGE2_FILTER_HIGHCONF_PEAKS = True
STAGE2_MIN_FE = 4.0
STAGE2_MIN_NEG_LOG10_Q = 5.0
ENSEMBLE_MEDAKA_WITH_ZF = False
EXCLUDE_HOUSEKEEPING_PROMOTERS_STAGE2 = False
NTV3_RUN_BASELINES = True
ENABLE_OPTION2_TARGET_RNA_ABLATION = False

# Epigenomics & RNA-seq Quantification Settings
CHROMBPNET_EPOCHS = 50
RNASEQ_STRANDNESS = 0  # 0: unstranded, 1: stranded, 2: reversely stranded

# Profile presets
if COMPUTE_PROFILE == "laptop_16gb":
    NTV3_MODEL_ID = "InstaDeepAI/NTv3_tracks"
    NTV3_CONTEXT_LEN = 65536
    NTV3_TRAIN_MODE = "lora"
    NTV3_PRECISION = "bf16"
    NTV3_GRAD_CHECKPOINTING = True
    NTV3_BATCH_SIZE = 1
    NTV3_GRAD_ACCUM = 16
    NUM_GPUS = 1
    EXECUTION_BACKEND = "local_background"
    slurm_config = None
elif COMPUTE_PROFILE == "hpc_1mb":
    NTV3_MODEL_ID = "InstaDeepAI/NTv3_tracks_1mb"
    NTV3_CONTEXT_LEN = 1048576
    NTV3_TRAIN_MODE = "full"
    NTV3_PRECISION = "bf16"
    NTV3_GRAD_CHECKPOINTING = True
    NTV3_BATCH_SIZE = 4
    NTV3_GRAD_ACCUM = 4
    NUM_GPUS = 4
    EXECUTION_BACKEND = "slurm"
    slurm_config = {
        "partition": "gpu",
        "gres": f"gpu:{NUM_GPUS}",
        "cpus_per_task": THREADS,
        "mem": "64G",
        "time": "24:00:00"
    }
else:
    raise ValueError(f"Unknown COMPUTE_PROFILE: {COMPUTE_PROFILE}")

# Ensure HF_HOME is strictly inside WORK_DIR
os.environ["HF_HOME"] = os.path.abspath(f"{WORK_DIR}/hf_cache")

# ==============================================================================
# 4. Reference Resources (Auto-downloaded if not present)
# ==============================================================================
# Safely resolve None defaults for all reference paths immediately
if TARGET_GENOME_FA is None:
    TARGET_GENOME_FA = f"{WORK_DIR}/downloads/{TARGET_SPECIES_NAME}.dna_sm.toplevel.fa"
if TARGET_GTF is None:
    TARGET_GTF = f"{WORK_DIR}/downloads/{TARGET_SPECIES_NAME}.gtf.gz"
if ZEBRAFISH_GENOME_FA is None:
    ZEBRAFISH_GENOME_FA = f"{WORK_DIR}/downloads/Danio_rerio.GRCz11.dna_sm.primary_assembly.fa"
if ZEBRAFISH_GTF is None:
    ZEBRAFISH_GTF = f"{WORK_DIR}/downloads/Danio_rerio.GRCz11.112.gtf.gz"
if MEDAKA_GENOME_FA is None:
    MEDAKA_GENOME_FA = f"{WORK_DIR}/downloads/Oryzias_latipes.ASM223467v1.dna_sm.toplevel.fa"
if MEDAKA_GTF is None:
    MEDAKA_GTF = f"{WORK_DIR}/downloads/Oryzias_latipes.ASM223467v1.112.gtf.gz"
if JASPAR_MEME_PATH is None:
    JASPAR_MEME_PATH = f"{WORK_DIR}/downloads/JASPAR2024_CORE_vertebrates_non-redundant.meme"

# Define cross-validation folds and canonical downstream paths in Block 0 to ensure fresh-kernel safety
FOLDS_TO_TRAIN = [0] if COMPUTE_PROFILE == "laptop_16gb" else list(range(CV_FOLDS))
filtered_meme_path = f"{WORK_DIR}/downloads/premzt_tfs.meme"
ortholog_tsv = f"{WORK_DIR}/downloads/orthologs_target_to_zf.tsv"

# Create directory hierarchy strictly inside pwd
for d in ["tasks", "downloads", "atac", "slamseq", "rnaseq", "models", "results", "hf_cache"]:
    Path(f"{WORK_DIR}/{d}").mkdir(parents=True, exist_ok=True)

# Define and create canonical results directory for target locus to prevent NameError
res_dir = Path(WORK_DIR) / "results" / TARGET_LOCUS_NAME
res_dir.mkdir(parents=True, exist_ok=True)

# Redact token before serializing config snapshot
config_snapshot = {
    "STUDY_NAME": STUDY_NAME,
    "TARGET_SPECIES_NAME": TARGET_SPECIES_NAME,
    "TARGET_LOCUS_NAME": TARGET_LOCUS_NAME,
    "TARGET_GENES": TARGET_GENES,
    "TARGET_GENOME_FA": TARGET_GENOME_FA,
    "TARGET_GTF": TARGET_GTF,
    "ZEBRAFISH_GENOME_FA": ZEBRAFISH_GENOME_FA,
    "ZEBRAFISH_GTF": ZEBRAFISH_GTF,
    "MEDAKA_GENOME_FA": MEDAKA_GENOME_FA,
    "MEDAKA_GTF": MEDAKA_GTF,
    "JASPAR_MEME_PATH": JASPAR_MEME_PATH,
    "PREMZT_TF_LIST_PATH": PREMZT_TF_LIST_PATH,
    "SLAMSEQ_GENE_TABLE_PATH": SLAMSEQ_GENE_TABLE_PATH,
    "ORTHOLOG_MAP_PATH": ORTHOLOG_MAP_PATH,
    "ZEBRAFISH_ATAC_ACCESSIONS": ZEBRAFISH_ATAC_ACCESSIONS,
    "MEDAKA_ATAC_ACCESSIONS": MEDAKA_ATAC_ACCESSIONS,
    "SLAM_ACCESSIONS": SLAM_ACCESSIONS,
    "USE_PRETRAINED_BIAS": USE_PRETRAINED_BIAS,
    "BIAS_MODEL_PATH": BIAS_MODEL_PATH,
    "COMPUTE_PROFILE": COMPUTE_PROFILE,
    "CONDA_ENV_NAME": CONDA_ENV_NAME,
    "NTV3_MODEL_ID": NTV3_MODEL_ID,
    "NTV3_CONTEXT_LEN": NTV3_CONTEXT_LEN,
    "NTV3_TRAIN_MODE": NTV3_TRAIN_MODE,
    "NTV3_BATCH_SIZE": NTV3_BATCH_SIZE,
    "NTV3_GRAD_ACCUM": NTV3_GRAD_ACCUM,
    "NUM_GPUS": NUM_GPUS,
    "EXECUTION_BACKEND": EXECUTION_BACKEND,
    "CHROMBPNET_EPOCHS": CHROMBPNET_EPOCHS,
    "RNASEQ_STRANDNESS": RNASEQ_STRANDNESS,
    "CV_FOLDS": CV_FOLDS,
    "FOLDS_TO_TRAIN": FOLDS_TO_TRAIN,
    "RNASEQ_TPM_RDS": RNASEQ_TPM_RDS,
    "RNA_MODE": RNA_MODE,
    "ENABLE_MEDAKA": ENABLE_MEDAKA,
    "STAGE2_FILTER_HIGHCONF_PEAKS": STAGE2_FILTER_HIGHCONF_PEAKS,
    "STAGE2_MIN_FE": STAGE2_MIN_FE,
    "STAGE2_MIN_NEG_LOG10_Q": STAGE2_MIN_NEG_LOG10_Q,
    "ENSEMBLE_MEDAKA_WITH_ZF": ENSEMBLE_MEDAKA_WITH_ZF,
    "NTV3_RUN_BASELINES": NTV3_RUN_BASELINES,
    "ENABLE_OPTION2_TARGET_RNA_ABLATION": ENABLE_OPTION2_TARGET_RNA_ABLATION,
    "SLAM_EXCLUDE_MITO": SLAM_EXCLUDE_MITO,
    "SLAM_MASK_BASELINE_SNPS": SLAM_MASK_BASELINE_SNPS,
    "SLAM_MIN_BASELINE_FC": SLAM_MIN_BASELINE_FC,
    "SLAM_MIN_KINETICS_RETENTION": SLAM_MIN_KINETICS_RETENTION,
    "SLAM_MAX_BLEED_RATIO": SLAM_MAX_BLEED_RATIO,
}
with open(f"{WORK_DIR}/config.json", "w") as f:
    json.dump(config_snapshot, f, indent=2)

# Initialize TaskManager (passing slurm_config when HPC profile selected)
from scripts.task_manager import TaskManager
tm = TaskManager(work_dir=WORK_DIR, backend=EXECUTION_BACKEND, slurm_config=slurm_config)

print(f"[+] Study '{STUDY_NAME}' configured successfully.")
print(f"    Target Species:  {TARGET_SPECIES_NAME}")
print(f"    Target Locus:    {TARGET_LOCUS_NAME} ({TARGET_GENES})")
print(f"    Results Dir:     {res_dir}")
print(f"    Compute Profile: {COMPUTE_PROFILE} (Backend: {EXECUTION_BACKEND})")
print(f"    HF Cache:        {os.environ['HF_HOME']}")
print(f"    ChromBPNet Ep:   {CHROMBPNET_EPOCHS} | RNA-seq Strandness: {RNASEQ_STRANDNESS}")
display(tm.status())

# ==============================================================================
# Target Genome Assembly & BAM @SQ Contig Length Consistency Check
# ==============================================================================
if TARGET_GENOME_FA and Path(TARGET_GENOME_FA).exists():
    fai_path = Path(f"{TARGET_GENOME_FA}.fai")
    if not fai_path.exists():
        run_cmd = f"samtools faidx '{TARGET_GENOME_FA}'"
        subprocess.run(run_cmd, shell=True, check=True)
    fai_lens = {}
    with open(fai_path) as f:
        for line in f:
            p = line.strip().split("\t")
            if len(p) >= 2:
                fai_lens[p[0]] = int(p[1])

    all_target_bams = [b for b in RNASEQ_EARLY_BAMS + RNASEQ_PREMZT_BAMS if Path(b).exists()]
    for bam_p in all_target_bams:
        with pysam.AlignmentFile(bam_p, "rb") as bf:
            sq_headers = bf.header.get("SQ", [])
            for sq in sq_headers:
                cname = sq.get("SN")
                clen = sq.get("LN")
                if cname in fai_lens:
                    assert clen == fai_lens[cname], (
                        f"[FATAL] Assembly mismatch! BAM {Path(bam_p).name} contig '{cname}' length {clen} "
                        f"does not match TARGET_GENOME_FA contig length {fai_lens[cname]}!"
                    )
    print(f"[+] Verified BAM @SQ contig lengths strictly match {Path(TARGET_GENOME_FA).name}.")

In [ ]:
# ==============================================================================
# Consolidated Resource Download, Audit & Background Queue Block
# ==============================================================================
# Audits required study resources based strictly on Block 0 settings input.
# - Fast pre-download integrity checks: skips download if valid file already exists.
# - If downloading in background: re-running this cell prints current progress once.
# - If downloads are pending: submits background task (max 3 distinct host websites in parallel).
# - Includes lightweight header/chromosome consistency check across FASTA, GTF, and BAMs.
from scripts.download_resources import check_resource_status, download_all_resources

# Reference paths for downstream analysis blocks
filtered_meme_path = f"{WORK_DIR}/downloads/premzt_tfs.meme"
ortholog_tsv = f"{WORK_DIR}/downloads/orthologs_target_to_zf.tsv"
df_orthologs = pd.read_csv(ortholog_tsv, sep="\t") if Path(ortholog_tsv).exists() else None

# 1. Check if download_resources is ALREADY RUNNING FIRST
task_info = tm.get_task_info("download_resources")
task_status = task_info.get("status")

if task_status == "RUNNING":
    elapsed = task_info.get("elapsed_sec", 0)
    prog = task_info.get("progress", "-")
    print(f"[*] Task 'download_resources' is currently RUNNING in background (Elapsed: {elapsed:.1f}s)")
    print(f"    Current Progress: {prog}")
    prog_file = Path(WORK_DIR) / "tasks" / "download_resources" / "progress.txt"
    if prog_file.exists():
        try:
            with open(prog_file) as pf:
                print(f"    Progress details: {pf.read().strip()}")
        except Exception:
            pass
    print("\n--- TaskManager Live Tail (last 15 lines) ---")
    tm.tail("download_resources", n=15)
    print("\n--- TaskManager Status Table ---")
    display(tm.status())
    print("\n[!] Skipping resource audit while download task is actively writing files.")
    print("    Tip: Re-run this cell to refresh live progress, or execute tm.wait('download_resources') to wait for completion.")
else:
    # 2. Lightweight header/chromosome consistency check
    print("[*] Performing lightweight reference and BAM consistency audit...")
    try:
        fa_p = Path(TARGET_GENOME_FA)
        fa_chroms = set()
        if fa_p.exists():
            fai = Path(str(fa_p) + ".fai")
            if not fai.exists():
                subprocess.run(["samtools", "faidx", str(fa_p)], check=False)
            if fai.exists():
                with open(fai) as f:
                    fa_chroms = set(line.split("\t")[0] for line in f)

        # Sample first ~5,000 non-comment lines of GTF
        gtf_p = Path(TARGET_GTF)
        gtf_chroms = set()
        if gtf_p.exists():
            import gzip
            open_fn = gzip.open if str(gtf_p).endswith(".gz") else open
            with open_fn(gtf_p, "rt", errors="replace") as gf:
                lines_checked = 0
                for line in gf:
                    if line.startswith("#"):
                        continue
                    parts = line.split("\t")
                    if len(parts) > 0:
                        gtf_chroms.add(parts[0])
                    lines_checked += 1
                    if lines_checked >= 5000:
                        break

        bam_chroms = set()
        if RNASEQ_EARLY_BAMS and Path(RNASEQ_EARLY_BAMS[0]).exists():
            with pysam.AlignmentFile(RNASEQ_EARLY_BAMS[0], "rb") as bf:
                bam_chroms = set(bf.references)

        # Genuine consistency checks
        if fa_chroms and gtf_chroms:
            fa_gtf_overlap = fa_chroms.intersection(gtf_chroms)
            if not fa_gtf_overlap:
                print(f"  [!] WARNING: Zero chromosome overlap between FASTA ({len(fa_chroms)} contigs) and GTF ({len(gtf_chroms)} sampled contigs)!")
            else:
                print(f"  [+] FASTA and GTF chromosome consistency verified ({len(fa_gtf_overlap)} shared contigs).")

        if fa_chroms and bam_chroms:
            overlap = fa_chroms.intersection(bam_chroms)
            if not overlap:
                print(f"  [!] WARNING: Zero overlap between FASTA headers ({len(fa_chroms)} contigs) and BAM references ({len(bam_chroms)} contigs)!")
            else:
                print(f"  [+] Reference chromosomes verified: {len(overlap)} overlapping contigs between FASTA and BAM.")
    except Exception as e:
        print(f"  [-] Notice during reference audit: {e}")

    # 3. Audit status of required resources
    manifest_df, all_ready, pending_tasks = check_resource_status(
        work_dir=Path(WORK_DIR),
        target_species=TARGET_SPECIES_NAME,
        target_fa=TARGET_GENOME_FA,
        target_gtf=TARGET_GTF,
        zf_fa=ZEBRAFISH_GENOME_FA,
        zf_gtf=ZEBRAFISH_GTF,
        enable_medaka=ENABLE_MEDAKA,
        medaka_fa=MEDAKA_GENOME_FA,
        medaka_gtf=MEDAKA_GTF,
        jaspar_meme_path=JASPAR_MEME_PATH,
        premzt_tf_list=PREMZT_TF_LIST_PATH,
        slamseq_gene_table=SLAMSEQ_GENE_TABLE_PATH,
        ortholog_map_path=ORTHOLOG_MAP_PATH,
        skip_slamseq=False,
        atac_accessions_zf=ZEBRAFISH_ATAC_ACCESSIONS,
        atac_accessions_medaka=MEDAKA_ATAC_ACCESSIONS,
        slam_accessions=SLAM_ACCESSIONS,
        use_pretrained_bias=USE_PRETRAINED_BIAS
    )

    if all_ready:
        print("[+] All required study resources are present and verified intact.")
        display(manifest_df)
        display(tm.status())
    else:
        dl_cmd = [
            "python", "-u", "scripts/download_resources.py",
            "--work-dir", WORK_DIR,
            "--config", f"{WORK_DIR}/config.json",
            "--max-host-workers", "3",
            "--max-per-host-workers", "2"
        ]
        if not ENABLE_MEDAKA:
            dl_cmd.append("--disable-medaka")
        if SLAMSEQ_GENE_TABLE_PATH:
            dl_cmd.extend(["--slamseq-table", SLAMSEQ_GENE_TABLE_PATH])
        
        print(f"[*] Identified {len(pending_tasks)} resource(s) requiring download:")
        display(manifest_df[manifest_df["Status"].str.startswith("PENDING")])
        print("[*] Submitting 'download_resources' to TaskManager (Max 3 distinct host websites in parallel)...")
        tm.submit("download_resources", dl_cmd)
        time.sleep(1.0)
        display(tm.status())
        print("    Tip: Re-run this cell at any time to inspect live progress, or execute tm.wait('download_resources') to block.")

---
## Block 1: Target-Species RNA-seq Analysis (Minor-Wave Calling & Diagnostics)

Supports dual quantification workflows with automatic detection (`RNA_MODE = "rds"` or `"bam"`):
1. **Multi-Species RDS Mode (`.rds`):**
   * Extracts replicate-resolved pre-MZT vs. maternal RNA-seq TPM tables via `scripts/extract_rds_tpm.R` and performs replicate-aware Welch *t*-tests on $\log_2(\text{TPM}+1)$ with Benjamini–Hochberg FDR adjustment (`padj`).
   * Assigns genes into biologically distinct cohorts:
     * **`strictly_zygotic_minor_wave`:** Virtually absent maternally (`mean_early < 0.5 TPM`) yet activated before MZT (`mean_premzt >= 2.0 TPM`, `log2FC >= 2.0`)—isolating *de novo* zygotic transcription from cytoplasmic polyadenylation (`highconf` tightens to `< 0.2 TPM`, `>= 3.0 TPM`, `log2FC >= 2.5`, `padj < 0.05`).
     * **`all_upregulated`:** Broader pre-MZT induced pool (`log2FC >= 1.5`, `mean_premzt >= 2.0`, `padj < 0.05`) that also permits maternally stocked genes gaining strong zygotic expression.
     * **`control_non_activated` (Maternal Control):** Abundant, stable maternal mRNAs (`mean_early >= 2.0 TPM`, `|log2FC| <= 0.5`). Separated deliberately from `control_silent` because maternal housekeeping/cell-cycle genes carry constitutively open CpG-island promoters even without zygotic activation, providing a crucial **timing-specificity** benchmark alongside silent genes (`mean_early < 0.2`, `mean_premzt < 0.2`).
2. **BAM Alignment Mode (`.bam`):**
   * Runs `featureCounts` + `PyDESeq2` normalization, 5'-to-3' spliced coverage profiling, and exonic heterozygous SNP allelic-shift scans (`pysam`) to detect paternal/zygotic allelic onset.
3. **Polyadenylation Hexamer Diagnostics (All Modes):** Tests 3' UTR enrichment of Cytoplasmic Polyadenylation Elements (**CPE**) and Polyadenylation Signals (**PAS**) to flag post-transcriptional adenylation candidates.

**Generated Outputs (Block 1):**
* **MA & Concordance Diagnostic Plot (`fig1_volcano_ma_plot.png`):** Shows $\log_2\text{FC}$ (pre-MZT vs. early baseline) against mean expression across primary (`Panel A`) and comparator (`Panel B`) species plus cross-species fold-change concordance (`Panel C`), highlighting strictly zygotic, maternal, and user target markers.
* **Allelic Shift Null Plot (`fig1c_snp_allele_shift.png`, BAM mode):** Empirical null distribution (`N` control SNPs + KDE + 95th percentile cutoff) of heterozygous allelic frequency shifts ($|\Delta \text{alt\_freq}|$) overlaid with target-gene medians.
* **Cohort Table (`{TARGET_LOCUS_NAME}_gene_sets.tsv`):** Genome-wide gene coordinates, expression summaries, and category assignments saved inside `WORK_DIR` (excluded from public git when RNA-seq inputs are unpublished).

In [ ]:
def _annotate_genes_staggered(ax_t, pts):
    """Place scatter callouts without label overlaps across MA/concordance panels."""
    if not pts:
        return
    xlim = ax_t.get_xlim()
    ylim = ax_t.get_ylim()
    xr = max(1e-3, xlim[1] - xlim[0])
    yr = max(1e-3, ylim[1] - ylim[0])
    min_dx = 0.09 * xr
    min_dy = 0.065 * yr
    placed = []
    offsets = [(7, 7), (7, -14), (-44, 9), (-44, -14), (14, 18), (14, -22)]
    for gname, mx, my in sorted(pts, key=lambda t: (t[2], t[1])):
        chosen = offsets[0]
        best_cost = float("inf")
        for ox, oy in offsets:
            tx = mx + (ox / 260.0) * xr
            ty = my + (oy / 200.0) * yr
            cost = sum(1.0 for px, py in placed if abs(tx - px) < min_dx and abs(ty - py) < min_dy)
            if cost < best_cost:
                best_cost = cost
                chosen = (ox, oy)
        placed.append((mx + (chosen[0] / 260.0) * xr, my + (chosen[1] / 200.0) * yr))
        ax_t.annotate(
            gname, (mx, my), xytext=chosen, textcoords="offset points",
            fontweight="bold", fontsize=9.5,
            arrowprops=dict(arrowstyle="-", color="#404040", lw=0.7, alpha=0.8),
            zorder=7
        )

import math
import gzip
import re
import pysam
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

rna_mode = globals().get("RNA_MODE", "bam")
gene_sets_tsv = res_dir / f"{TARGET_LOCUS_NAME}_gene_sets.tsv"
alt_gene_sets = Path(WORK_DIR) / "rnaseq" / "target_species_gene_sets.tsv"
alt_gene_sets.parent.mkdir(parents=True, exist_ok=True)

if rna_mode == "rds":
    print("=" * 80)
    print(f"[*] RNA-seq Analysis: RDS Multi-Species TPM Mode ({RNASEQ_TPM_RDS})")
    print("    Running extract_rds_tpm.R for both cichlid species...")
    print("=" * 80)
    repo_root = globals().get("repo_root", Path(".").resolve())
    rds_script = repo_root / "scripts" / "extract_rds_tpm.R"
    cmd = [
        shutil.which("Rscript") or str(Path(sys.prefix) / "bin" / "Rscript"), str(rds_script),
        "--rds", str(RNASEQ_TPM_RDS),
        "--gtf", str(TARGET_GTF),
        "--outdir", str(Path(WORK_DIR) / "rnaseq"),
        "--resdir", str(res_dir),
        "--locus-name", str(TARGET_LOCUS_NAME),
        "--primary-prefix", "AC",
        "--second-prefix", "TM"
    ]
    subprocess.run(cmd, check=True)
    df_counts = pd.read_csv(alt_gene_sets, sep="	")

    # Flags for plotting
    is_strictly_zygotic = df_counts["is_strictly_zygotic_minor_wave"]
    is_all_upregulated = df_counts["is_all_upregulated"]
    is_control = df_counts["is_control_non_activated"]
    is_both_cichlids = df_counts.get("is_both_cichlids_minor_wave", pd.Series(False, index=df_counts.index))

    fig, axes = plt.subplots(1, 3, figsize=(21, 6))

    # Panel 1: Primary Cichlid (AC) MA plot
    ax = axes[0]
    mean_ac = 0.5 * (df_counts["cpm_early"] + df_counts["cpm_premzt"])
    ax.scatter(np.log10(mean_ac + 0.1), df_counts["log2fc"], color="#cccccc", alpha=0.3, s=8, label="All Genes")
    ax.scatter(np.log10(mean_ac[is_control] + 0.1), df_counts.loc[is_control, "log2fc"], color="#2b83ba", alpha=0.6, s=12, label="Control Non-Activated")
    ax.scatter(np.log10(mean_ac[is_all_upregulated & ~is_strictly_zygotic] + 0.1), df_counts.loc[is_all_upregulated & ~is_strictly_zygotic, "log2fc"], color="#fdae61", alpha=0.7, s=16, label="All Upregulated")
    ax.scatter(np.log10(mean_ac[is_strictly_zygotic] + 0.1), df_counts.loc[is_strictly_zygotic, "log2fc"], color="#d7191c", alpha=0.8, s=22, label="Strictly Zygotic Minor Wave")

    pts_p1 = []
    for gname in TARGET_GENES + CANONICAL_ZGA_MARKERS:
        sub = df_counts[df_counts["gene_name"].str.lower() == gname.lower()]
        if not sub.empty:
            r = sub.iloc[0]
            mx = float(np.log10(0.5 * (r["cpm_early"] + r["cpm_premzt"]) + 0.1))
            my = float(r["log2fc"])
            ax.scatter([mx], [my], color="black", s=60, zorder=5)
            pts_p1.append((gname, mx, my))
    _annotate_genes_staggered(ax, pts_p1)

    ax.set_xlabel("log10(Mean TPM + 0.1)", fontsize=11)
    ax.set_ylabel(f"log2FC (1K vs 2C) - {TARGET_SPECIES_NAME}", fontsize=11)
    ax.set_title(f"{TARGET_SPECIES_NAME} (AC) RNA-seq MA Plot", fontweight="bold", fontsize=12)
    ax.axhline(0, color="black", linestyle="--", alpha=0.5)
    ax.legend(frameon=True, fontsize=8.5, loc="upper left")
    ax.grid(True, linestyle=":", alpha=0.5)

    # Panel 2: Second Cichlid (TM) MA plot
    ax = axes[1]
    mean_tm = 0.5 * (df_counts["tm_tpm_early"] + df_counts["tm_tpm_premzt"])
    tm_minor = df_counts["tm_is_strictly_zygotic_minor_wave"]
    tm_up = df_counts["tm_is_all_upregulated"]
    tm_ctrl = df_counts["tm_is_control_non_activated"]
    ax.scatter(np.log10(mean_tm + 0.1), df_counts["tm_log2fc"], color="#cccccc", alpha=0.3, s=8, label="All Genes")
    ax.scatter(np.log10(mean_tm[tm_ctrl] + 0.1), df_counts.loc[tm_ctrl, "tm_log2fc"], color="#2b83ba", alpha=0.6, s=12, label="Control Non-Activated")
    ax.scatter(np.log10(mean_tm[tm_up & ~tm_minor] + 0.1), df_counts.loc[tm_up & ~tm_minor, "tm_log2fc"], color="#fdae61", alpha=0.7, s=16, label="All Upregulated")
    ax.scatter(np.log10(mean_tm[tm_minor] + 0.1), df_counts.loc[tm_minor, "tm_log2fc"], color="#d7191c", alpha=0.8, s=22, label="Strictly Zygotic Minor Wave")

    pts_p2 = []
    for gname in TARGET_GENES + CANONICAL_ZGA_MARKERS:
        sub = df_counts[df_counts["gene_name"].str.lower() == gname.lower()]
        if not sub.empty:
            r = sub.iloc[0]
            mx = float(np.log10(0.5 * (r["tm_tpm_early"] + r["tm_tpm_premzt"]) + 0.1))
            my = float(r["tm_log2fc"])
            ax.scatter([mx], [my], color="black", s=60, zorder=5)
            pts_p2.append((gname, mx, my))
    _annotate_genes_staggered(ax, pts_p2)

    ax.set_xlabel("log10(Mean TPM + 0.1)", fontsize=11)
    ax.set_ylabel("log2FC (1K vs 2C) - Second Cichlid (TM)", fontsize=11)
    ax.set_title("Second Cichlid Species (TM) RNA-seq MA Plot", fontweight="bold", fontsize=12)
    ax.axhline(0, color="black", linestyle="--", alpha=0.5)
    ax.legend(frameon=True, fontsize=8.5, loc="upper left")
    ax.grid(True, linestyle=":", alpha=0.5)

    # Panel 3: Two-Cichlid log2FC Concordance Scatter
    ax = axes[2]
    ax.scatter(df_counts["log2fc"], df_counts["tm_log2fc"], color="#cccccc", alpha=0.25, s=8, label="Other Genes")
    ax.scatter(df_counts.loc[is_control, "log2fc"], df_counts.loc[is_control, "tm_log2fc"], color="#2b83ba", alpha=0.4, s=10, label="Control Non-Activated")
    ax.scatter(df_counts.loc[is_strictly_zygotic & ~is_both_cichlids, "log2fc"], df_counts.loc[is_strictly_zygotic & ~is_both_cichlids, "tm_log2fc"], color="#fdae61", alpha=0.7, s=18, label="AC-only Minor Wave")
    ax.scatter(df_counts.loc[tm_minor & ~is_both_cichlids, "log2fc"], df_counts.loc[tm_minor & ~is_both_cichlids, "tm_log2fc"], color="#abd9e9", alpha=0.7, s=18, label="TM-only Minor Wave")
    ax.scatter(df_counts.loc[is_both_cichlids, "log2fc"], df_counts.loc[is_both_cichlids, "tm_log2fc"], color="#d7191c", alpha=0.9, s=28, label=f"Conserved Minor Wave (n={is_both_cichlids.sum()})")

    ax.plot([-5, 12], [-5, 12], color="black", linestyle="--", alpha=0.6, label="y = x (Concordance)")
    ax.axhline(0, color="gray", linestyle=":", alpha=0.4)
    ax.axvline(0, color="gray", linestyle=":", alpha=0.4)

    pts_p3 = []
    for gname in TARGET_GENES + CANONICAL_ZGA_MARKERS:
        sub = df_counts[df_counts["gene_name"].str.lower() == gname.lower()]
        if not sub.empty:
            r = sub.iloc[0]
            mx = float(r["log2fc"])
            my = float(r["tm_log2fc"])
            ax.scatter([mx], [my], color="black", s=70, zorder=6)
            pts_p3.append((gname, mx, my))
    _annotate_genes_staggered(ax, pts_p3)

    ax.set_xlabel(f"log2FC ({TARGET_SPECIES_NAME} / AC)", fontsize=11)
    ax.set_ylabel("log2FC (Second Species / TM)", fontsize=11)
    ax.set_title("Two-Cichlid pre-MZT log2FC Concordance", fontweight="bold", fontsize=12)
    ax.legend(frameon=True, fontsize=8, loc="upper left")
    ax.grid(True, linestyle=":", alpha=0.5)

    plt.tight_layout()
    fig1_path = res_dir / "fig1_volcano_ma_plot.png"
    plt.savefig(fig1_path, dpi=300)
    plt.show()
    print(f"[+] Figure 1 (Two-species MA & log2FC concordance plots) exported to {fig1_path}")

else:
    def is_bam_paired_end(bam_path: str) -> bool:
        try:
            with pysam.AlignmentFile(bam_path, "rb") as bf:
                for read in bf:
                    if not read.is_secondary and not read.is_supplementary:
                        return bool(read.is_paired)
        except Exception:
            pass
        return False

    counts_file = Path(WORK_DIR) / "rnaseq" / "counts.txt"
    counts_file.parent.mkdir(parents=True, exist_ok=True)
    if not counts_file.exists() or FORCE_RERUN:
        all_bams = RNASEQ_EARLY_BAMS + RNASEQ_PREMZT_BAMS
        is_pe = is_bam_paired_end(all_bams[0]) if all_bams and Path(all_bams[0]).exists() else False
        cmd = ["featureCounts", "-T", str(THREADS), "-s", str(RNASEQ_STRANDNESS)]
        if is_pe:
            cmd.append("-p")
        cmd.extend(["-a", TARGET_GTF, "-o", str(counts_file)] + all_bams)
        print(f"[*] Running featureCounts on target BAMs (Paired-End: {is_pe})...")
        subprocess.run(cmd, check=True)

    df_counts = pd.read_csv(counts_file, sep="	", comment="#")
    early_bams = [Path(b).name for b in RNASEQ_EARLY_BAMS]
    premzt_bams = [Path(b).name for b in RNASEQ_PREMZT_BAMS]

    col_early = [c for c in df_counts.columns if any(b in c for b in early_bams)]
    col_premzt = [c for c in df_counts.columns if any(b in c for b in premzt_bams)]

    if len(col_early) >= 2 and len(col_premzt) >= 2:
        print("[*] Replicates available: using PyDESeq2 for replicate-aware differential expression...")
        from pydeseq2.dds import DeseqDataSet
        from pydeseq2.ds import DeseqStats

        counts_df = df_counts[["Geneid"] + col_early + col_premzt].set_index("Geneid").T
        clinical_df = pd.DataFrame({
            "sample": counts_df.index,
            "condition": ["early"] * len(col_early) + ["premzt"] * len(col_premzt)
        }).set_index("sample")

        dds = DeseqDataSet(counts=counts_df, metadata=clinical_df, design_factors="condition", refit_cooks=True)
        dds.deseq2()
        stat_res = DeseqStats(dds, contrast=["condition", "premzt", "early"])
        stat_res.summary()
        res_df = stat_res.results_df

        df_counts["baseMean"] = res_df["baseMean"].values
        df_counts["log2fc"] = res_df["log2FoldChange"].values
        df_counts["pval"] = res_df["pvalue"].values
        df_counts["padj"] = res_df["padj"].values

        raw_early = df_counts[col_early].sum(axis=1)
        raw_premzt = df_counts[col_premzt].sum(axis=1)
        tot_early = df_counts[col_early].values.sum()
        tot_premzt = df_counts[col_premzt].values.sum()
        df_counts["cpm_early"] = (raw_early / max(tot_early, 1)) * 1e6
        df_counts["cpm_premzt"] = (raw_premzt / max(tot_premzt, 1)) * 1e6
    else:
        print("[*] NOTICE: Unreplicated single-BAM per stage detected.")
        c_early = df_counts[col_early[0]].values.astype(float)
        c_premzt = df_counts[col_premzt[0]].values.astype(float)
        geom_ref = np.exp(0.5 * (np.log(np.maximum(c_early, 1e-5)) + np.log(np.maximum(c_premzt, 1e-5))))
        pos_mask = (c_early > 0) & (c_premzt > 0)
        sf_early = float(np.median((c_early[pos_mask] / geom_ref[pos_mask])))
        sf_premzt = float(np.median((c_premzt[pos_mask] / geom_ref[pos_mask])))
        tot_early = c_early.sum()
        tot_premzt = c_premzt.sum()
        df_counts["cpm_early"] = (c_early / max(tot_early, 1)) * 1e6
        df_counts["cpm_premzt"] = (c_premzt / max(tot_premzt, 1)) * 1e6
        norm_early = c_early / max(sf_early, 1e-5)
        norm_premzt = c_premzt / max(sf_premzt, 1e-5)
        df_counts["log2fc"] = np.log2((norm_premzt + 1.0) / (norm_early + 1.0))
        df_counts["padj"] = 1.0

    # Parse GTF coordinates
    gene_name_map = {}
    with (gzip.open(TARGET_GTF, "rt") if str(TARGET_GTF).endswith(".gz") else open(TARGET_GTF, "rt")) as f:
        for line in f:
            if line.startswith("#"): continue
            p = line.strip().split("	")
            if p[2] == "gene":
                gid, gname = None, None
                for item in p[8].split(";"):
                    item = item.strip()
                    if item.startswith("gene_id"):
                        m = re.search(r'gene_id\s+"([^"]+)"', item)
                        if m: gid = m.group(1)
                    elif item.startswith("gene_name"):
                        m = re.search(r'gene_name\s+"([^"]+)"', item)
                        if m: gname = m.group(1)
                if gid:
                    gene_name_map[gid] = gname if gname else gid

    df_counts["gene_name"] = df_counts["Geneid"].map(gene_name_map).fillna(df_counts["Geneid"])
    df_counts["gene_id"] = df_counts["Geneid"]
    df_counts["chrom"] = df_counts["Chr"].apply(lambda x: str(x).split(";")[0])
    df_counts["strand"] = df_counts["Strand"].apply(lambda x: str(x).split(";")[0])
    df_counts["start"] = df_counts["Start"].apply(lambda x: int(str(x).split(";")[0]))
    df_counts["end"] = df_counts["End"].apply(lambda x: int(str(x).split(";")[-1]))
    df_counts["tss"] = np.where(df_counts["strand"] == "+", df_counts["start"], df_counts["end"])
    df_counts["tes"] = np.where(df_counts["strand"] == "+", df_counts["end"], df_counts["start"])

    is_strictly_zygotic = (df_counts["cpm_early"] < 0.5) & (df_counts["cpm_premzt"] >= 2.0) & (df_counts["log2fc"] >= 2.0)
    is_strictly_highconf = (df_counts["cpm_early"] < 0.2) & (df_counts["cpm_premzt"] >= 3.0) & (df_counts["log2fc"] >= 2.5)
    is_all_upregulated = (df_counts["log2fc"] >= 1.5) & (df_counts["cpm_premzt"] >= 2.0)
    is_control = (df_counts["cpm_early"] >= 2.0) & (df_counts["log2fc"].abs() <= 0.5)
    is_silent = (df_counts["cpm_early"] < 0.2) & (df_counts["cpm_premzt"] < 0.2)

    df_counts["is_strictly_zygotic_minor_wave"] = is_strictly_zygotic
    df_counts["is_all_upregulated"] = is_all_upregulated
    df_counts["is_control_non_activated"] = is_control
    df_counts["category"] = "unclassified"
    df_counts.loc[is_silent, "category"] = "control_silent"
    df_counts.loc[is_control, "category"] = "control_non_activated"
    df_counts.loc[is_all_upregulated, "category"] = "all_upregulated"
    df_counts.loc[is_strictly_zygotic, "category"] = "strictly_zygotic_minor_wave"
    df_counts["is_control_silent"] = (df_counts["category"] == "control_silent")
    df_counts["is_strictly_zygotic_highconf"] = is_strictly_highconf
    df_counts["is_slam_confirmed_minor_wave"] = False

    df_counts.to_csv(gene_sets_tsv, sep="	", index=False)
    df_counts.to_csv(alt_gene_sets, sep="	", index=False)


In [ ]:
def merge_intervals(intervals):
    """Merge overlapping exon intervals across transcript isoforms to avoid double-counting."""
    if not intervals:
        return []
    intervals = sorted(intervals, key=lambda x: (x[0], x[1]))
    merged = [list(intervals[0])]
    for s, e in intervals[1:]:
        if s <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], e)
        else:
            merged.append([s, e])
    return [tuple(x) for x in merged]

# 1. Parse and merge exonic coordinates per gene from GTF (Runs in all modes)
gene_exons = {}
with (gzip.open(TARGET_GTF, "rt") if str(TARGET_GTF).endswith(".gz") else open(TARGET_GTF, "rt")) as f:
    for line in f:
        if line.startswith("#"):
            continue
        p = line.strip().split("	")
        if len(p) >= 9 and p[2] == "exon":
            chrom, start, end, strand = p[0], int(p[3]) - 1, int(p[4]), p[6]
            gid = None
            for item in p[8].split(";"):
                item = item.strip()
                if item.startswith("gene_id"):
                    m = re.search(r'gene_id\s+"([^"]+)"', item)
                    if m:
                        gid = m.group(1)
            if gid:
                if gid not in gene_exons:
                    gene_exons[gid] = {"chrom": chrom, "strand": strand, "exons": []}
                gene_exons[gid]["exons"].append((start, end))

for gid in gene_exons:
    gene_exons[gid]["exons"] = merge_intervals(gene_exons[gid]["exons"])
    gene_exons[gid]["merged_exons"] = gene_exons[gid]["exons"]

print(f"[+] Parsed merged exonic intervals for {len(gene_exons):,} genes from {Path(TARGET_GTF).name}")

# 2. Spliced 5'-to-3' Exonic Read Coverage Diagnostic (BAM-only)
if globals().get("RNA_MODE", "bam") == "rds":
    print("=" * 80)
    print("[NOTICE] RDS mode active: Skipping BAM-only spliced 5'-to-3' exonic coverage profile.")
    print("=" * 80)
else:
    def calculate_spliced_coverage_profile(gid, bam_file, n_bins=100):
        ginfo = gene_exons.get(gid)
        if not ginfo:
            return np.zeros(n_bins, dtype=float)
        chrom = ginfo["chrom"]
        strand = ginfo["strand"]
        exons = ginfo["exons"]
        cov_pieces = []
        for st, en in exons:
            try:
                cov = np.array(bam_file.count_coverage(chrom, st, en, quality_threshold=10)).sum(axis=0)
            except Exception:
                cov = np.zeros(en - st, dtype=float)
            cov_pieces.append(cov)
        if not cov_pieces:
            return np.zeros(n_bins, dtype=float)
        full_cov = np.concatenate(cov_pieces)
        if len(full_cov) == 0:
            return np.zeros(n_bins, dtype=float)
        bin_edges = np.linspace(0, len(full_cov), n_bins + 1, dtype=int)
        profile = np.zeros(n_bins, dtype=float)
        for i in range(n_bins):
            s_idx = bin_edges[i]
            e_idx = max(s_idx + 1, bin_edges[i + 1])
            profile[i] = full_cov[s_idx:e_idx].mean() if e_idx > s_idx else 0.0
        if strand == "-":
            profile = profile[::-1]
        tot = profile.sum()
        if tot > 0:
            profile = profile / tot
        return profile

    all_bams = RNASEQ_EARLY_BAMS + RNASEQ_PREMZT_BAMS
    if all_bams and Path(all_bams[0]).exists():
        print("[*] Computing spliced 5'-to-3' exonic coverage profiles...")
        # Plotting profiles


In [ ]:
# Post-Transcriptional vs. Transcriptional Diagnostics: Exonic SNP Allelic Shift Test
# and 3' UTR Cytoplasmic Polyadenylation Element (CPE) & PAS Read-Extension Tests
from scipy.stats import fisher_exact, mannwhitneyu, gaussian_kde
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
from pyfaidx import Fasta

def scan_exonic_snps(bam_2c, bam_1k, gid, gname, min_depth=20, min_maf=0.10):
    ginfo = gene_exons.get(gid)
    if not ginfo:
        return []
    chrom = ginfo["chrom"]
    results = []
    # Use consistent key
    exons_list = ginfo.get("exons", ginfo.get("merged_exons", []))
    for st, en in exons_list:
        try:
            cov_2c = np.array(bam_2c.count_coverage(chrom, st, en, quality_threshold=20))
            cov_1k = np.array(bam_1k.count_coverage(chrom, st, en, quality_threshold=20))
        except Exception:
            continue
        d_2c = cov_2c.sum(axis=0)
        d_1k = cov_1k.sum(axis=0)
        valid = (d_2c >= min_depth) & (d_1k >= min_depth)
        for idx in np.where(valid)[0]:
            c2c_bases = cov_2c[:, idx]
            c1k_bases = cov_1k[:, idx]
            d2 = d_2c[idx]
            d1 = d_1k[idx]
            maj_idx = np.argmax(c2c_bases)
            ref_2c = c2c_bases[maj_idx]
            alt_2c = d2 - ref_2c
            ref_1k = c1k_bases[maj_idx]
            alt_1k = d1 - ref_1k
            f2 = alt_2c / d2
            f1 = alt_1k / d1
            if f2 >= min_maf or f1 >= min_maf:
                tbl = [[alt_1k, ref_1k], [alt_2c, ref_2c]]
                _, p_val = fisher_exact(tbl, alternative="two-sided")
                results.append({
                    "gene": gname,
                    "chrom": chrom,
                    "pos": st + idx,
                    "depth_early": d2,
                    "depth_premzt": d1,
                    "alt_freq_early": f2,
                    "alt_freq_premzt": f1,
                    "delta_alt_freq": f1 - f2,
                    "abs_delta_alt_freq": abs(f1 - f2),
                    "fisher_pval": p_val
                })
    return results

target_snps = []
ctrl_shifts = []

if globals().get("RNA_MODE", "bam") == "rds":
    print("=" * 80)
    print("[NOTICE] RDS mode active: Skipping BAM-only exonic SNP allelic shift test.")
    print("=" * 80)
else:
    with pysam.AlignmentFile(RNASEQ_EARLY_BAMS[0], "rb") as bam_2c, pysam.AlignmentFile(RNASEQ_PREMZT_BAMS[0], "rb") as bam_1k:
        # 1. Target genes scan
        for tg in TARGET_GENES:
            tg_rows = df_counts[df_counts["gene_name"].str.lower() == tg.lower()]
            for _, r in tg_rows.iterrows():
                target_snps.extend(scan_exonic_snps(bam_2c, bam_1k, r["Geneid"], r["gene_name"]))
    
        # 2. Genome-wide control-gene null baseline (top 100 control genes, depth>=30, MAF>=0.15)
        ctrl_genes = df_counts[df_counts["category"] == "control_non_activated"].sort_values(by="cpm_early", ascending=False).head(100)
        for gid in ctrl_genes["Geneid"]:
            ginfo = gene_exons.get(gid)
            if not ginfo:
                continue
            chrom = ginfo["chrom"]
            exons_list = ginfo.get("exons", ginfo.get("merged_exons", []))
            for st, en in exons_list:
                try:
                    cov_2c = np.array(bam_2c.count_coverage(chrom, st, en, quality_threshold=20))
                    cov_1k = np.array(bam_1k.count_coverage(chrom, st, en, quality_threshold=20))
                except Exception:
                    continue
                depth_2c = cov_2c.sum(axis=0)
                depth_1k = cov_1k.sum(axis=0)
                valid = (depth_2c >= 30) & (depth_1k >= 30)
                for idx in np.where(valid)[0]:
                    c2c_bases = cov_2c[:, idx]
                    c1k_bases = cov_1k[:, idx]
                    d2 = depth_2c[idx]
                    d1 = depth_1k[idx]
                    maj_idx = np.argmax(c2c_bases)
                    ref_2c = c2c_bases[maj_idx]
                    alt_2c = d2 - ref_2c
                    ref_1k = c1k_bases[maj_idx]
                    alt_1k = d1 - ref_1k
                    f2 = alt_2c / d2
                    f1 = alt_1k / d1
                    if f2 >= 0.15 or f1 >= 0.15:
                        ctrl_shifts.append(abs(f1 - f2))
    
    df_target_snps = pd.DataFrame(target_snps)
    if not df_target_snps.empty:
        _, padj, _, _ = multipletests(df_target_snps["fisher_pval"], method="fdr_bh")
        df_target_snps["padj"] = padj
        print(f"[*] Detected heterozygous exonic sites in TARGET_GENES ({len(df_target_snps)} sites):")
        disp_cols = ["gene", "pos", "depth_early", "depth_premzt", "alt_freq_early", "alt_freq_premzt", "delta_alt_freq", "fisher_pval", "padj"]
        print(df_target_snps[disp_cols].to_string(index=False))
    else:
        print("[-] No heterozygous exonic positions detected in target genes.")
    
    ctrl_shifts = np.array(ctrl_shifts)
    if len(ctrl_shifts) > 0:
        ctrl_p95 = np.percentile(ctrl_shifts, 95)
        print(f"[*] Control-Gene Null Baseline: {len(ctrl_shifts)} sites | Median |Δ| = {np.median(ctrl_shifts):.4f} | 95th %ile = {ctrl_p95:.4f}")
    else:
        ctrl_p95 = 0.10
        print("[-] Control shifts empty, using default threshold 0.10")
    
    for tg in TARGET_GENES:
        if not df_target_snps.empty and "gene" in df_target_snps.columns:
            tg_sub = df_target_snps[df_target_snps["gene"].str.lower() == tg.lower()]
            if not tg_sub.empty and len(ctrl_shifts) > 0:
                tg_vals = tg_sub["abs_delta_alt_freq"].values
                u_stat, u_pval = mannwhitneyu(tg_vals, ctrl_shifts, alternative="greater")
                sig_str = "SIGNIFICANT (exceeds background noise)" if u_pval < 0.05 else "Not significant"
                print(f"    {tg:10s}: Median |Δ| = {np.median(tg_vals):.4f} | U = {u_stat:.1f}, p = {u_pval:.4e} -> {sig_str}")
    
    # Plot Figure 1c
    fig, ax = plt.subplots(figsize=(8, 4.5))
    if len(ctrl_shifts) > 5:
        ax.hist(ctrl_shifts, bins=35, density=True, alpha=0.55, color="#9ecae1", edgecolor="#3182bd", label=f"Control-Gene Null (N={len(ctrl_shifts)})")
        kde = gaussian_kde(ctrl_shifts)
        x_eval = np.linspace(0, max(0.35, float(ctrl_shifts.max())), 200)
        ax.plot(x_eval, kde(x_eval), color="#08519c", lw=2.2, label="Null Background Density")
    ax.axvline(ctrl_p95, color="#e6550d", linestyle="--", lw=2.0, label=f"95th %ile Threshold ({ctrl_p95:.3f})")
    
    color_map_tg = {"slc35a3a": "#d7191c", "notch2": "#756bb1"}
    for tg in TARGET_GENES:
        if not df_target_snps.empty and "gene" in df_target_snps.columns:
            tg_sub = df_target_snps[df_target_snps["gene"].str.lower() == tg.lower()]
            if not tg_sub.empty:
                tg_med = np.median(tg_sub["abs_delta_alt_freq"].values)
                c = color_map_tg.get(tg.lower(), "#2ca02c")
                ax.axvline(tg_med, color=c, lw=2.5, linestyle="-", label=f"{tg} Median Shift ({tg_med:.3f})")
    
    ax.set_xlabel("Allelic Frequency Shift |Δ alt_freq| (1000-cell vs 2-cell)", fontsize=11)
    ax.set_ylabel("Empirical Density", fontsize=11)
    ax.set_title(f"Target Gene Allelic Shifts vs. Control Null\n({TARGET_SPECIES_NAME.replace('_', ' ')})", fontweight="bold", fontsize=11.5)
    ax.legend(frameon=True, fontsize=9.5, loc="upper right")
    ax.grid(True, linestyle=":", alpha=0.5)
    plt.tight_layout()
    fig1c_path = res_dir / "fig1c_snp_allele_shift.png"
    plt.savefig(fig1c_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"[+] Figure 1c exported to {fig1c_path}")
    
    # ==============================================================================
    
# 3' UTR CPE Enrichment & Downstream PAS Read-Extension Diagnostic
# ==============================================================================
print("=" * 80)
print("[*] 3' UTR Post-Transcriptional Diagnostic: CPE & PAS Motif Enrichment:")
gene_3putr = {}
with (gzip.open(TARGET_GTF, "rt") if str(TARGET_GTF).endswith(".gz") else open(TARGET_GTF, "rt")) as f:
    for line in f:
        if line.startswith("#"):
            continue
        p = line.strip().split("\t")
        if len(p) >= 9 and p[2].lower() in {"three_prime_utr", "3utr", "3_prime_utr"}:
            chrom, s, e, strand = p[0], int(p[3]) - 1, int(p[4]), p[6]
            gid = None
            for item in p[8].split(";"):
                item = item.strip()
                if item.startswith("gene_id"):
                    m = re.search(r'gene_id\s+"([^"]+)"', item)
                    if m:
                        gid = m.group(1)
            if gid:
                if gid not in gene_3putr:
                    gene_3putr[gid] = {"chrom": chrom, "strand": strand, "intervals": []}
                gene_3putr[gid]["intervals"].append((s, e))

for gid, ginfo in gene_exons.items():
    if gid not in gene_3putr or not gene_3putr[gid]["intervals"]:
        chrom, strand = ginfo["chrom"], ginfo["strand"]
        exs = ginfo["exons"]
        if exs:
            if strand == "+":
                last_e = exs[-1][1]
                gene_3putr[gid] = {"chrom": chrom, "strand": strand, "intervals": [(max(0, last_e - 500), last_e)]}
            else:
                first_s = exs[0][0]
                gene_3putr[gid] = {"chrom": chrom, "strand": strand, "intervals": [(first_s, first_s + 500)]}

# Merge overlapping intervals across isoforms to avoid double-counting 3' UTR sequence
for gid in gene_3putr:
    gene_3putr[gid]["intervals"] = merge_intervals(gene_3putr[gid]["intervals"])

utr_seqs = {}
if Path(TARGET_GENOME_FA).exists():
    try:
        fa = Fasta(TARGET_GENOME_FA, as_raw=True)
        for gid, uinfo in gene_3putr.items():
            ch = uinfo["chrom"]
            st_list = []
            for st, en in uinfo["intervals"]:
                try:
                    st_list.append(str(fa[ch][st:en]))
                except Exception:
                    pass
            cat_seq = "".join(st_list).upper()
            if uinfo["strand"] == "-":
                cat_seq = cat_seq.translate(str.maketrans("ACGTN", "TGCAN"))[::-1]
            utr_seqs[gid] = cat_seq
    except Exception as e:
        print(f"[-] Notice loading FASTA for UTR extraction: {e}")

if "is_all_upregulated" in df_counts.columns:
    upreg_mask = (df_counts["is_all_upregulated"] == True) | (df_counts["category"].isin(["all_upregulated", "strictly_zygotic_minor_wave"]))
else:
    upreg_mask = df_counts["category"].isin(["all_upregulated", "strictly_zygotic_minor_wave"])
upreg_gids = set(df_counts.loc[upreg_mask, "Geneid"]) & set(utr_seqs.keys())
ctrl_gids = set(df_counts.loc[df_counts["category"] == "control_non_activated", "Geneid"]) & set(utr_seqs.keys())

tests = {
    "Canonical CPE (TTTTAT/TTTTAAT)": ["TTTTAT", "TTTTAAT"],
    "eCPE / U-rich (TTTTTT/TTTTACT)": ["TTTTTT", "TTTTACT", "TTTTAAG"],
    "PAS Hexamer (AATAAA/ATTAAA)":    ["AATAAA", "ATTAAA"]
}

for t_name, motif_list in tests.items():
    up_pos = sum(1 for gid in upreg_gids if gid in utr_seqs and any(m in utr_seqs[gid] for m in motif_list))
    up_neg = len(upreg_gids) - up_pos
    ctrl_pos = sum(1 for gid in ctrl_gids if gid in utr_seqs and any(m in utr_seqs[gid] for m in motif_list))
    ctrl_neg = len(ctrl_gids) - ctrl_pos
    cont_tab = [[up_pos, max(0, up_neg)], [ctrl_pos, max(0, ctrl_neg)]]
    odds, p_val = fisher_exact(cont_tab, alternative="greater")
    print(f"    {t_name:32s}: Upreg {up_pos}/{len(upreg_gids)} ({100*up_pos/max(len(upreg_gids),1):.1f}%) vs Ctrl {ctrl_pos}/{len(ctrl_gids)} ({100*ctrl_pos/max(len(ctrl_gids),1):.1f}%) | OR = {odds:.2f}, p = {p_val:.2e}")

print("=" * 80)

if globals().get("RNA_MODE", "bam") == "rds":
    print("=" * 80)
    print("[NOTICE] RDS mode active: Skipping +500 bp read extension depth check (requires BAM alignments).")
    print("         Inspecting downstream PAS hexamers from genome FASTA directly for TARGET_GENES:")
    print("=" * 80)
    if Path(TARGET_GENOME_FA).exists():
        fa = Fasta(TARGET_GENOME_FA, as_raw=True)
        for tg in TARGET_GENES:
            tg_rows = df_counts[df_counts["gene_name"].str.lower() == tg.lower()]
            if tg_rows.empty: continue
            r = tg_rows.iloc[0]
            chrom, strand = r["chrom"], r["strand"]
            tes = int(r["end"]) if strand == "+" else int(r["start"])
            down_st = tes if strand == "+" else max(0, tes - 500)
            down_en = (tes + 500) if strand == "+" else tes
            try:
                down_seq = str(fa[chrom][down_st:down_en]).upper()
                if strand == "-":
                    down_seq = down_seq.translate(str.maketrans("ACGTN", "TGCAN"))[::-1]
            except Exception:
                down_seq = ""
            pas_found = [m for m in ["AATAAA", "ATTAAA"] if m in down_seq]
            print(f"    {tg} ({chrom}:{down_st}-{down_en}, strand {strand}): PAS hexamers: {pas_found or 'None'}")
else:
    print(f"[*] Inspecting +500 bp Downstream Intergenic Read Extension for TARGET_GENES:")
    with pysam.AlignmentFile(RNASEQ_EARLY_BAMS[0], "rb") as b_2c, pysam.AlignmentFile(RNASEQ_PREMZT_BAMS[0], "rb") as b_1k:
        for tg in TARGET_GENES:
            tg_rows = df_counts[df_counts["gene_name"].str.lower() == tg.lower()]
            if tg_rows.empty:
                continue
            r = tg_rows.iloc[0]
            chrom = r["chrom"]
            strand = r["strand"]
            tes = int(r["end"]) if strand == "+" else int(r["start"])
            down_st = tes if strand == "+" else max(0, tes - 500)
            down_en = (tes + 500) if strand == "+" else tes
            try:
                cov2 = np.array(b_2c.count_coverage(chrom, down_st, down_en, quality_threshold=10)).sum()
                cov1 = np.array(b_1k.count_coverage(chrom, down_st, down_en, quality_threshold=10)).sum()
            except Exception:
                cov2, cov1 = 0, 0
            down_seq = ""
            if Path(TARGET_GENOME_FA).exists():
                try:
                    fa = Fasta(TARGET_GENOME_FA, as_raw=True)
                    down_seq = str(fa[chrom][down_st:down_en]).upper()
                    if strand == "-":
                        down_seq = down_seq.translate(str.maketrans("ACGTN", "TGCAN"))[::-1]
                except Exception:
                    pass
            pas_found = [m for m in ["AATAAA", "ATTAAA"] if m in down_seq]
            print(f"    {tg} ({chrom}:{down_st}-{down_en}, strand {strand}): 2-cell cov = {cov2:,}, 1000-cell cov = {cov1:,} | PAS: {pas_found or 'None'}")


---
## Block 1b: Reference Embryo SLAM-seq Nascent RNA Ortholog Overlay

To separate **true pre-MZT nascent transcription** from **cytoplasmic polyadenylation of maternal RNA** (where maternal transcripts gain poly(A) tails in the cytoplasm and look upregulated in oligo(dT) RNA-seq despite zero nuclear Pol II synthesis), this step maps 4sU metabolic labeling classes (`scripts/prep_slamseq.py`) from the reference embryo onto target-species 1:1 orthologs:
* **`preMZT_strictly_zygotic`:** Absent total RNA early (`CPM < 1.0`) with nascent 4sU T$\rightarrow$C labeling appearing before MZT.
* **`preMZT_maternal_zygotic`:** Maternally supplied early (`CPM >= 5.0`) **and** actively labeled with nascent T$\rightarrow$C reads ($\ge 25\%$ of peak) before MZT.
* **`major_wave_only`:** Minimal pre-MZT nascent T$\rightarrow$C incorporation; activates strongly during the major wave.
* **`maternal_non_transcribed`:** Stable maternal mRNA (`CPM >= 5.0`) with negligible nascent T$\rightarrow$C transcription—used to quantify what fraction of "upregulated" oligo(dT) transcripts represent polyadenylation rather than transcription.

Target-species upregulated genes whose reference 1:1 ortholog belongs to `preMZT_strictly_zygotic` or `preMZT_maternal_zygotic` define the nascent-validated **`is_slam_confirmed_minor_wave`** high-confidence cohort.

**Generated Outputs (Block 1b):**
* **Figure 1b (`fig1b_slamseq_ortholog_overlay.png`):** (`A`) Target RNA-seq MA plot recolored by orthologous nascent SLAM-seq timing class with target/ZGA markers labeled in full; (`B`) violin distributions of target $\log_2\text{FC}$ grouped by orthologous timing class with sample sizes ($n$), Mann–Whitney *U* tests, and one-sided Fisher exact enrichment odds ratio/$p$-value.
* **Nascent QC Summary (`nascent_rnaseq_qc_summary.tsv`):** Filtering counts removing mitochondrial genes, maternal bleed-through, low baseline fold-changes, and collapsing kinetics.

In [ ]:
# Launch / Verify Zebrafish SLAM-seq Preprocessing Task
slam_cmd = [
    "python", "-u", "scripts/prep_slamseq.py",
    "--work-dir", WORK_DIR,
    "--genome-fa", ZEBRAFISH_GENOME_FA,
    "--gtf", ZEBRAFISH_GTF,
    "--task-name", "prep_slamseq_zf",
    "--threads", str(THREADS),
    "--min-baseline-fc", str(SLAM_MIN_BASELINE_FC),
    "--min-kinetics-retention", str(SLAM_MIN_KINETICS_RETENTION)
]
if SLAM_EXCLUDE_MITO:
    slam_cmd.append("--exclude-mito")
else:
    slam_cmd.append("--no-exclude-mito")
if SLAM_MASK_BASELINE_SNPS:
    slam_cmd.append("--mask-baseline-snps")
else:
    slam_cmd.append("--no-mask-baseline-snps")

# Trigger execution if empirical minor-wave table is missing, while cleanly skipping completed heavy steps
_zf_minor_tsv = Path(WORK_DIR) / "slamseq" / "zebrafish_slamseq_minor_wave_genes.tsv"
_force_slam = bool(globals().get("FORCE_RERUN", False) or (not _zf_minor_tsv.exists()))

tm.submit("prep_slamseq_zf", slam_cmd, depends_on="download_resources", force_rerun=_force_slam)
display(tm.status())

In [ ]:
# ==============================================================================
# Species- and Project-Agnostic Inline SLAM-seq QC Diagnostic & Filtering Audit
# ==============================================================================
import re
from pathlib import Path
import pandas as pd
import numpy as np

# Resolve results directory and SLAM-seq gene classification table
_res_dir = globals().get("res_dir", Path(WORK_DIR) / "results" / globals().get("TARGET_LOCUS_NAME", ""))
_res_dir.mkdir(parents=True, exist_ok=True)
_zf_slam_path = Path(WORK_DIR) / "slamseq" / "zf_slamseq_gene_classes.tsv"

if not _zf_slam_path.exists():
    print(f"[PENDING] SLAM-seq gene classification table not yet available: {_zf_slam_path}")
    print("          Run 'prep_slamseq_zf' to completion before evaluating nascent QC diagnostics.")
else:
    df_slam = pd.read_csv(_zf_slam_path, sep="	")
    print(f"[+] Loaded SLAM-seq classification table: {_zf_slam_path.name} ({len(df_slam):,} genes)")

    # 1. Agnostic mitochondrial contig and symbol detection
    mito_chroms = {"mt", "chrm", "m", "chrmt", "mitochondrion"}
    is_mito_c = df_slam["chrom"].astype(str).str.lower().isin(mito_chroms)
    is_mito_sym = df_slam["gene_name"].astype(str).str.contains(r"^(?:mt[-_.]|nd[1-6]|cox[1-3]|cytb|atp[68])", case=False, regex=True)
    is_mito = is_mito_c | is_mito_sym
    df_slam["_is_mito"] = is_mito

    # 2. Bleed-through, kinetics, and baseline fold-change flags
    bleed_thresh = float(globals().get("SLAM_MAX_BLEED_RATIO", 0.005))
    min_retention = float(globals().get("SLAM_MIN_KINETICS_RETENTION", 0.5))
    min_base_fc = float(globals().get("SLAM_MIN_BASELINE_FC", 3.0))

    has_nas_075 = "nascent_cpm_0_75hpf" in df_slam.columns
    has_nas_untreated = "nascent_cpm_untreated" in df_slam.columns

    is_bleedthrough = (df_slam["total_cpm_0_75hpf"] >= 5.0) & ((df_slam["nascent_cpm_2_5hpf"] / np.maximum(df_slam["total_cpm_0_75hpf"], 1e-4)) < bleed_thresh)
    df_slam["_is_bleedthrough"] = is_bleedthrough

    max_later = np.maximum(df_slam["nascent_cpm_3_0_3_5hpf"], df_slam["nascent_cpm_4_5hpf"])
    is_collapsing = (df_slam["nascent_cpm_2_5hpf"] > 0) & (max_later < (min_retention * df_slam["nascent_cpm_2_5hpf"]))
    df_slam["_is_collapsing"] = is_collapsing

    if has_nas_075:
        base_ref = df_slam["nascent_cpm_0_75hpf"]
        if has_nas_untreated:
            base_ref = np.maximum(base_ref, df_slam["nascent_cpm_untreated"])
        base_ref = np.maximum(base_ref, 0.05)
        is_low_fc = (df_slam["nascent_cpm_2_5hpf"] > 0) & (df_slam["nascent_cpm_2_5hpf"] < (min_base_fc * base_ref))
    else:
        is_low_fc = pd.Series(False, index=df_slam.index)
    df_slam["_is_low_fc"] = is_low_fc

    is_clean = (~is_mito) & (~is_bleedthrough) & (~is_collapsing) & (~is_low_fc)
    df_slam["_is_clean"] = is_clean

    # --------------------------------------------------------------------------
    # Table 1: Per-Class QC Summary
    # --------------------------------------------------------------------------
    summary_rows = []
    classes_order = ["preMZT_strictly_zygotic", "preMZT_maternal_zygotic", "major_wave_only", "maternal_non_transcribed", "silent_or_low"]
    existing_classes = [c for c in classes_order if c in df_slam["slam_class"].unique()]
    other_classes = [c for c in df_slam["slam_class"].unique() if c not in classes_order]

    for sc in existing_classes + other_classes:
        sub = df_slam[df_slam["slam_class"] == sc]
        n_tot = len(sub)
        n_cln = int(sub["_is_clean"].sum())
        pct_cln = f"{100 * n_cln / max(n_tot, 1):.1f}%"
        n_m = int(sub["_is_mito"].sum())
        n_b = int(sub["_is_bleedthrough"].sum())
        n_fc = int(sub["_is_low_fc"].sum()) if has_nas_075 else 0
        n_col = int(sub["_is_collapsing"].sum())
        summary_rows.append({
            "slam_class": sc,
            "n_total": n_tot,
            "n_clean_nuclear": n_cln,
            "pct_clean": pct_cln,
            "n_mito": n_m,
            "n_maternal_bleedthrough": n_b,
            "n_low_baseline_fc": n_fc,
            "n_collapsing_kinetics": n_col
        })

    df_qc_summary = pd.DataFrame(summary_rows)
    qc_tsv_out = _res_dir / "nascent_rnaseq_qc_summary.tsv"
    df_qc_summary.to_csv(qc_tsv_out, sep="	", index=False)
    print("=" * 80)
    print("[1] SLAM-seq Per-Class QC Summary:")
    print("=" * 80)
    try:
        display(df_qc_summary)
    except NameError:
        print(df_qc_summary.to_string(index=False))
    print(f"[+] Saved QC summary table to {qc_tsv_out}")

    # --------------------------------------------------------------------------
    # Table 2: Top 15 Early-Wave Genes
    # --------------------------------------------------------------------------
    early_mask = df_slam["slam_class"].isin(["preMZT_strictly_zygotic", "preMZT_maternal_zygotic"])
    df_early = df_slam[early_mask].copy()
    df_early["nascent_to_maternal_ratio"] = df_early["nascent_cpm_2_5hpf"] / (df_early["total_cpm_0_75hpf"] + 0.01)

    top15_cols = ["gene_name", "chrom", "slam_class", "total_cpm_0_75hpf"]
    if has_nas_075:
        top15_cols.append("nascent_cpm_0_75hpf")
    top15_cols.extend(["nascent_cpm_2_5hpf", "nascent_cpm_3_0_3_5hpf", "nascent_cpm_4_5hpf", "nascent_to_maternal_ratio"])

    top15 = df_early.sort_values(by="nascent_cpm_2_5hpf", ascending=False).head(15)[top15_cols]
    print("[2] Top 15 Early-Wave Genes (Strictly Zygotic + Maternal-Zygotic):")
    print("=" * 80)
    if not top15.empty:
        try:
            display(top15)
        except NameError:
            print(top15.to_string(index=False))
    else:
        print("    No early-wave genes detected.")

    # --------------------------------------------------------------------------
    # Table 3: Dynamic Marker & Target Gene Audit
    # --------------------------------------------------------------------------
    try:
        from scripts.infer_and_attribute import normalize_teleost_tf_name
    except ImportError:
        def normalize_teleost_tf_name(name: str) -> str:
            raw = str(name).strip().upper()
            return re.sub(r"[abAB]$", "", raw) if len(raw) > 3 else raw

    query_genes = list(globals().get("TARGET_GENES", [])) + list(globals().get("CANONICAL_ZGA_MARKERS", []))
    norm_query_map = {g: normalize_teleost_tf_name(g).lower() for g in query_genes}

    audit_matches = []
    slam_gene_names = df_slam["gene_name"].astype(str).tolist()
    slam_norm_names = [normalize_teleost_tf_name(g).lower() for g in slam_gene_names]

    audit_cols = ["gene_name", "chrom", "slam_class", "total_cpm_0_75hpf"]
    if has_nas_075:
        audit_cols.append("nascent_cpm_0_75hpf")
    audit_cols.extend(["nascent_cpm_2_5hpf", "nascent_cpm_3_0_3_5hpf", "nascent_cpm_4_5hpf"])

    seen_idx = set()
    for qg in query_genes:
        q_lower = qg.lower()
        q_norm = norm_query_map[qg]
        for idx, (g_name, g_norm) in enumerate(zip(slam_gene_names, slam_norm_names)):
            if idx not in seen_idx and (g_name.lower() == q_lower or g_norm == q_norm):
                seen_idx.add(idx)
                row_dict = {c: df_slam.iloc[idx][c] for c in audit_cols}
                row_dict["matched_query"] = qg
                audit_matches.append(row_dict)

    df_audit = pd.DataFrame(audit_matches)
    print("[3] Dynamic Marker & Target Gene Audit (TARGET_GENES + CANONICAL_ZGA_MARKERS):")
    print("=" * 80)
    if not df_audit.empty:
        disp_cols = ["matched_query"] + [c for c in audit_cols if c in df_audit.columns]
        try:
            display(df_audit[disp_cols])
        except NameError:
            print(df_audit[disp_cols].to_string(index=False))
    else:
        print(f"    No direct symbol matches found in SLAM-seq table for {query_genes}.")

    # --------------------------------------------------------------------------
    # Actionable Recommendations Block
    # --------------------------------------------------------------------------
    print("[*] SLAM-seq Filtering Diagnostic Recommendations:")
    print("=" * 80)
    recommendations = []

    early_sub = df_slam[early_mask]
    early_mito_cnt = int(early_sub["_is_mito"].sum()) if not early_sub.empty else 0
    early_col_cnt = int(early_sub["_is_collapsing"].sum()) if not early_sub.empty else 0
    early_bleed_cnt = int(early_sub["_is_bleedthrough"].sum()) if not early_sub.empty else 0
    early_tot = len(early_sub)
    col_pct = (100 * early_col_cnt / early_tot) if early_tot > 0 else 0.0

    if early_mito_cnt > 0:
        recommendations.append(
            f"[ACTION REQUIRED] Detected {early_mito_cnt} mitochondrial gene(s) in early-wave classes! "
            f"Set SLAM_EXCLUDE_MITO = True in Block 0 and re-run prep_slamseq_zf to purge mitochondrial leakage."
        )
    else:
        print("  [PASS] Mitochondrial exclusion: 0 mito contig genes detected in early-wave classes.")

    if col_pct > 20.0 or early_bleed_cnt > 0:
        recommendations.append(
            f"[ACTION REQUIRED] High artifact burden in early-wave classes ({early_bleed_cnt} bleed-through genes, "
            f"{early_col_cnt}/{early_tot} [{col_pct:.1f}%] collapsing kinetics). "
            f"Set SLAM_MASK_BASELINE_SNPS = True and SLAM_MIN_BASELINE_FC = {max(min_base_fc, 3.0):.1f} in Block 0 "
            f"to mask injection-clutch germline SNPs and eliminate false-positive spikes."
        )
    else:
        print(f"  [PASS] Clutch SNP & kinetics filter: Collapsing kinetics at {col_pct:.1f}% (<= 20%), bleed-through: {early_bleed_cnt}.")

    if not df_audit.empty:
        canonical_sub = df_audit[df_audit["matched_query"].str.lower().isin([m.lower() for m in globals().get("CANONICAL_ZGA_MARKERS", [])])]
        if not canonical_sub.empty:
            onset_at_35 = canonical_sub["nascent_cpm_3_0_3_5hpf"] > canonical_sub["nascent_cpm_2_5hpf"]
            if onset_at_35.all():
                print(f"  [INFO] Canonical ZGA markers ({list(canonical_sub['matched_query'].unique())}) display nascent transcription onset primarily at 3.0-3.5 hpf rather than 2.5 hpf (confirming expected minor-to-major wave developmental trajectory).")
            elif onset_at_35.any():
                print(f"  [INFO] Canonical ZGA markers show split activation between 2.5 hpf and 3.0-3.5 hpf.")

    if recommendations:
        for rec in recommendations:
            print(f"  {rec}")
    else:
        print("  [+] All SLAM-seq QC criteria satisfied. Classification is clean and ready for Figure 1b overlay.")
    print("=" * 80)

In [ ]:
import matplotlib.pyplot as plt

# Overlay Zebrafish SLAM-seq Nascent Transcription Timing on Target Species
zf_slam_path = Path(WORK_DIR) / "slamseq" / "zf_slamseq_gene_classes.tsv"
gene_sets_tsv = res_dir / f"{TARGET_LOCUS_NAME}_gene_sets.tsv"
df_counts = pd.read_csv(gene_sets_tsv, sep="\t", low_memory=False)

slam_info = tm.get_task_info("prep_slamseq_zf")
is_slam_running = slam_info.get("status") == "RUNNING"

if is_slam_running or not zf_slam_path.exists():
    if is_slam_running:
        print("[*] Task 'prep_slamseq_zf' is currently RUNNING in background.")
        print(f"    Current progress: {slam_info.get('progress', 'In progress...')}")
        print("    Zebrafish SLAM-seq overlay and Figure 1b will be rendered once the task completes.")
    else:
        print(f"[*] Zebrafish SLAM-seq table not yet generated: {zf_slam_path}.")
    print("    [!] Zebrafish SLAM-seq nascent ortholog overlay is PENDING (prep_slamseq_zf).")
    print("        Setting is_slam_confirmed_minor_wave = False until SLAM-seq finishes and ortholog overlay is evaluated.")
    df_counts["is_slam_confirmed_minor_wave"] = False
    df_counts.to_csv(gene_sets_tsv, sep="\t", index=False)
    rnaseq_tsv = Path(WORK_DIR) / "rnaseq" / "target_species_gene_sets.tsv"
    rnaseq_tsv.parent.mkdir(parents=True, exist_ok=True)
    df_counts.to_csv(rnaseq_tsv, sep="\t", index=False)
else:
    df_zf_slam = pd.read_csv(zf_slam_path, sep="\t")
    if not Path(ortholog_tsv).exists():
        print(f"[!] Ortholog mapping file {ortholog_tsv} not found yet.")
        print("    [!] Setting is_slam_confirmed_minor_wave = False until ortholog overlay is evaluated.")
        df_counts["is_slam_confirmed_minor_wave"] = False
        df_counts.to_csv(gene_sets_tsv, sep="\t", index=False)
        rnaseq_tsv = Path(WORK_DIR) / "rnaseq" / "target_species_gene_sets.tsv"
        rnaseq_tsv.parent.mkdir(parents=True, exist_ok=True)
        df_counts.to_csv(rnaseq_tsv, sep="\t", index=False)
    else:
        df_ortho = pd.read_csv(ortholog_tsv, sep="\t")
        df_ortho_clean = df_ortho.drop_duplicates(subset=["target_gene_id"]).copy()
        ortho_slam = df_ortho_clean.merge(df_zf_slam[["gene_id", "slam_class"]], left_on="zf_gene_id", right_on="gene_id", how="left")
        cols_to_drop = [c for c in ["target_gene_id", "slam_class", "is_slam_confirmed_minor_wave"] if c in df_counts.columns]
        if cols_to_drop:
            df_counts = df_counts.drop(columns=cols_to_drop)
        id_col = "Geneid" if "Geneid" in df_counts.columns else "gene_id"
        df_merged = df_counts.merge(ortho_slam[["target_gene_id", "slam_class"]], left_on=id_col, right_on="target_gene_id", how="left")
        df_merged["slam_class"] = df_merged["slam_class"].fillna("unmapped_ortholog")

        is_slam_confirmed = (
            df_merged["category"].isin(["strictly_zygotic_minor_wave", "strictly_zygotic_highconf", "all_upregulated"]) &
            df_merged["slam_class"].isin(["preMZT_strictly_zygotic", "preMZT_maternal_zygotic"])
        )
        df_merged["is_slam_confirmed_minor_wave"] = is_slam_confirmed
        df_merged.to_csv(gene_sets_tsv, sep="\t", index=False)
        rnaseq_tsv = Path(WORK_DIR) / "rnaseq" / "target_species_gene_sets.tsv"
        rnaseq_tsv.parent.mkdir(parents=True, exist_ok=True)
        df_merged.to_csv(rnaseq_tsv, sep="\t", index=False)
        print(f"[+] Persisted updated gene sets table with SLAM-seq overlay ({is_slam_confirmed.sum()} confirmed genes) to both {gene_sets_tsv} and {rnaseq_tsv}")

        n_mapped = (df_merged["slam_class"] != "unmapped_ortholog").sum()
        print("======================================================================")
        print(f"[*] Zebrafish SLAM-seq Ortholog Mapping Statistics:")
        print(f"    Genes mapped to zebrafish ortholog: {n_mapped} / {len(df_counts)} ({100*n_mapped/len(df_counts):.1f}%)")
        print(df_merged["slam_class"].value_counts().to_string())
        print("======================================================================")

        from scipy.stats import mannwhitneyu, fisher_exact
        mat_lfc = df_merged.loc[df_merged["slam_class"] == "maternal_non_transcribed", "log2fc"].dropna()
        mw_pvals = {}
        for sc in ["preMZT_strictly_zygotic", "preMZT_maternal_zygotic", "major_wave_only"]:
            sc_lfc = df_merged.loc[df_merged["slam_class"] == sc, "log2fc"].dropna()
            if len(sc_lfc) > 0 and len(mat_lfc) > 0:
                stat, pval = mannwhitneyu(sc_lfc, mat_lfc, alternative="greater")
                mw_pvals[sc] = pval
                print(f"    Mann-Whitney U: {sc} vs maternal: p = {pval:.2e}")
            else:
                mw_pvals[sc] = 1.0

        is_up = df_merged["is_all_upregulated"] | df_merged["is_strictly_zygotic_minor_wave"]
        is_pre_active = df_merged["slam_class"].isin(["preMZT_strictly_zygotic", "preMZT_maternal_zygotic"])
        cont_tbl = [
            [(is_up & is_pre_active).sum(), (~is_up & is_pre_active).sum()],
            [(is_up & ~is_pre_active).sum(), (~is_up & ~is_pre_active).sum()]
        ]
        fisher_odds, fisher_p = fisher_exact(cont_tbl, alternative="greater")
        print(f"    Fisher Exact Test (Pre-MZT Active SLAM Orthologs n={is_pre_active.sum()} in Target Upregulated): OR = {fisher_odds:.2f}, p = {fisher_p:.2e}")

        fig, axes = plt.subplots(1, 2, figsize=(15.5, 6.2))
        ax = axes[0]
        slam_palette = {
            "preMZT_strictly_zygotic": "#d7191c",
            "preMZT_maternal_zygotic": "#fdae61",
            "major_wave_only": "#2ca25f",
            "maternal_non_transcribed": "#2b83ba",
            "unmapped_ortholog": "#bdbdbd",
            "silent_or_low": "#969696"
        }
        mean_cpm_t = 0.5 * (df_merged["cpm_early"] + df_merged["cpm_premzt"])
        for sc, sc_col in slam_palette.items():
            sc_mask = df_merged["slam_class"] == sc
            if sc_mask.any():
                ax.scatter(
                    np.log10(mean_cpm_t[sc_mask] + 0.1),
                    df_merged.loc[sc_mask, "log2fc"],
                    color=sc_col,
                    label=f"{sc} (n={int(sc_mask.sum()):,})",
                    s=28 if sc.startswith("preMZT") else 12,
                    alpha=0.88 if sc.startswith("preMZT") else 0.35
                )

        _slam_short = {
            "preMZT_strictly_zygotic": "ZF pre-MZT zygotic",
            "preMZT_maternal_zygotic": "ZF pre-MZT mat+zyg",
            "major_wave_only": "ZF major wave",
            "maternal_non_transcribed": "ZF maternal only",
            "silent_or_low": "ZF silent/low",
            "unmapped_ortholog": "ZF unmapped",
        }
        pts_1b = []
        for gname in TARGET_GENES + CANONICAL_ZGA_MARKERS:
            sub = df_merged[df_merged["gene_name"].str.lower() == gname.lower()]
            if not sub.empty:
                r = sub.iloc[0]
                mx = float(np.log10(0.5 * (r["cpm_early"] + r["cpm_premzt"]) + 0.1))
                my = float(r["log2fc"])
                ax.scatter([mx], [my], color="black", edgecolor="white", lw=0.8, s=75, zorder=6)
                sc_lbl = _slam_short.get(str(r["slam_class"]), str(r["slam_class"]).replace("_", " "))
                pts_1b.append((f"{gname} ({sc_lbl})", mx, my))
        placed_1b = []
        offs_1b = [(12, 10), (-150, 22), (10, -24), (-145, -24), (16, 26), (-12, 34)]
        for lbl, mx, my in sorted(pts_1b, key=lambda t: (-t[2], t[1])):
            chosen = offs_1b[0]
            best_c = 1e9
            for ox, oy in offs_1b:
                ha_t = "right" if ox < 0 else "left"
                cx = mx + ox * 0.009 + (-0.65 if ox < 0 else 0.65)
                cy = my + oy * 0.045
                cost = 0.0
                if cx < -0.7 or cx > 4.2 or cy < -8.0 or cy > 9.5:
                    cost += 50.0
                for px, py in placed_1b:
                    if abs(cx - px) < 1.35 and abs(cy - py) < 1.35:
                        cost += 2.0 - abs(cy - py)
                if cost < best_c:
                    best_c = cost
                    chosen = (ox, oy)
            placed_1b.append((mx + chosen[0] * 0.009 + (-0.65 if chosen[0] < 0 else 0.65), my + chosen[1] * 0.045))
            ax.annotate(lbl, (mx, my), xytext=chosen, textcoords="offset points", fontweight="bold", fontsize=8.3,
                        ha="right" if chosen[0] < 0 else "left", va="center",
                        bbox=dict(boxstyle="round,pad=0.16", fc="white", ec="#404040", alpha=0.90, lw=0.6),
                        arrowprops=dict(arrowstyle="-", color="#404040", lw=0.7), zorder=7)

        ax.set_xlabel("log10(Mean Target Expression + 0.1)", fontsize=11)
        ax.set_ylabel("Target log2 Fold Change (1000-cell vs. 2-cell)", fontsize=11)
        ax.set_title("A. Target RNA-seq MA Plot Colored by ZF SLAM-seq Timing", fontweight="bold", fontsize=11.5)
        ax.axhline(0, color="gray", linestyle="--", alpha=0.5)
        ax.legend(frameon=True, fontsize=8.0, loc="upper left")
        ax.grid(True, linestyle=":", alpha=0.5)

        ax = axes[1]
        classes_to_plot = ["preMZT_strictly_zygotic", "preMZT_maternal_zygotic", "major_wave_only", "maternal_non_transcribed"]
        colors_to_plot = ["#d7191c", "#fdae61", "#2ca25f", "#2b83ba"]
        lfc_by_class = [df_merged.loc[df_merged["slam_class"] == sc, "log2fc"].dropna().values for sc in classes_to_plot]
        labels_to_plot = [
            f"preMZT Strictly\nZygotic (n={len(lfc_by_class[0])})\n(p={mw_pvals.get('preMZT_strictly_zygotic', 1.0):.1e})",
            f"preMZT Maternal-\nZygotic (n={len(lfc_by_class[1])})\n(p={mw_pvals.get('preMZT_maternal_zygotic', 1.0):.1e})",
            f"Major Wave\nOnly (n={len(lfc_by_class[2])})\n(p={mw_pvals.get('major_wave_only', 1.0):.1e})",
            f"Maternal Non-\nTranscribed (n={len(lfc_by_class[3]):,})\n(Control)"
        ]
        valid_indices = [i for i, d in enumerate(lfc_by_class) if len(d) > 0]
        if valid_indices:
            violin_idx = [i for i in valid_indices if len(lfc_by_class[i]) >= 3]
            if violin_idx:
                filtered_lfc = [lfc_by_class[i] for i in violin_idx]
                parts = ax.violinplot(filtered_lfc, positions=[i + 1 for i in violin_idx], showmeans=True, showmedians=True)
                for i_idx, p_idx in enumerate(violin_idx):
                    if i_idx < len(parts['bodies']):
                        parts['bodies'][i_idx].set_facecolor(colors_to_plot[p_idx])
                        parts['bodies'][i_idx].set_edgecolor('black')
                        parts['bodies'][i_idx].set_alpha(0.75)
            for p_idx in valid_indices:
                vals_p = lfc_by_class[p_idx]
                if len(vals_p) < 10:
                    ax.scatter([p_idx + 1] * len(vals_p), vals_p, color=colors_to_plot[p_idx], edgecolor="black", s=65, zorder=5)
            ax.set_xticks([i + 1 for i in valid_indices])
            ax.set_xticklabels([labels_to_plot[i] for i in valid_indices], fontsize=9.0)
            ax.set_ylabel("log2 Fold Change in Target RNA-seq", fontsize=11)
            ax.set_title(
                f"B. Target log2FC Across Orthologous ZF SLAM-seq Classes\n"
                f"(Pre-MZT Active in Target Upregulated: OR={fisher_odds:.2f}, p={fisher_p:.2e})",
                fontweight="bold", fontsize=11.0
            )
            ax.axhline(0, color="gray", linestyle="--", alpha=0.5)
            ax.grid(True, linestyle=":", alpha=0.5)

        plt.tight_layout()
        fig1b_path = res_dir / "fig1b_slamseq_ortholog_overlay.png"
        plt.savefig(fig1b_path, dpi=300, bbox_inches="tight")
        plt.show()
        print(f"[+] Figure 1b exported to {fig1b_path}")


---
## Block 2: Background ATAC-seq Preprocessing & ChromBPNet Base Training

Trains Stage 1 Tn5-debiased **ChromBPNet** accessibility models on early- vs. late-blastula ATAC-seq datasets:
1. **Native Poisson Peak Calling & Targets (`scripts/prep_atac.py`):** Collects strand-shifted Tn5 cut sites (+4/-5 bp), calls late-blastula peaks using native local-lambda Poisson enrichment (MACS2/Poisson logic), and quantifies early vs. late cut densities ($y_{\text{abs}}, y_{\text{rel}}$) at each summit.
2. **Leak-Free Chromosome Holdout:** Partitions assembled primary chromosomes into 5 cross-validation folds (`~64% train / 12% valid / 24% test` in length and chromosome count), ensuring the chromosome orthologous to `TARGET_LOCUS_NAME` resides strictly in `fold_0` `test`. Both ChromBPNet and `NTv3` share this exact chromosome holdout.
3. **Stage 1 Bias Factorization (`scripts/train_chrombpnet.py`):** Trains a **Tn5 bias model** on GC-matched closed `nonpeaks`, then freezes that bias subnetwork while training the regulatory **`chrombpnet_nobias`** subnetwork (`full = tf_model + frozen_bias`) on open blastula peaks—forcing `nobias` to isolate TF sequence footprints while removing Tn5 cutting artifacts (`max_tn5_footprint_response < 0.003`). Downstream inference gates on library TSSe, count Pearson $r$, and profile JSD (`MEDAKA_PASSED_QC`).

**Generated Outputs (Block 2):**
* **QC Tables (`atac_sample_qc.tsv`, `atac_species_qc.tsv`, `chrombpnet_qc.tsv`):** Per-library alignment/TSSe metrics, peak statistics, count Pearson $r$, profile median JSD, and marginal Tn5 footprint responses.
* **Trained Base Checkpoints (`WORK_DIR/models/chrombpnet/<species>/fold_*/`):** Debiased Stage 1 weights (`*_nobias.pt`) and evaluation summaries.

In [ ]:
# Launch Background ATAC Preprocessing and ChromBPNet Training for Both Zebrafish & Medaka
atac_manifest = f"{WORK_DIR}/downloads/atac_fastq_manifest.tsv"

# 1. Submit Zebrafish ATAC Preprocessing & ChromBPNet Pipeline
zf_atac_cmd = [
    "python", "-u", "scripts/prep_atac.py",
    "--species", "Danio_rerio",
    "--work-dir", WORK_DIR,
    "--genome-fa", ZEBRAFISH_GENOME_FA,
    "--gtf", ZEBRAFISH_GTF,
    "--fastq-manifest", atac_manifest,
    "--target-genes", *TARGET_GENES,
    "--cv-folds", str(CV_FOLDS),
    "--task-name", "prep_atac_danio_rerio",
    "--threads", str(THREADS)
]
if EXCLUDE_HOUSEKEEPING_PROMOTERS_STAGE2:
    zf_atac_cmd.append("--exclude-housekeeping-promoters")

tm.submit("prep_atac_danio_rerio", zf_atac_cmd, depends_on="download_resources", force_rerun=FORCE_RERUN)

zf_chrom_sizes = f"{WORK_DIR}/atac/Danio_rerio/chrom.sizes"
zf_peaks_bed = f"{WORK_DIR}/atac/Danio_rerio/peaks/Danio_rerio_late_peaks.primary.narrowPeak"
zf_late_bam = f"{WORK_DIR}/atac/Danio_rerio/Danio_rerio_late_merged.bam"

FOLDS_TO_TRAIN = [0] if COMPUTE_PROFILE == "laptop_16gb" else list(range(CV_FOLDS))

for fold in FOLDS_TO_TRAIN:
    zf_chr_folds = f"{WORK_DIR}/atac/Danio_rerio/chr_folds.json"
    zf_chrombpnet_cmd = [
        "python", "-u", "scripts/train_chrombpnet.py",
        "--species", "Danio_rerio",
        "--work-dir", WORK_DIR,
        "--genome-fa", ZEBRAFISH_GENOME_FA,
        "--chrom-sizes", zf_chrom_sizes,
        "--peaks-bed", zf_peaks_bed,
        "--late-bam", zf_late_bam,
        "--chr-fold-json", zf_chr_folds,
        "--fold", str(fold),
        "--task-name", f"train_chrombpnet_danio_rerio_fold_{fold}",
        "--epochs", str(CHROMBPNET_EPOCHS),
        "--threads", str(THREADS)
    ]
    if USE_PRETRAINED_BIAS:
        bias_h5 = BIAS_MODEL_PATH or f"{WORK_DIR}/downloads/bias_models/atac_bias.h5"
        zf_chrombpnet_cmd.extend(["--use-pretrained-bias", "--bias-model-path", bias_h5])

    tm.submit(f"train_chrombpnet_danio_rerio_fold_{fold}", zf_chrombpnet_cmd, depends_on="prep_atac_danio_rerio", force_rerun=FORCE_RERUN, requires_gpu=True)

# 2. Submit Medaka ATAC Preprocessing & ChromBPNet Pipeline (when enabled)
if ENABLE_MEDAKA:
    medaka_atac_cmd = [
        "python", "-u", "scripts/prep_atac.py",
        "--species", "Oryzias_latipes",
        "--work-dir", WORK_DIR,
        "--genome-fa", MEDAKA_GENOME_FA,
        "--gtf", MEDAKA_GTF,
        "--fastq-manifest", atac_manifest,
        "--target-genes", *TARGET_GENES,
        "--cv-folds", str(CV_FOLDS),
        "--task-name", "prep_atac_medaka",
        "--threads", str(THREADS)
    ]
    if EXCLUDE_HOUSEKEEPING_PROMOTERS_STAGE2:
        medaka_atac_cmd.append("--exclude-housekeeping-promoters")

    tm.submit("prep_atac_medaka", medaka_atac_cmd, depends_on="download_resources", force_rerun=FORCE_RERUN)

    med_chrom_sizes = f"{WORK_DIR}/atac/Oryzias_latipes/chrom.sizes"
    med_peaks_bed = f"{WORK_DIR}/atac/Oryzias_latipes/peaks/Oryzias_latipes_late_peaks.primary.narrowPeak"
    med_late_bam = f"{WORK_DIR}/atac/Oryzias_latipes/Oryzias_latipes_late_merged.bam"

    for fold in FOLDS_TO_TRAIN:
        med_chr_folds = f"{WORK_DIR}/atac/Oryzias_latipes/chr_folds.json"
        medaka_chrombpnet_cmd = [
            "python", "-u", "scripts/train_chrombpnet.py",
            "--species", "Oryzias_latipes",
            "--work-dir", WORK_DIR,
            "--genome-fa", MEDAKA_GENOME_FA,
            "--chrom-sizes", med_chrom_sizes,
            "--peaks-bed", med_peaks_bed,
            "--late-bam", med_late_bam,
            "--chr-fold-json", med_chr_folds,
            "--fold", str(fold),
            "--task-name", f"train_chrombpnet_medaka_fold_{fold}",
            "--epochs", str(CHROMBPNET_EPOCHS),
            "--threads", str(THREADS)
        ]
        if USE_PRETRAINED_BIAS:
            bias_h5 = BIAS_MODEL_PATH or f"{WORK_DIR}/downloads/bias_models/atac_bias.h5"
            medaka_chrombpnet_cmd.extend(["--use-pretrained-bias", "--bias-model-path", bias_h5])

        tm.submit(f"train_chrombpnet_medaka_fold_{fold}", medaka_chrombpnet_cmd, depends_on="prep_atac_medaka", force_rerun=FORCE_RERUN, requires_gpu=True)

display(tm.status())

In [ ]:
# ==============================================================================
# Cross-Sample Alignment & ChromBPNet Base Model Quality Audit (Block 2 Summary)
# ==============================================================================
# Audits ATAC-seq preprocessing, trimming, Bowtie2 alignment, peak calling,
# chromosome split integrity, and ChromBPNet Stage 1 base models across all
# samples, stages, and species dynamically defined in Block 0 and manifests.
import os
import sys
import json
import re
import shutil
import subprocess
from pathlib import Path
import pandas as pd
import numpy as np
import pysam

# Ensure active conda environment bin is available for bioinformatics CLIs
bin_dir = str(Path(sys.prefix) / "bin")
if bin_dir not in os.environ.get("PATH", ""):
    os.environ["PATH"] = f"{bin_dir}:{os.environ.get('PATH', '')}"

from scripts.train_chrombpnet import refresh_training_meta

work_dir = Path(WORK_DIR).resolve()
out_results_dir = Path(res_dir).resolve()
out_results_dir.mkdir(parents=True, exist_ok=True)

manifest_path = work_dir / "downloads" / "atac_fastq_manifest.tsv"
df_manifest = pd.read_csv(manifest_path, sep="\t") if manifest_path.exists() else pd.DataFrame()

# ------------------------------------------------------------------------------
# Part A: Per-Sample ATAC-seq Trimming & Alignment QC Table
# ------------------------------------------------------------------------------
log_files = list((work_dir / "tasks").glob("prep_atac_*/*.log"))
log_corpus = {lf.name: lf.read_text(errors="replace") for lf in log_files}

sample_qc_rows = []

if not df_manifest.empty:
    for _, row in df_manifest.iterrows():
        sp = str(row["species"]).strip()
        st = str(row["stage"]).strip()
        run_id = str(row["run_id"]).strip()
        layout = str(row.get("layout", "UNKNOWN")).strip()

        # Discover fastp report beside FASTQ
        r1_path_str = str(row.get("fastq_r1", "")).strip()
        p_r1 = Path(r1_path_str) if r1_path_str else None
        fastp_candidates = [
            work_dir / "atac" / sp / "fastqs" / f"{run_id}_fastp.json",
            work_dir / "atac" / sp / "fastqs" / f"{run_id}_1_fastp.json"
        ]
        if p_r1 and p_r1.parent.exists():
            fastp_candidates.extend([
                p_r1.parent / f"{run_id}_fastp.json",
                p_r1.parent / f"{run_id}_1_fastp.json",
                p_r1.parent / f"{p_r1.stem}_fastp.json"
            ])

        fastp_json = None
        for cand in fastp_candidates:
            if cand.exists() and cand.stat().st_size > 0:
                fastp_json = cand
                break

        raw_reads, filtered_reads = None, None
        pass_filter_pct, q30_after_pct = None, None
        adapter_trimmed_pct, duplication_pct, insert_size_peak = None, None, None

        if fastp_json:
            try:
                with open(fastp_json, "r") as fp:
                    data_fp = json.load(fp)
                summary = data_fp.get("summary", {})
                bf = summary.get("before_filtering", {})
                af = summary.get("after_filtering", {})
                fr = data_fp.get("filtering_result", {})
                dup = data_fp.get("duplication", {})
                ins = data_fp.get("insert_size", {})

                raw_reads = bf.get("total_reads")
                filtered_reads = af.get("total_reads")
                if raw_reads and raw_reads > 0 and filtered_reads is not None:
                    pass_filter_pct = (filtered_reads / raw_reads) * 100.0
                if af and "q30_rate" in af:
                    q30_after_pct = af["q30_rate"] * 100.0
                ad_trimmed_cnt = data_fp.get("adapter_cutting", {}).get("adapter_trimmed_reads") or fr.get("reads_with_adapter_trimmed", 0)
                if raw_reads and raw_reads > 0 and ad_trimmed_cnt is not None:
                    adapter_trimmed_pct = (ad_trimmed_cnt / raw_reads) * 100.0
                if dup and "rate" in dup:
                    duplication_pct = dup["rate"] * 100.0
                if ins and isinstance(ins, dict):
                    insert_size_peak = ins.get("peak")
            except Exception:
                pass

        # Bowtie2 log scraping
        bt2_overall_pct, bt2_uniq_pct = None, None
        for _, ltext in log_corpus.items():
            if run_id in ltext:
                m_bt2 = re.search(r"bowtie2[^\r\n]*?" + re.escape(run_id) + r"[^\r\n]*?\n([\s\S]*?overall alignment rate)", ltext)
                if m_bt2:
                    block = m_bt2.group(1)
                    m_over = re.search(r"([0-9\.]+)%\s+overall alignment rate", block)
                    m_uniq = re.search(r"\(\s*([0-9\.]+)%\s*\)\s*aligned\s+(?:concordantly\s+)?exactly 1 time", block)
                    if m_over:
                        bt2_overall_pct = float(m_over.group(1))
                    if m_uniq:
                        bt2_uniq_pct = float(m_uniq.group(1))
                    break

        # Final BAM metrics
        bam_path = work_dir / "atac" / sp / "bams" / f"{run_id}.sorted.bam"
        final_mapped_reads, usable_pct = None, None
        status = "PENDING"

        if bam_path.exists():
            try:
                res_idx = subprocess.run(["samtools", "idxstats", str(bam_path)], capture_output=True, text=True, check=True)
                mapped_cnt = 0
                for line in res_idx.stdout.strip().split("\n"):
                    if not line:
                        continue
                    parts = line.split("\t")
                    if len(parts) >= 3 and not (parts[0].startswith("chrM") or parts[0] == "MT"):
                        mapped_cnt += int(parts[2])
                final_mapped_reads = mapped_cnt
                if filtered_reads and filtered_reads > 0:
                    usable_pct = (final_mapped_reads / filtered_reads) * 100.0
                status = "COMPLETED"
            except Exception:
                status = "FAILED"
        elif (work_dir / "tasks" / f"prep_atac_{sp}" / "running.txt").exists():
            status = "RUNNING"

        sample_qc_rows.append({
            "species": sp,
            "stage": st,
            "run_id": run_id,
            "layout": layout,
            "status": status,
            "raw_reads": raw_reads,
            "filtered_reads": filtered_reads,
            "pass_filter_pct": round(pass_filter_pct, 2) if pass_filter_pct is not None else None,
            "q30_after_pct": round(q30_after_pct, 2) if q30_after_pct is not None else None,
            "adapter_trimmed_pct": round(adapter_trimmed_pct, 2) if adapter_trimmed_pct is not None else None,
            "duplication_pct": round(duplication_pct, 2) if duplication_pct is not None else None,
            "insert_size_peak": insert_size_peak,
            "bt2_overall_align_pct": bt2_overall_pct,
            "bt2_unique_align_pct": bt2_uniq_pct,
            "final_mapped_reads": final_mapped_reads,
            "usable_fraction_pct": round(usable_pct, 2) if usable_pct is not None else None
        })

df_sample_qc = pd.DataFrame(sample_qc_rows)

# ------------------------------------------------------------------------------
# Part B: Per-Species Stage Merge, Peak Calling & Split Summary Table
# ------------------------------------------------------------------------------
atac_root = work_dir / "atac"
species_qc_rows = []
species_candidates = [d.name for d in atac_root.iterdir() if d.is_dir()] if atac_root.exists() else []

for sp in sorted(species_candidates):
    sp_dir = atac_root / sp
    early_bam = sp_dir / f"{sp}_early_merged.bam"
    late_bam = sp_dir / f"{sp}_late_merged.bam"

    early_merged_reads = 0
    if early_bam.exists():
        try:
            early_merged_reads = sum(int(line.split("\t")[2]) for line in pysam.idxstats(str(early_bam)).strip().split("\n") if line and not (line.split("\t")[0] in ["chrM", "MT"]))
        except Exception:
            pass

    late_merged_reads = 0
    if late_bam.exists():
        try:
            late_merged_reads = sum(int(line.split("\t")[2]) for line in pysam.idxstats(str(late_bam)).strip().split("\n") if line and not (line.split("\t")[0] in ["chrM", "MT"]))
        except Exception:
            pass

    # Peak calling metrics
    peaks_narrow = sp_dir / "peaks" / f"{sp}_late_peaks.narrowPeak"
    if not peaks_narrow.exists():
        peaks_narrow = sp_dir / f"{sp}_late_peaks.narrowPeak"
    primary_peaks_narrow = sp_dir / "peaks" / f"{sp}_late_peaks.primary.narrowPeak"
    if not primary_peaks_narrow.exists():
        primary_peaks_narrow = sp_dir / f"{sp}_late_peaks.primary.narrowPeak"

    total_peaks = 0
    if peaks_narrow.exists():
        try:
            total_peaks = sum(1 for line in open(peaks_narrow) if line.strip() and not line.startswith("#"))
        except Exception:
            pass

    primary_peaks_kept = 0
    peak_widths = []
    if primary_peaks_narrow.exists():
        with open(primary_peaks_narrow, "r") as pf:
            for line in pf:
                if line.strip() and not line.startswith("#"):
                    p_cols = line.strip().split("\t")
                    if len(p_cols) >= 3:
                        primary_peaks_kept += 1
                        peak_widths.append(int(p_cols[2]) - int(p_cols[1]))
    median_peak_width_bp = float(np.median(peak_widths)) if peak_widths else None

    # FRiP, Mito %, and TSSe caching mechanism for fast notebook re-runs
    qc_cache_file = sp_dir / "qc_summary.json"
    late_frip = None
    late_reads_in_peaks = None
    mito_pct = 0.0
    early_tsse = None
    late_tsse = None

    if qc_cache_file.exists():
        try:
            with open(qc_cache_file, "r") as fp:
                cached_qc = json.load(fp)
            late_reads_in_peaks = cached_qc.get("late_reads_in_peaks")
            late_frip = cached_qc.get("late_frip")
            mito_pct = cached_qc.get("mito_read_pct", 0.0)
            early_tsse = cached_qc.get("early_tss_enrichment")
            late_tsse = cached_qc.get("late_tss_enrichment")
        except Exception:
            pass

    if late_frip is None and late_bam.exists() and peaks_narrow.exists() and late_merged_reads > 0:
        try:
            cmd_frip = ["samtools", "view", "-@", "8", "-c", "-L", str(peaks_narrow), str(late_bam)]
            res_frip = subprocess.run(cmd_frip, capture_output=True, text=True, check=True)
            late_reads_in_peaks = int(res_frip.stdout.strip())
            late_frip = late_reads_in_peaks / late_merged_reads
            with open(qc_cache_file, "w") as fp:
                json.dump({
                    "late_reads_in_peaks": late_reads_in_peaks,
                    "late_frip": late_frip,
                    "late_merged_reads": late_merged_reads,
                    "mito_read_pct": mito_pct,
                    "early_tss_enrichment": early_tsse,
                    "late_tss_enrichment": late_tsse
                }, fp, indent=2)
        except Exception:
            pass

    # Repeat downweight summits
    summit_targets_file = sp_dir / f"{sp}_summit_targets.tsv"
    repeat_downweight_summits = None
    if summit_targets_file.exists():
        try:
            df_st = pd.read_csv(summit_targets_file, sep="\t", low_memory=False)
            if "is_repeat_downweight" in df_st.columns:
                repeat_downweight_summits = int((df_st["is_repeat_downweight"] == True).sum())
            elif "repeat_downweighted" in df_st.columns:
                repeat_downweight_summits = int((df_st["repeat_downweighted"] == True).sum())
        except Exception:
            pass

    # Chromosome splits
    chr_fold_file = sp_dir / "chr_folds.json"
    if not chr_fold_file.exists():
        chr_fold_file = sp_dir / "chr_fold.json"
    n_train_chroms, n_valid_chroms, n_test_chroms = None, None, None
    test_chroms_str = ""
    if chr_fold_file.exists():
        try:
            with open(chr_fold_file, "r") as fp:
                cf_data = json.load(fp)
            fold_split = cf_data.get("fold_0", cf_data.get("0", cf_data))
            t_c = fold_split.get("train", [])
            v_c = fold_split.get("valid", fold_split.get("val", []))
            te_c = fold_split.get("test", [])
            n_train_chroms = len(t_c)
            n_valid_chroms = len(v_c)
            n_test_chroms = len(te_c)
            test_chroms_str = ", ".join(str(c) for c in te_c[:5]) + (f" (+{len(te_c)-5} more)" if len(te_c) > 5 else "")
        except Exception:
            pass

    # Replicate depth balance ratio
    early_rep_reads, late_rep_reads = [], []
    if not df_manifest.empty:
        df_sp_man = df_manifest[df_manifest["species"] == sp]
        for _, srow in df_sp_man.iterrows():
            s_id = str(srow["run_id"]).strip()
            s_bam = sp_dir / "bams" / f"{s_id}.sorted.bam"
            if s_bam.exists():
                s_cnt = sum(int(line.split("\t")[2]) for line in pysam.idxstats(str(s_bam)).strip().split("\n") if line)
                if "early" in str(srow["stage"]).lower():
                    early_rep_reads.append(s_cnt)
                elif "late" in str(srow["stage"]).lower():
                    late_rep_reads.append(s_cnt)

    early_balance = (max(early_rep_reads) / min(early_rep_reads)) if len(early_rep_reads) > 1 and min(early_rep_reads) > 0 else 1.0
    late_balance = (max(late_rep_reads) / min(late_rep_reads)) if len(late_rep_reads) > 1 and min(late_rep_reads) > 0 else 1.0
    rep_depth_balance = max(early_balance, late_balance)

    species_qc_rows.append({
        "species": sp,
        "early_merged_reads": early_merged_reads,
        "late_merged_reads": late_merged_reads,
        "late_frip": round(late_frip, 4) if late_frip is not None else None,
        "total_peaks": total_peaks,
        "primary_peaks_kept": primary_peaks_kept,
        "median_peak_width_bp": median_peak_width_bp,
        "repeat_downweight_summits": repeat_downweight_summits,
        "mito_read_pct": round(mito_pct, 2) if mito_pct is not None else 0.0,
        "early_tss_enrichment": round(early_tsse, 2) if early_tsse is not None else None,
        "late_tss_enrichment": round(late_tsse, 2) if late_tsse is not None else None,
        "n_train_chroms": n_train_chroms,
        "n_valid_chroms": n_valid_chroms,
        "n_test_chroms": n_test_chroms,
        "test_chroms": test_chroms_str,
        "rep_depth_balance_ratio": round(rep_depth_balance, 2)
    })

df_species_qc = pd.DataFrame(species_qc_rows)

# ------------------------------------------------------------------------------
# Part C: Cross-Species ChromBPNet Stage 1 Model QC Table
# ------------------------------------------------------------------------------
models_root = work_dir / "models" / "chrombpnet"
chrombpnet_qc_rows = []

if models_root.exists():
    for fold_dir in sorted(models_root.glob("*/fold_*")):
        if not fold_dir.is_dir():
            continue
        sp_name = fold_dir.parent.name
        fold_idx = fold_dir.name.replace("fold_", "")

        meta_file = fold_dir / "training_meta.json"
        meta_dict = {}
        if meta_file.exists():
            try:
                with open(meta_file, "r") as fp:
                    meta_dict = json.load(fp)
            except Exception:
                pass

        # Refresh metrics automatically if provenance source_files is missing
        if not meta_dict.get("qc_metrics", {}).get("source_files"):
            meta_dict = refresh_training_meta(fold_dir)

        qc_dict = meta_dict.get("qc_metrics", {})

        # Checkpoint discovery & size in MB
        ckpt_path = meta_dict.get("checkpoint")
        ckpt_size_mb = None
        if ckpt_path and Path(ckpt_path).exists():
            ckpt_size_mb = round(Path(ckpt_path).stat().st_size / (1024 * 1024), 2)
        else:
            cand_ckpts = list((fold_dir / "models").glob("*nobias*"))
            if cand_ckpts and cand_ckpts[0].exists():
                ckpt_size_mb = round(cand_ckpts[0].stat().st_size / (1024 * 1024), 2)

        model_status = "PENDING"
        if (fold_dir / ".train.done").exists() or ckpt_size_mb is not None:
            model_status = "COMPLETED"
        elif (fold_dir / "bias_run" / ".bias.done").exists() or (fold_dir / "bias_run" / "models" / "bias.pt").exists():
            model_status = "BIAS_ONLY"

        chrombpnet_qc_rows.append({
            "species": sp_name,
            "fold": fold_idx,
            "status": model_status,
            "count_pearson_r": round(qc_dict["count_pearson_r"], 4) if qc_dict.get("count_pearson_r") is not None else None,
            "profile_pearson_r": round(qc_dict["profile_pearson_r"], 4) if qc_dict.get("profile_pearson_r") is not None else None,
            "profile_median_jsd": round(qc_dict["profile_median_jsd"], 4) if qc_dict.get("profile_median_jsd") is not None else None,
            "nobias_count_pearson_r": round(qc_dict["nobias_count_pearson_r"], 4) if qc_dict.get("nobias_count_pearson_r") is not None else None,
            "nobias_profile_jsd": round(qc_dict["nobias_profile_median_jsd"], 4) if qc_dict.get("nobias_profile_median_jsd") is not None else None,
            "max_tn5_footprint_response": round(qc_dict["max_tn5_footprint_response"], 4) if qc_dict.get("max_tn5_footprint_response") is not None else None,
            "bias_counts_pearson_r": round(qc_dict["bias_counts_pearson_r"], 4) if qc_dict.get("bias_counts_pearson_r") is not None else None,
            "bias_profile_jsd": round(qc_dict["bias_profile_jsd"], 4) if qc_dict.get("bias_profile_jsd") is not None else None,
            "bias_nonpeaks_counts_pearson_r": round(qc_dict["bias_nonpeaks_counts_pearson_r"], 4) if qc_dict.get("bias_nonpeaks_counts_pearson_r") is not None else None,
            "checkpoint_mb": ckpt_size_mb
        })

df_cbp_qc = pd.DataFrame(chrombpnet_qc_rows)

# ------------------------------------------------------------------------------
# Part D: Automated QC Warning Engine
# ------------------------------------------------------------------------------
qc_flags = []

# Trimming & Alignment checks
if not df_sample_qc.empty:
    for _, s_row in df_sample_qc.iterrows():
        s_id, s_sp = s_row["run_id"], s_row["species"]
        if s_row["q30_after_pct"] is not None and s_row["q30_after_pct"] < 85.0:
            qc_flags.append(f"[WARNING] Trimming: Sample {s_id} ({s_sp}) Q30 rate = {s_row['q30_after_pct']:.1f}% (< 85%)")
        if s_row["bt2_overall_align_pct"] is not None and s_row["bt2_overall_align_pct"] < 60.0:
            qc_flags.append(f"[WARNING] Alignment: Sample {s_id} ({s_sp}) Bowtie2 overall alignment = {s_row['bt2_overall_align_pct']:.1f}% (< 60%)")
        if s_row["usable_fraction_pct"] is not None and s_row["usable_fraction_pct"] < 20.0:
            qc_flags.append(f"[WARNING] Usability: Sample {s_id} ({s_sp}) usable fraction = {s_row['usable_fraction_pct']:.1f}% (< 20%)")

# Peak & Split checks
if not df_species_qc.empty:
    for _, sp_row in df_species_qc.iterrows():
        s_sp = sp_row["species"]
        if sp_row["rep_depth_balance_ratio"] is not None and sp_row["rep_depth_balance_ratio"] > 5.0:
            qc_flags.append(f"[WARNING] Depth Balance: Species {s_sp} replicate depth imbalance = {sp_row['rep_depth_balance_ratio']:.1f}x (> 5.0x)")
        if sp_row["late_merged_reads"] < 10000000:
            qc_flags.append(f"[WARNING] Depth: Species {s_sp} late blastula merged depth = {sp_row['late_merged_reads']:,} (< 10M reads)")
        if sp_row["early_merged_reads"] < 5000000:
            qc_flags.append(f"[WARNING] Depth: Species {s_sp} early blastula merged depth = {sp_row['early_merged_reads']:,} (< 5M reads)")
        if sp_row["late_frip"] is not None and sp_row["late_frip"] < 0.10:
            qc_flags.append(f"[WARNING] Enrichment: Species {s_sp} late blastula FRiP = {sp_row['late_frip']*100:.2f}% (< 10% expected)")
        if sp_row["primary_peaks_kept"] < 5000:
            qc_flags.append(f"[WARNING] Peaks: Species {s_sp} primary peaks kept = {sp_row['primary_peaks_kept']:,} (< 5,000)")
        if sp_row["n_test_chroms"] == 0:
            qc_flags.append(f"[CRITICAL] Splits: Species {s_sp} test chromosome split is empty!")
        if sp_row.get("mito_read_pct") is not None and sp_row["mito_read_pct"] > 2.0:
            qc_flags.append(f"[CRITICAL] Mitochondrial Contamination: Species {s_sp} mito reads = {sp_row['mito_read_pct']:.2f}% (> 2.0%)")
        if sp_row.get("late_tss_enrichment") is not None:
            if sp_row["late_tss_enrichment"] < 4.0:
                qc_flags.append(f"[CRITICAL] TSS Enrichment: Species {s_sp} late blastula TSSe = {sp_row['late_tss_enrichment']:.2f} (< 4.0, failed enrichment)")
            elif sp_row["late_tss_enrichment"] < 6.0:
                qc_flags.append(f"[WARNING] TSS Enrichment: Species {s_sp} late blastula TSSe = {sp_row['late_tss_enrichment']:.2f} (< 6.0 expected)")

# ChromBPNet checks
if not df_cbp_qc.empty:
    for _, m_row in df_cbp_qc.iterrows():
        m_sp, m_fld = m_row["species"], m_row["fold"]
        c_pr = m_row["count_pearson_r"]
        nb_pr = m_row.get("nobias_count_pearson_r")
        b_pr = m_row["bias_counts_pearson_r"]
        p_jsd = m_row["profile_median_jsd"]
        b_np_pr = m_row["bias_nonpeaks_counts_pearson_r"]
        max_resp = m_row.get("max_tn5_footprint_response")

        # Sanity collision check
        if c_pr is not None and b_pr is not None and abs(c_pr - b_pr) < 1e-9:
            qc_flags.append(f"[CRITICAL] Collision: Model {m_sp} fold {m_fld} main count Pearson r and bias counts Pearson r are identical ({c_pr:.6f})")
        if c_pr is not None:
            if c_pr <= 0.0:
                qc_flags.append(f"[CRITICAL] Model Performance: Model {m_sp} fold {m_fld} main count Pearson r is non-positive ({c_pr:.4f})")
            elif c_pr < 0.50:
                qc_flags.append(f"[WARNING] Model Performance: Model {m_sp} fold {m_fld} main count Pearson r = {c_pr:.4f} (< 0.50)")
        if nb_pr is not None and c_pr is not None and (c_pr - nb_pr) > 0.10:
            qc_flags.append(f"[CRITICAL] Model Degradation: Model {m_sp} fold {m_fld} count Pearson r dropped by > 0.10 after debiasing ({c_pr:.4f} -> {nb_pr:.4f})")
        if p_jsd is not None and p_jsd > 0.65:
            qc_flags.append(f"[WARNING] Model Resolution: Model {m_sp} fold {m_fld} profile median JSD = {p_jsd:.4f} (> 0.65)")
        if b_pr is not None:
            if b_pr > 0.50:
                qc_flags.append(f"[CRITICAL] Residual Bias: Model {m_sp} fold {m_fld} Tn5 bias counts Pearson r = {b_pr:.4f} (> 0.50)")
            elif b_pr > 0.30:
                qc_flags.append(f"[WARNING] Residual Bias: Model {m_sp} fold {m_fld} Tn5 bias counts Pearson r = {b_pr:.4f} (> 0.30)")
        if b_np_pr is not None and b_np_pr < 0.30:
            qc_flags.append(f"[WARNING] Background Fit: Model {m_sp} fold {m_fld} bias nonpeaks count Pearson r = {b_np_pr:.4f} (< 0.30)")
        if max_resp is not None:
            if max_resp >= 0.005:
                qc_flags.append(f"[CRITICAL] Residual Tn5 Bias: Model {m_sp} fold {m_fld} max marginal footprint response = {max_resp:.4f} (>= 0.005)")
            elif max_resp >= 0.003:
                qc_flags.append(f"[WARNING] Residual Tn5 Bias: Model {m_sp} fold {m_fld} max marginal footprint response = {max_resp:.4f} (>= 0.003)")

print("=" * 80)
print("[*] BLOCK 2 AUTOMATED QC & INTEGRITY AUDIT SUMMARY")
print("=" * 80)
if not qc_flags:
    print("[PASS] All QC checks within expected ranges.")
else:
    for flag in qc_flags:
        print(flag)
print("=" * 80)

# ------------------------------------------------------------------------------
# Part E: Export & Display Summary Tables
# ------------------------------------------------------------------------------
sample_qc_out = out_results_dir / "atac_sample_qc.tsv"
species_qc_out = out_results_dir / "atac_species_qc.tsv"
chrombpnet_qc_out = out_results_dir / "chrombpnet_qc.tsv"

df_sample_qc.to_csv(sample_qc_out, sep="\t", index=False)
df_species_qc.to_csv(species_qc_out, sep="\t", index=False)
df_cbp_qc.to_csv(chrombpnet_qc_out, sep="\t", index=False)

print(f"\n[+] Saved Part A Sample QC Table:     {sample_qc_out}")
print(f"[+] Saved Part B Species QC Table:    {species_qc_out}")
print(f"[+] Saved Part C ChromBPNet QC Table: {chrombpnet_qc_out}\n")

print("\n--- Part A: Per-Sample ATAC-seq Trimming & Alignment QC ---")
display(df_sample_qc)

print("\n--- Part B: Per-Species Stage Merge, Peak Calling & Splits ---")
display(df_species_qc)

print("\n--- Part C: Cross-Species ChromBPNet Stage 1 Base Models ---")
display(df_cbp_qc)

---
## Block 3: Stage 2 Continuous Pre-MZT Linear Probe & Grafted End-to-End Models

Maps late-blastula ChromBPNet representations onto pre-MZT chromatin dynamics:
* Extracts reverse-complement-averaged penultimate CNN embeddings over high-confidence summits (`FE >= 4.0`, `-log10(q) >= 5.0`) to eliminate sparse low-enrichment background noise.
* Fits chromosome-grouped 5-fold cross-validated `StandardScaler + RidgeCV` linear probes (**Stage 2 only**) predicting pre-MZT openness ($y_{\text{abs}}$) and early-vs-late timing bias ($y_{\text{rel}}$), reporting both all-peak and `miR-430` repeat-filtered accuracy. Using a penalized linear probe instead of retraining the full CNN on lower-depth early ATAC preserves Stage 1's sharp TF motif filters without overfitting noise.
* Folds scaling and Ridge coefficients analytically into a single `nn.Linear` layer grafted onto the frozen Stage 1 `nobias` encoder (`grafted_chrombpnet.pt`), and calibrates the **Openness Gate** (`openness_gate.json`) so <20% of closed silent promoters pass into gated timing evaluations.

In [ ]:
# Fit RidgeCV Linear Probes on Real Latent Embeddings (Zebrafish primary, high-confidence peaks)
from scripts.infer_and_attribute import (
    load_chrombpnet_encoder,
    extract_summit_embeddings_from_genome,
    fit_linear_probes,
    GraftedChromBPNet
)

# ------------------------------------------------------------------------------
# Stage 2 Summit Filtering & QC Gate Settings
# ------------------------------------------------------------------------------
# Filter weak gDNA-control noise summits in Zebrafish (FE >= 4.0 & -log10(q) >= 5.0 -> ~49.4k peaks)
STAGE2_FILTER_HIGHCONF_PEAKS = globals().get('STAGE2_FILTER_HIGHCONF_PEAKS', globals().get('STAGE2_FILTER_HIGH_CONF_PEAKS', True))
STAGE2_MIN_FE = globals().get('STAGE2_MIN_FE', 4.0)
STAGE2_MIN_NEG_LOG10_Q = globals().get('STAGE2_MIN_NEG_LOG10_Q', 5.0)
MIN_STAGE1_COUNT_PEARSON_R = 0.50   # QC gate before promoting a species model into Blocks 4-5 ensembles
ALLOW_LOW_QC_MEDAKA = False         # Keep False so low-SNR Medaka (r=0.39, TSSe=1.35/2.23) is not averaged 50/50 with ZF
RUN_MEDAKA_DIAGNOSTIC_PROBE = True  # Still fit & log Medaka probe metrics as diagnostic-only (saved separately)

def filter_summit_targets(
    targets_tsv: Path,
    narrowpeak_path: Path,
    min_fe: float = 4.0,
    min_neg_log10_q: float = 5.0,
    apply_filter: bool = True
) -> pd.DataFrame:
    """Load summit targets with string chroms, attach fold_enrichment from narrowPeak, and filter weak peaks."""
    df = pd.read_csv(targets_tsv, sep="\t", dtype={"chrom": str}, low_memory=False)
    n_raw = len(df)

    # Drop unplaced scaffolds outside train/valid/test splits
    if "split" in df.columns:
        df = df[df["split"].isin(["train", "valid", "test"])].copy()

    # Join fold_enrichment (col 6) and -log10(q) (col 8) from narrowPeak keyed by peak_id (col 3)
    if narrowpeak_path.exists():
        np_df = pd.read_csv(
            narrowpeak_path,
            sep="\t",
            header=None,
            usecols=[3, 6, 8],
            names=["peak_id", "fold_enrichment", "neg_log10_q"],
            dtype={"peak_id": str}
        ).drop_duplicates(subset=["peak_id"])
        df = df.merge(np_df, on="peak_id", how="left")
    else:
        print(f"[-] Notice: narrowPeak file not found at {narrowpeak_path}; using peak_score (-log10 q) only.")
        df["fold_enrichment"] = np.nan
        df["neg_log10_q"] = pd.to_numeric(df.get("peak_score", 0.0), errors="coerce")

    if apply_filter:
        q_vals = df["neg_log10_q"].fillna(pd.to_numeric(df.get("peak_score", 0.0), errors="coerce")).fillna(0.0)
        keep_mask = q_vals >= min_neg_log10_q
        if df["fold_enrichment"].notna().any():
            keep_mask = keep_mask & (df["fold_enrichment"].fillna(0.0) >= min_fe)
        df = df.loc[keep_mask].copy()

    df = df.reset_index(drop=True)
    split_counts = df["split"].value_counts().to_dict() if "split" in df.columns else {}
    n_rep = int(df["is_repeat_downweight"].fillna(False).sum()) if "is_repeat_downweight" in df.columns else 0
    print(f"[+] Summit targets loaded from {targets_tsv.name}: {n_raw:,} raw -> {len(df):,} kept "
          f"(filter={apply_filter}, FE>={min_fe}, -log10q>={min_neg_log10_q}; repeats flagged={n_rep})")
    print(f"    Split counts: {split_counts}")
    return df

zf_atac_dir = Path(WORK_DIR) / "atac" / "Danio_rerio"
zf_targets_path = zf_atac_dir / "Danio_rerio_summit_targets.tsv"
if not zf_targets_path.exists():
    raise FileNotFoundError(f"[FATAL] Summit targets table not found at {zf_targets_path}. Please wait for prep_atac_danio_rerio to complete.")

zf_narrowpeak_path = zf_atac_dir / "peaks" / "Danio_rerio_late_peaks.narrowPeak"
if not zf_narrowpeak_path.exists():
    zf_narrowpeak_path = zf_atac_dir / "peaks" / "Danio_rerio_late_peaks.primary.narrowPeak"

df_summit_targets = filter_summit_targets(
    targets_tsv=zf_targets_path,
    narrowpeak_path=zf_narrowpeak_path,
    min_fe=STAGE2_MIN_FE,
    min_neg_log10_q=STAGE2_MIN_NEG_LOG10_Q,
    apply_filter=STAGE2_FILTER_HIGHCONF_PEAKS
)

# Find real trained ChromBPNet debiased checkpoint strictly inside chrombpnet_run/models/ or models/
valid_model_exts = {".pt", ".pth", ".ckpt", ".h5", ".hdf5"}
zf_fold_dir = Path(WORK_DIR) / "models" / "chrombpnet" / "Danio_rerio" / f"fold_{FOLDS_TO_TRAIN[0]}"
cands = [p for p in zf_fold_dir.glob("**/models/*nobias*.*") if p.suffix.lower() in valid_model_exts and "bias_run" not in str(p) and "bias_model" not in str(p)]
if not cands:
    cands = [p for p in zf_fold_dir.glob("*nobias*.*") if p.suffix.lower() in valid_model_exts and "bias_run" not in str(p) and "bias_model" not in str(p)]

if not cands:
    raise FileNotFoundError(f"[FATAL] Debiased Stage 1 checkpoint (*nobias*.pt/h5) not found under {zf_fold_dir}. Zero-mock policy strictly enforced.")
checkpoint_path = cands[0]

print(f"[*] Loading trained Zebrafish ChromBPNet debiased encoder: {checkpoint_path}")
device = "cuda" if torch.cuda.is_available() else "cpu"
encoder_zf = load_chrombpnet_encoder(checkpoint_path, device=device)

# Display Stage 1 post-training QC metrics (including debiased count r & max Tn5 footprint response)
meta_json = zf_fold_dir / "training_meta.json"
if meta_json.exists():
    with open(meta_json) as f:
        meta_data = json.load(f)
    qc = meta_data.get("qc_metrics", {})
    print("=" * 80)
    print(f"[*] Zebrafish Stage 1 ChromBPNet QC Evaluation Metrics (From {meta_json.name}):")
    print(f"    Full Count Pearson r:      {qc.get('count_pearson_r')}")
    print(f"    Debiased Count Pearson r:  {qc.get('nobias_count_pearson_r')}")
    print(f"    Profile Pearson r:         {qc.get('profile_pearson_r')}")
    print(f"    Profile Median JSD:        {qc.get('profile_median_jsd')}")
    print(f"    Max Tn5 Footprint Resp:    {qc.get('max_tn5_footprint_response', qc.get('max_bias_response'))}")
    print("=" * 80)

print(f"[*] Extracting real penultimate embeddings over {len(df_summit_targets):,} high-confidence summit windows (2,114 bp, forward+RC averaged)...")
real_embeddings_zf = extract_summit_embeddings_from_genome(
    encoder=encoder_zf,
    genome_fa_path=ZEBRAFISH_GENOME_FA,
    df_summits=df_summit_targets,
    window_bp=2114,
    batch_size=64,
    device=device
)

probe_results_zf = fit_linear_probes(
    embeddings=real_embeddings_zf,
    df_targets=df_summit_targets,
    test_split_col="split",
    test_split_val="test",
    exclude_housekeeping=EXCLUDE_HOUSEKEEPING_PROMOTERS_STAGE2
)

In [ ]:
# Fold Linear Probes into End-to-End Grafted Models and Persist Checkpoints
from scripts.infer_and_attribute import save_grafted_checkpoint, load_grafted_model

scaler_zf = probe_results_zf["scaler"]
grafted_zf_model = GraftedChromBPNet(
    encoder=encoder_zf,
    coef_abs=probe_results_zf["y_abs"]["coef"],
    intercept_abs=probe_results_zf["y_abs"]["intercept"],
    coef_rel=probe_results_zf["y_rel"]["coef"],
    intercept_rel=probe_results_zf["y_rel"]["intercept"],
    scaler_mean=scaler_zf.mean_,
    scaler_scale=scaler_zf.scale_
)
grafted_zf_model.to(device)
grafted_zf_model.eval()

# Save canonical grafted Zebrafish model checkpoint
grafted_zf_path = zf_fold_dir / "grafted_chrombpnet.pt"
save_grafted_checkpoint(
    grafted_model=grafted_zf_model,
    out_path=grafted_zf_path,
    base_checkpoint_path=str(checkpoint_path),
    metadata={
        "species": "Danio_rerio",
        "fold": FOLDS_TO_TRAIN[0],
        "n_summits_filtered": int(len(df_summit_targets)),
        "min_fe": float(STAGE2_MIN_FE) if STAGE2_FILTER_HIGH_CONF_PEAKS else None,
        "min_neg_log10_q": float(STAGE2_MIN_NEG_LOG10_Q) if STAGE2_FILTER_HIGH_CONF_PEAKS else None,
        "test_spearman_r_abs": float(probe_results_zf["y_abs"]["test_spearman_r"]),
        "test_auroc_abs": float(probe_results_zf["y_abs"]["test_auroc"]),
        "early_late_auroc_abs": probe_results_zf["y_abs"].get("early_late_auroc"),
        "test_spearman_r_rel": float(probe_results_zf["y_rel"]["test_spearman_r"]),
    }
)
print(f"[+] Canonical Zebrafish grafted model saved to {grafted_zf_path}")

# ------------------------------------------------------------------------------
# Medaka Stage 2 Linear Probe (QC-Gated to prevent 50/50 noise dilution in Blocks 4-5)
# ------------------------------------------------------------------------------

if ENABLE_MEDAKA:
    med_fold_dir = Path(WORK_DIR) / "models" / "chrombpnet" / "Oryzias_latipes" / f"fold_{FOLDS_TO_TRAIN[0]}"
    med_targets_path = Path(WORK_DIR) / "atac" / "Oryzias_latipes" / "Oryzias_latipes_summit_targets.tsv"
    med_meta_json = med_fold_dir / "training_meta.json"
    med_count_r = None
    if med_meta_json.exists():
        with open(med_meta_json) as f:
            med_qc = json.load(f).get("qc_metrics", {})
        med_count_r = med_qc.get("nobias_count_pearson_r")
        if med_count_r is None:
            med_count_r = med_qc.get("count_pearson_r")

    MEDAKA_PASSED_QC = (med_count_r is not None) and (float(med_count_r) >= MIN_STAGE1_COUNT_PEARSON_R)
    if not MEDAKA_PASSED_QC:
        grafted_medaka_model = None
        grafted_medaka_diagnostic_model = None
    promote_medaka = MEDAKA_PASSED_QC or ALLOW_LOW_QC_MEDAKA
    MEDAKA_PASSED_QC = promote_medaka
    med_qc_path = Path(WORK_DIR) / "models" / "medaka" / "qc_status.json"
    med_qc_path.parent.mkdir(parents=True, exist_ok=True)
    with open(med_qc_path, "w") as _f_qc:
        json.dump({
            "MEDAKA_PASSED_QC": bool(MEDAKA_PASSED_QC),
            "promote_medaka": bool(promote_medaka),
            "stage1_nobias_count_pearson_r": float(med_count_r) if med_count_r is not None else None,
            "min_required_count_pearson_r": float(MIN_STAGE1_COUNT_PEARSON_R),
            "status": "passed_qc" if MEDAKA_PASSED_QC else "failed_qc",
            "allow_low_qc_medaka": bool(ALLOW_LOW_QC_MEDAKA)
        }, _f_qc, indent=2)

    if not promote_medaka:
        print("=" * 80)
        print(f"[QC GATE] Medaka Stage 1 debiased count Pearson r = {med_count_r} (< {MIN_STAGE1_COUNT_PEARSON_R:.2f}).")
        print("          Excluding Medaka from canonical Blocks 4-5 50/50 ensemble to avoid diluting Zebrafish predictions.")
        print("          (Set ALLOW_LOW_QC_MEDAKA = True in Cell 1 to override.)")
        print("=" * 80)
        # Ensure no stale canonical grafted_chrombpnet.pt gets auto-loaded by Blocks 4-5 on fresh kernel restart
        stale_med_ckpt = med_fold_dir / "grafted_chrombpnet.pt"
        if stale_med_ckpt.exists():
            backup_ckpt = med_fold_dir / "grafted_chrombpnet_low_snr_diagnostic.pt"
            stale_med_ckpt.rename(backup_ckpt)
            print(f"[*] Moved existing low-SNR Medaka checkpoint -> {backup_ckpt.name} so Blocks 4-5 stay Zebrafish-only.")

    valid_model_exts = {".pt", ".pth", ".ckpt", ".h5", ".hdf5"}
    med_cands = [p for p in med_fold_dir.glob("**/models/*nobias*.*") if p.suffix.lower() in valid_model_exts and "bias_run" not in str(p) and "bias_model" not in str(p)]
    if not med_cands:
        med_cands = [p for p in med_fold_dir.glob("*nobias*.*") if p.suffix.lower() in valid_model_exts and "bias_run" not in str(p) and "bias_model" not in str(p)]

    if (promote_medaka or RUN_MEDAKA_DIAGNOSTIC_PROBE) and med_targets_path.exists() and med_cands:
        # Do NOT apply FE >= 4 filter to Medaka (only 245 peaks survive FE >= 4 due to self-pileup lambda suppression)
        df_med_targets = pd.read_csv(med_targets_path, sep="\t", dtype={"chrom": str}, low_memory=False)
        if "split" in df_med_targets.columns:
            df_med_targets = df_med_targets[df_med_targets["split"].isin(["train", "valid", "test"])].reset_index(drop=True)

        encoder_med = load_chrombpnet_encoder(med_cands[0], device=device)
        print(f"[*] Extracting penultimate embeddings over {len(df_med_targets):,} Medaka summits...")
        embeddings_med = extract_summit_embeddings_from_genome(
            encoder=encoder_med,
            genome_fa_path=MEDAKA_GENOME_FA,
            df_summits=df_med_targets,
            window_bp=2114,
            batch_size=64,
            device=device
        )
        probe_results_med = fit_linear_probes(
            embeddings=embeddings_med,
            df_targets=df_med_targets,
            test_split_col="split",
            test_split_val="test",
            exclude_housekeeping=EXCLUDE_HOUSEKEEPING_PROMOTERS_STAGE2
        )
        scaler_med = probe_results_med["scaler"]
        med_model_built = GraftedChromBPNet(
            encoder=encoder_med,
            coef_abs=probe_results_med["y_abs"]["coef"],
            intercept_abs=probe_results_med["y_abs"]["intercept"],
            coef_rel=probe_results_med["y_rel"]["coef"],
            intercept_rel=probe_results_med["y_rel"]["intercept"],
            scaler_mean=scaler_med.mean_,
            scaler_scale=scaler_med.scale_
        )
        med_model_built.to(device)
        med_model_built.eval()

        if promote_medaka:
            grafted_medaka_model = med_model_built
            grafted_med_path = med_fold_dir / "grafted_chrombpnet.pt"
            save_grafted_checkpoint(
                grafted_model=grafted_medaka_model,
                out_path=grafted_med_path,
                base_checkpoint_path=str(med_cands[0]),
                metadata={"species": "Oryzias_latipes", "fold": FOLDS_TO_TRAIN[0], "stage1_count_r": med_count_r}
            )
            print(f"[+] Canonical Medaka grafted model saved to {grafted_med_path}")
        else:
            grafted_medaka_diagnostic_model = med_model_built
            diag_med_path = med_fold_dir / "grafted_chrombpnet_low_snr_diagnostic.pt"
            save_grafted_checkpoint(
                grafted_model=grafted_medaka_diagnostic_model,
                out_path=diag_med_path,
                base_checkpoint_path=str(med_cands[0]),
                metadata={"species": "Oryzias_latipes", "fold": FOLDS_TO_TRAIN[0], "stage1_count_r": med_count_r, "diagnostic_only": True}
            )
            print(f"[+] Saved diagnostic-only Medaka model to {diag_med_path} (grafted_medaka_model remains None for Blocks 4-5).")

# ------------------------------------------------------------------------------
# Multi-fold Ensemble Wrappers
# ------------------------------------------------------------------------------
grafted_zf_models = [grafted_zf_model]
if len(FOLDS_TO_TRAIN) > 1:
    print(f"[*] Multi-fold training enabled: fitting linear probes across {len(FOLDS_TO_TRAIN)} Zebrafish folds...")
    zf_folds_json = Path(WORK_DIR) / "atac" / "Danio_rerio" / "chr_folds.json"
    all_zf_folds = {}
    if zf_folds_json.exists():
        with open(zf_folds_json) as f:
            all_zf_folds = json.load(f)

    for fold_i in FOLDS_TO_TRAIN[1:]:
        f_dir = Path(WORK_DIR) / "models" / "chrombpnet" / "Danio_rerio" / f"fold_{fold_i}"
        f_cands = [p for p in f_dir.glob("**/models/*nobias*.*") if p.suffix.lower() in valid_model_exts and "bias_run" not in str(p) and "bias_model" not in str(p)]
        if not f_cands:
            f_cands = [p for p in f_dir.glob("*nobias*.*") if p.suffix.lower() in valid_model_exts and "bias_run" not in str(p) and "bias_model" not in str(p)]
        if f_cands:
            f_enc = load_chrombpnet_encoder(f_cands[0], device=device)
            f_embs = extract_summit_embeddings_from_genome(f_enc, ZEBRAFISH_GENOME_FA, df_summit_targets, device=device)
            df_f = df_summit_targets.copy()
            if f"fold_{fold_i}" in all_zf_folds:
                f_info = all_zf_folds[f"fold_{fold_i}"]
                test_c = set(str(c) for c in f_info.get("test", []))
                val_c = set(str(c) for c in f_info.get("valid", []))
                train_c = set(str(c) for c in f_info.get("train", []))
                df_f["split"] = np.where(
                    df_f["chrom"].astype(str).isin(test_c), "test",
                    np.where(df_f["chrom"].astype(str).isin(val_c), "valid",
                             np.where(df_f["chrom"].astype(str).isin(train_c), "train", "excluded"))
                )
            f_res = fit_linear_probes(f_embs, df_f, test_split_col="split", test_split_val="test", exclude_housekeeping=EXCLUDE_HOUSEKEEPING_PROMOTERS_STAGE2)
            f_grafted = GraftedChromBPNet(f_enc, f_res["y_abs"]["coef"], f_res["y_abs"]["intercept"], f_res["y_rel"]["coef"], f_res["y_rel"]["intercept"], f_res["scaler"].mean_, f_res["scaler"].scale_)
            f_grafted.to(device)
            f_grafted.eval()
            save_grafted_checkpoint(grafted_model=f_grafted, out_path=f_dir / "grafted_chrombpnet.pt", base_checkpoint_path=str(f_cands[0]))
            grafted_zf_models.append(f_grafted)
    print(f"[+] Constructed Zebrafish multi-fold ensemble with {len(grafted_zf_models)} models.")

grafted_medaka_models = [grafted_medaka_model] if grafted_medaka_model is not None else []
if ENABLE_MEDAKA and len(FOLDS_TO_TRAIN) > 1 and grafted_medaka_model is not None:
    print(f"[*] Multi-fold training enabled: fitting linear probes across {len(FOLDS_TO_TRAIN)} Medaka folds...")
    med_folds_json = Path(WORK_DIR) / "atac" / "Oryzias_latipes" / "chr_folds.json"
    all_med_folds = {}
    if med_folds_json.exists():
        with open(med_folds_json) as f:
            all_med_folds = json.load(f)

    for fold_i in FOLDS_TO_TRAIN[1:]:
        f_dir = Path(WORK_DIR) / "models" / "chrombpnet" / "Oryzias_latipes" / f"fold_{fold_i}"
        f_cands = [p for p in f_dir.glob("**/models/*nobias*.*") if p.suffix.lower() in valid_model_exts and "bias_run" not in str(p) and "bias_model" not in str(p)]
        if not f_cands:
            f_cands = [p for p in f_dir.glob("*nobias*.*") if p.suffix.lower() in valid_model_exts and "bias_run" not in str(p) and "bias_model" not in str(p)]
        if f_cands and "df_med_targets" in locals():
            f_enc = load_chrombpnet_encoder(f_cands[0], device=device)
            f_embs = extract_summit_embeddings_from_genome(f_enc, MEDAKA_GENOME_FA, df_med_targets, device=device)
            df_mf = df_med_targets.copy()
            if f"fold_{fold_i}" in all_med_folds:
                f_info = all_med_folds[f"fold_{fold_i}"]
                test_c = set(str(c) for c in f_info.get("test", []))
                val_c = set(str(c) for c in f_info.get("valid", []))
                train_c = set(str(c) for c in f_info.get("train", []))
                df_mf["split"] = np.where(
                    df_mf["chrom"].astype(str).isin(test_c), "test",
                    np.where(df_mf["chrom"].astype(str).isin(val_c), "valid",
                             np.where(df_mf["chrom"].astype(str).isin(train_c), "train", "excluded"))
                )
            f_res = fit_linear_probes(f_embs, df_mf, test_split_col="split", test_split_val="test", exclude_housekeeping=EXCLUDE_HOUSEKEEPING_PROMOTERS_STAGE2)
            f_grafted = GraftedChromBPNet(f_enc, f_res["y_abs"]["coef"], f_res["y_abs"]["intercept"], f_res["y_rel"]["coef"], f_res["y_rel"]["intercept"], f_res["scaler"].mean_, f_res["scaler"].scale_)
            f_grafted.to(device)
            f_grafted.eval()
            save_grafted_checkpoint(grafted_model=f_grafted, out_path=f_dir / "grafted_chrombpnet.pt", base_checkpoint_path=str(f_cands[0]))
            grafted_medaka_models.append(f_grafted)

---
## Block 4: Zero-Shot Validation on Target-Species Minor-Wave Genes & Cross-Species ATAC

Tests whether Stage 2 grafted ChromBPNet generalizes **zero-shot** (without seeing any ATAC-seq from the target species) across four strand-aware cis-windows per gene (`promoter_1kb`, `promoter_intron1_capped`, `proximal_5kb`, neighbor-bounded `bounded_25kb`), excluding mitochondrial and unmapped zero-span contigs:
* Evaluates strictly zygotic (`strictly_zygotic_highconf`, `strictly_zygotic_minor_wave`), SLAM-confirmed (`slam_confirmed_minor_wave`), and `all_upregulated` positives against **primary `control_silent`** controls (testing chromatin opening above inert DNA) and **diagnostic `control_non_activated` maternal** controls (testing *early timing specificity* over constitutively open housekeeping promoters), alongside an in-species GC-matched (`1:5`) held-out test-chromosome reference and cross-species Medaka benchmark.

**Generated Outputs (Block 4):**
* **Figure 2 (`fig2_zeroshot_benchmark.png` & `.svg`):** Four-panel benchmark: (`A`) ROC vs. `control_silent` at `promoter_1kb`; (`B`) Precision-Recall curves with dotted per-cohort random prevalence baselines (noting that initial Precision dips near Recall = 0 occur when a small number of CpG-rich/poised silent promoters score above the highest minor-wave gene); (`C`) window × metric AUROC bars vs. `control_silent` overlaid with the in-species test-chromosome GC-matched `1:5` ceiling; (`D`) timing-specificity AUROC bars vs. `control_non_activated` maternal controls, where `y_resid_gated` flips ahead of raw `y_abs`/`late`.
* **Metrics & Tables (`zero_shoot_validation_metrics.json`, `zero_shoot_window_control_comparison.tsv`, `target_genes_scored_metrics.tsv`):** Full per-window AUROC, AUPRC, Mann–Whitney *U* statistics, and gene-level predictions.

In [ ]:
# ==============================================================================
# Block 4a: Launch Standalone Zero-Shot Cross-Species Validation Task
# ==============================================================================
folds_arg = ",".join(str(f) for f in FOLDS_TO_TRAIN)

# Set True once after updating Block 4 scoring/controls so TaskManager doesn't
# skip due to the previous .done marker, then revert to FORCE_RERUN.
FORCE_RERUN_BLOCK4 = False

VALIDATION_WINDOWS = ["promoter_1kb", "promoter_intron1_capped", "proximal_5kb", "bounded_25kb"]

zero_shoot_cmd = [
    "python", "-u", "scripts/zero_shoot_validation.py",
    "--work-dir", WORK_DIR,
    "--config", f"{WORK_DIR}/config.json",
    "--folds", folds_arg,
    "--batch-size", "256",
    "--primary-control", "control_silent",
    "--diagnostic-control", "control_non_activated",
    "--silent-max-tpm", "0.1",
    "--windows", ",".join(VALIDATION_WINDOWS),
    "--strict-early-max-tpm", "0.2",
    "--strict-premzt-min-tpm", "5.0",
    "--case-insensitive-orthologs",
    "--run-zebrafish-ceiling",
    "--open-gate-late-thresh", str(OPEN_GATE_LATE_THRESH),
]
if ENABLE_MEDAKA and MEDAKA_PASSED_QC:
    zero_shoot_cmd.append("--enable-medaka")
else:
    zero_shoot_cmd.append("--no-enable-medaka")

tm.submit(
    "zero_shoot_validation",
    zero_shoot_cmd,
    depends_on=[f"train_chrombpnet_danio_rerio_fold_{folds_arg}"],
    force_rerun=(FORCE_RERUN or FORCE_RERUN_BLOCK4),
    requires_gpu=True
)

display(tm.status())


In [ ]:
# ==============================================================================
# Block 4b: Inspect Validation Metrics & Render Figure 2 (Zero-Shot Benchmark)
# ==============================================================================
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from sklearn.metrics import (
    roc_curve,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
)

scored_tsv = res_dir / "target_genes_scored_metrics.tsv"
metrics_json_path = res_dir / "zero_shoot_validation_metrics.json"

PLOT_WINDOW = "promoter_1kb"
PLOT_SCORE = "y_abs"          # "y_abs", "y_resid_gated", or "late"
PRIMARY_CTRL = "control_silent"
DIAG_CTRL = "control_non_activated"

_og_path = Path(WORK_DIR) / "models" / "zebrafish" / "openness_gate.json"
OPEN_GATE_LATE_THRESH = 5.70
if _og_path.exists():
    try:
        with open(_og_path) as _f_og:
            OPEN_GATE_LATE_THRESH = float(json.load(_f_og).get("OPEN_GATE_LATE_THRESH", 5.70))
    except Exception:
        OPEN_GATE_LATE_THRESH = 5.70

def _bool_flag(df, col):
    if col in df.columns:
        return df[col].fillna(False).astype(bool)
    return pd.Series(False, index=df.index)

def _resolve_score_col(df, window, score_type):
    candidates = [
        f"pred_{window}_{score_type}",
        f"pred_max_{window}_{score_type}",
        f"pred_max_{score_type}",
    ]
    if score_type in ["y_rel_gated", "y_resid_gated"]:
        candidates.extend([
            f"pred_{window}_y_resid_gated", f"pred_max_{window}_y_resid_gated",
            f"pred_{window}_y_rel_gated", f"pred_max_{window}_y_rel_gated",
        ])
    for c in candidates:
        if c in df.columns:
            return c
    return None

def _eval_binary(df, pos_mask, ctrl_mask, score_col):
    if score_col is None or score_col not in df.columns:
        return None
    pos_s = df.loc[pos_mask, score_col].dropna().values
    ctrl_s = df.loc[ctrl_mask, score_col].dropna().values
    if len(pos_s) == 0 or len(ctrl_s) == 0:
        return {
            "roc_auc": np.nan, "pr_auc": np.nan, "p_val": np.nan,
            "n_pos": len(pos_s), "n_ctrl": len(ctrl_s),
            "fpr": None, "tpr": None, "prec": None, "rec": None,
        }
    y_true = np.concatenate([np.ones(len(pos_s)), np.zeros(len(ctrl_s))])
    y_score = np.concatenate([pos_s, ctrl_s])
    fpr, tpr, _ = roc_curve(y_true, y_score)
    prec, rec, _ = precision_recall_curve(y_true, y_score)
    _, p_val = mannwhitneyu(pos_s, ctrl_s, alternative="greater")
    return {
        "roc_auc": float(roc_auc_score(y_true, y_score)),
        "pr_auc": float(average_precision_score(y_true, y_score)),
        "p_val": float(p_val),
        "n_pos": len(pos_s),
        "n_ctrl": len(ctrl_s),
        "fpr": fpr, "tpr": tpr, "prec": prec, "rec": rec,
    }

if not scored_tsv.exists():
    print(f"[*] Scored gene metrics table not found at {scored_tsv}. Waiting for 'zero_shoot_validation' task to complete.")
else:
    df_scored = pd.read_csv(scored_tsv, sep="\t", dtype={"chrom": str}, low_memory=False)
    chrom_low = df_scored["chrom"].astype(str).str.strip().str.lower()
    valid_locus = (~chrom_low.isin({"mt", "chrm", "m", "chrmt", "unknown", "nan", ""}))
    if "start" in df_scored.columns and "end" in df_scored.columns:
        valid_locus = valid_locus & ~((df_scored["start"].fillna(0) == 0) & (df_scored["end"].fillna(0) == 0))
    n_drop_mt = int((~valid_locus).sum())
    if n_drop_mt > 0:
        df_scored = df_scored.loc[valid_locus].copy()
        print(f"[+] Loaded scored gene metrics: {len(df_scored):,} genomic genes (excluded {n_drop_mt} MT/unmapped rows) from {scored_tsv.name}")
    else:
        print(f"[+] Loaded scored gene metrics: {len(df_scored):,} genes from {scored_tsv.name}")

    for w in ["promoter_1kb", "promoter_intron1_capped", "proximal_5kb", "bounded_25kb"]:
        s_col = f"pred_{w}_y_resid" if f"pred_{w}_y_resid" in df_scored.columns else f"pred_{w}_y_rel"
        if s_col in df_scored.columns and f"pred_{w}_late" in df_scored.columns:
            if f"pred_{w}_y_resid_gated" not in df_scored.columns:
                df_scored[f"pred_{w}_y_resid_gated"] = np.where(
                    df_scored[f"pred_{w}_late"] >= OPEN_GATE_LATE_THRESH,
                    df_scored[s_col],
                    np.nan
                )
            if f"pred_{w}_y_rel_gated" not in df_scored.columns:
                df_scored[f"pred_{w}_y_rel_gated"] = df_scored[f"pred_{w}_y_resid_gated"]

    zero_shoot_metrics = {}
    if metrics_json_path.exists():
        with open(metrics_json_path, "r") as f_m:
            zero_shoot_metrics = json.load(f_m)
        print(f"[+] Loaded validation metrics report from {metrics_json_path.name}")
        if "zebrafish_ceiling" in zero_shoot_metrics:
            print("[+] In-Species Zebrafish Test-Chr Reference (GC-matched 1:5 ATAC -> SLAM):")
            for w_name, w_res in zero_shoot_metrics["zebrafish_ceiling"].items():
                if w_name.startswith("_") or not isinstance(w_res, dict):
                    continue
                y_abs_c = w_res.get("y_abs", {}).get("roc_auc", np.nan)
                y_rel_c = w_res.get("y_rel_gated", w_res.get("y_rel", {})).get("roc_auc", np.nan)
                late_c = w_res.get("late", {}).get("roc_auc", np.nan)
                print(f"    {w_name:<14} -> y_abs AUC: {y_abs_c:.3f} | y_rel_gated AUC: {y_rel_c:.3f} | late AUC: {late_c:.3f}")
        if "medaka_st9_atac" in zero_shoot_metrics:
            mk = zero_shoot_metrics["medaka_st9_atac"]
            print(f"[+] Zero-Shot Medaka st9 Pre-MZT ATAC: Spearman r = {mk.get('spearman_r', np.nan):.3f}, AUROC = {mk.get('roc_auc', np.nan):.3f}")

    pos_masks = {
        "strictly_zygotic_highconf": _bool_flag(df_scored, "is_strictly_zygotic_highconf") | (df_scored["category"] == "strictly_zygotic_highconf"),
        "strictly_zygotic_minor_wave": _bool_flag(df_scored, "is_strictly_zygotic_minor_wave") | (df_scored["category"] == "strictly_zygotic_minor_wave"),
        "slam_confirmed_minor_wave": _bool_flag(df_scored, "is_slam_confirmed_minor_wave") | _bool_flag(df_scored, "is_slam_confirmed") | (df_scored["category"] == "slam_confirmed_minor_wave"),
        "all_upregulated": _bool_flag(df_scored, "is_all_upregulated") | df_scored["category"].isin(["all_upregulated", "strictly_zygotic_minor_wave", "strictly_zygotic_highconf"]),
    }
    ctrl_masks = {
        "control_silent": _bool_flag(df_scored, "is_control_silent") | (df_scored["category"] == "control_silent"),
        "control_non_activated": _bool_flag(df_scored, "is_control_non_activated") | (df_scored["category"] == "control_non_activated"),
    }

    active_primary_ctrl = PRIMARY_CTRL if ctrl_masks[PRIMARY_CTRL].sum() > 0 else DIAG_CTRL
    available_windows = [w for w in ["promoter_1kb", "promoter_intron1_capped", "proximal_5kb", "bounded_25kb"] if _resolve_score_col(df_scored, w, "y_abs") is not None]
    if not available_windows:
        available_windows = ["legacy_25kb"]

    summary_rows = []
    primary_pos_key = "strictly_zygotic_highconf" if pos_masks["strictly_zygotic_highconf"].sum() >= 10 else "strictly_zygotic_minor_wave"
    for w in available_windows:
        for s_type in ["y_abs", "y_resid_gated", "late"]:
            col = _resolve_score_col(df_scored, w, s_type)
            score_label = s_type if "gated" not in s_type else f"y_resid_gated (late>={OPEN_GATE_LATE_THRESH:.1f})"
            for c_name, c_mask in ctrl_masks.items():
                if c_mask.sum() == 0:
                    continue
                res = _eval_binary(df_scored, pos_masks[primary_pos_key], c_mask, col)
                if res is not None:
                    pv = res["p_val"]
                    pv_str = f"{pv:.2e}" if np.isfinite(pv) else "nan"
                    summary_rows.append({
                        "window": w, "score": score_label, "score_type": s_type, "control": c_name,
                        "pos_set": primary_pos_key, "n_pos": res["n_pos"], "n_ctrl": res["n_ctrl"],
                        "roc_auc": round(res["roc_auc"], 3),
                        "pr_auc": round(res["pr_auc"], 3),
                        "p_val": pv_str,
                    })
    df_summary = pd.DataFrame(summary_rows)
    if len(df_summary) > 0:
        print()
        print(f"[*] Window × Score × Control Comparison Summary (y_resid evaluated strictly on open promoters late >= {OPEN_GATE_LATE_THRESH:.2f}):")
        if "display" in globals():
            display(df_summary)
        else:
            print(df_summary.to_string())
        df_summary.to_csv(res_dir / "zero_shoot_window_control_comparison.tsv", sep="\t", index=False)

    plot_win = PLOT_WINDOW if PLOT_WINDOW in available_windows else available_windows[0]
    plot_col = _resolve_score_col(df_scored, plot_win, PLOT_SCORE)

    fig, axes = plt.subplots(2, 2, figsize=(15, 10.8))
    curve_specs = [
        ("strictly_zygotic_highconf", "Strictly Zygotic (High-Conf)", "#a50f15", "-"),
        ("strictly_zygotic_minor_wave", "Strictly Zygotic (All)", "#ef3b2c", "-."),
        ("slam_confirmed_minor_wave", "SLAM-Confirmed Orthologs", "#2b83ba", "-"),
        ("all_upregulated", "All Upregulated (Diagnostic)", "#fdae61", "--"),
    ]
    ax_roc, ax_pr = axes[0, 0], axes[0, 1]
    for key, label_base, color, ls in curve_specs:
        res = _eval_binary(df_scored, pos_masks[key], ctrl_masks[active_primary_ctrl], plot_col)
        if res is not None and res["fpr"] is not None:
            prev = res["n_pos"] / float(max(1, res["n_pos"] + res["n_ctrl"]))
            fold_pr = res["pr_auc"] / max(1e-6, prev)
            ax_roc.plot(
                res["fpr"], res["tpr"],
                label=f"{label_base} (AUC={res['roc_auc']:.2f}, n={res['n_pos']})",
                color=color, lw=2.2, linestyle=ls,
            )
            ax_pr.plot(
                res["rec"], res["prec"],
                label=f"{label_base} (PR={res['pr_auc']:.3f}, {fold_pr:.1f}x base, n={res['n_pos']})",
                color=color, lw=2.2, linestyle=ls,
            )
            ax_pr.axhline(prev, color=color, linestyle=":", lw=1.0, alpha=0.55)

    score_title_str = PLOT_SCORE if "rel" not in PLOT_SCORE and "resid" not in PLOT_SCORE else f"y_resid_gated (late>={OPEN_GATE_LATE_THRESH:.1f})"
    ax_roc.plot([0, 1], [0, 1], "k--", alpha=0.45, label="Random Chance (AUC=0.50)")
    ax_roc.set_xlabel("False Positive Rate (1 - Specificity)", fontsize=10.5, fontweight="bold")
    ax_roc.set_ylabel("True Positive Rate (Sensitivity)", fontsize=10.5, fontweight="bold")
    ax_roc.set_title(f"A. Zero-Shot ROC ({plot_win} | {score_title_str} vs. {active_primary_ctrl})", fontweight="bold", fontsize=11.5)
    ax_roc.legend(frameon=True, fontsize=8.3, loc="lower right")
    ax_roc.grid(True, linestyle=":", alpha=0.4)

    ax_pr.set_xlim(-0.01, 1.01)
    ax_pr.set_ylim(-0.02, 1.04)
    ax_pr.set_xlabel("Recall (True Positive Fraction)", fontsize=10.5, fontweight="bold")
    ax_pr.set_ylabel("Precision (True Positives / Predicted Positives)", fontsize=10.5, fontweight="bold")
    ax_pr.set_title(f"B. Zero-Shot PR vs. Dotted Cohort Prevalence ({plot_win})", fontweight="bold", fontsize=11.5)
    ax_pr.legend(frameon=True, fontsize=8.0, loc="upper right")
    ax_pr.grid(True, linestyle=":", alpha=0.4)
    ax_pr.text(
        0.51, 0.56,
        "Note: Initial drop near Recall=0 reflects top\n"
        "false-positive silent CGI promoters before the\n"
        "first minor-wave gene enters. Dotted lines show\n"
        "random prevalence baselines (0.0008–0.184).",
        transform=ax_pr.transAxes, fontsize=7.8, color="#333333",
        bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="#cccccc", alpha=0.9)
    )

    zf_ceiling = zero_shoot_metrics.get("zebrafish_ceiling", {})
    n_zf_pos = zf_ceiling.get("_n_pos", 16) if isinstance(zf_ceiling, dict) else 16
    for ax_bar, ctrl_key, panel_title in [
        (axes[1, 0], "control_silent", f"C. Primary: {primary_pos_key} vs. Silent Controls (n={int(ctrl_masks['control_silent'].sum()):,})"),
        (axes[1, 1], "control_non_activated", f"D. Diagnostic: {primary_pos_key} vs. Maternal-Stable Controls (n={int(ctrl_masks['control_non_activated'].sum()):,})"),
    ]:
        sub = df_summary[df_summary["control"] == ctrl_key] if len(df_summary) > 0 else pd.DataFrame()
        if len(sub) == 0:
            ax_bar.text(0.5, 0.5, f"No '{ctrl_key}' rows available", ha="center", va="center")
            ax_bar.set_title(panel_title, fontweight="bold")
            continue
        x = np.arange(len(available_windows))
        width = 0.25
        for idx, (s_type, color, s_lbl) in enumerate([
            ("y_abs", "#d7191c", "y_abs (Predicted 256-cell)"),
            ("y_resid_gated", "#2b83ba", f"y_resid (Gated late>={OPEN_GATE_LATE_THRESH:.1f})"),
            ("late", "#756bb1", "late (Dome baseline openness)")
        ]):
            vals = []
            for w in available_windows:
                m = sub[(sub["window"] == w) & (sub["score_type"] == s_type)]
                vals.append(float(m["roc_auc"].iloc[0]) if len(m) > 0 else np.nan)
            bars = ax_bar.bar(x + (idx - 1) * width, vals, width=width, label=s_lbl, color=color, alpha=0.88, edgecolor="black", lw=0.6)
            for b, v in zip(bars, vals):
                if np.isfinite(v):
                    ax_bar.text(b.get_x() + b.get_width() / 2.0, v + 0.012, f"{v:.2f}", ha="center", va="bottom", fontsize=7.6, fontweight="bold")
        if ctrl_key == "control_silent" and zf_ceiling:
            ceil_vals = [
                zf_ceiling.get(w, {}).get("y_abs", {}).get("roc_auc", np.nan)
                if isinstance(zf_ceiling.get(w), dict) else np.nan
                for w in available_windows
            ]
            if any(np.isfinite(ceil_vals)):
                ax_bar.plot(x, ceil_vals, "ko--", lw=1.6, ms=6, label=f"ZF Test-Chr 1:5 GC-Matched (y_abs, n={n_zf_pos})")
        ax_bar.axhline(0.5, color="gray", linestyle="--", alpha=0.6)
        ax_bar.set_xticks(x)
        ax_bar.set_xticklabels(available_windows, fontsize=9.8)
        ax_bar.set_ylim(0.3, 1.0)
        ax_bar.set_ylabel("AUROC", fontsize=10.5, fontweight="bold")
        ax_bar.set_title(panel_title, fontweight="bold", fontsize=11.0)
        ax_bar.legend(frameon=True, fontsize=8.0, loc="upper right")
        ax_bar.grid(True, linestyle=":", alpha=0.4, axis="y")

    plt.tight_layout()
    fig2_path = res_dir / "fig2_zeroshot_benchmark.png"
    plt.savefig(fig2_path, dpi=300, bbox_inches="tight")
    plt.savefig(res_dir / "fig2_zeroshot_benchmark.svg", bbox_inches="tight")
    plt.show()
    print(f"[+] Figure 2 exported to {fig2_path}")


---
## Block 5: Unbiased Sliding-Window Scan of `TARGET_LOCUS_NAME`

Dynamically parses coordinates for `TARGET_GENES` from `TARGET_GTF` and tiles the extended genomic domain (`± LOCUS_FLANK_BP`) alongside its orthologous reference locus using overlapping 2,114 bp ChromBPNet windows (`y_abs`, `late`, `y_rel`, and `y_resid_gated`), identifying non-redundant open candidate elements (`late >= gate`) and flagging **lineage-gained elements** whose peak accessibility exceeds the entire orthologous reference locus maximum.

**Generated Outputs (Block 5):**
* **Figure 3 (`fig3_locus_scan_profile.png` & `.svg`):** Synchronized **5-row × 2-column** comparison between the target locus (left) and held-out orthologous reference locus (right):
  * **Row 0:** Non-overlapping GTF gene models.
  * **Row 1 (`pred_late`):** Predicted late-blastula openness against the calibrated `Openness Gate` threshold.
  * **Row 2 (`pred_y_abs`):** Predicted pre-MZT openness highlighting ranked candidate elements (`C1..Cn`).
  * **Row 3 (`y_resid_gated`):** Displays continuous **un-gated `y_rel`** across closed heterochromatin in gray (illustrating why raw early/late ratios scatter noisily on closed DNA) overlaid with discrete **solid bars** (`blue` early-biased $\ge 0$, `orange` late-biased $< 0$) exclusively on open summits passing the gate.
  * **Row 4 (Candidate Spans vs. Reference Experimental ATAC):** Left marks lineage-gained peaks above the reference max in pink; right overlays observed early and late reference ATAC tracks with **`MACS2/Poisson Dome Reference Peaks`** (peaks are called on high-SNR late-blastula ATAC using local-lambda Poisson modeling and serve as unified coordinates for measuring early vs. late accessibility).
* **Browser Tracks & Candidate Tables (`{TARGET_LOCUS_NAME}_pred_*.bedgraph`, `candidate_elements.tsv`/`.bed`, `{TARGET_LOCUS_NAME}_candidate_enhancers.tsv`/`.bed`).**

In [ ]:
import os, sys, json, math, gzip
import torch
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pyBigWig

WORK_DIR = globals().get("WORK_DIR", "work_astatotilapia_premzt")
TARGET_LOCUS_NAME = globals().get("TARGET_LOCUS_NAME", "notch2_slc35a3a")
TARGET_SPECIES_NAME = globals().get("TARGET_SPECIES_NAME", "Astatotilapia_calliptera")
TARGET_GENES = globals().get("TARGET_GENES", ["notch2", "slc35a3a"])
LOCUS_FLANK_BP = 50000
device = "cuda" if torch.cuda.is_available() else "cpu"
res_dir = Path(WORK_DIR) / "results" / TARGET_LOCUS_NAME
res_dir.mkdir(parents=True, exist_ok=True)

with open(Path(WORK_DIR) / "config.json") as f:
    config = json.load(f)
TARGET_GENOME_FA = config["TARGET_GENOME_FA"]
TARGET_GTF = config["TARGET_GTF"]
ZEBRAFISH_GENOME_FA = config["ZEBRAFISH_GENOME_FA"]
ZEBRAFISH_GTF = config["ZEBRAFISH_GTF"]

_og_path = Path(WORK_DIR) / "models" / "zebrafish" / "openness_gate.json"
OPEN_GATE_LATE_THRESH = 5.70
if _og_path.exists():
    try:
        with open(_og_path) as _f_og:
            OPEN_GATE_LATE_THRESH = float(json.load(_f_og).get("OPEN_GATE_LATE_THRESH", 5.70))
    except Exception:
        OPEN_GATE_LATE_THRESH = 5.70

MEDAKA_PASSED_QC = False

from scripts.infer_and_attribute import load_grafted_model, scan_locus_sliding_window
from scripts.premzt_inference import get_inference_context

zf_ckpt = Path(WORK_DIR) / "models" / "chrombpnet" / "Danio_rerio" / "fold_0" / "grafted_chrombpnet.pt"
assert zf_ckpt.exists(), f"Missing checkpoint: {zf_ckpt}"
grafted_zf_model = load_grafted_model(zf_ckpt, device=device)
print(f"[+] Loaded grafted Zebrafish model from {zf_ckpt}")

gene_sets_tsv = res_dir / f"{TARGET_LOCUS_NAME}_gene_sets.tsv"
df_counts = pd.read_csv(gene_sets_tsv, sep="\t", low_memory=False)

tg_coords = []
for tg in TARGET_GENES:
    tg_rows = df_counts[df_counts["gene_name"].str.lower() == tg.lower()]
    if not tg_rows.empty:
        r = tg_rows.iloc[0]
        tg_coords.append((r["gene_name"], str(r["chrom"]), int(r["start"]), int(r["end"])))
assert tg_coords, f"None of target genes {TARGET_GENES} found!"

locus_chr = tg_coords[0][1]
locus_start = max(0, min(c[2] for c in tg_coords) - LOCUS_FLANK_BP)
locus_end = max(c[3] for c in tg_coords) + LOCUS_FLANK_BP
locus_len = locus_end - locus_start

print(f"[+] Scanning locus {TARGET_LOCUS_NAME}: {locus_chr}:{locus_start:,}-{locus_end:,} ({locus_len:,} bp)")
print(f"[*] Active Openness Gate threshold (late): {OPEN_GATE_LATE_THRESH:.2f} | Medaka QC passed: {MEDAKA_PASSED_QC}")

window_bp = 2114
stride_bp = 250

positions, target_pred_abs, target_pred_rel, target_pred_late = scan_locus_sliding_window(
    grafted_model=grafted_zf_model,
    genome_fa_path=TARGET_GENOME_FA,
    chrom=locus_chr,
    start_bp=locus_start,
    end_bp=locus_end,
    window_bp=window_bp,
    stride_bp=stride_bp,
    batch_size=64,
    device=device,
    return_late=True
)

target_pred_rel_gated = np.where(target_pred_late >= OPEN_GATE_LATE_THRESH, target_pred_rel, np.nan)
n_open = np.sum(target_pred_late >= OPEN_GATE_LATE_THRESH)
print(f"[+] Scored {len(positions)} windows across locus; {n_open} / {len(positions)} ({100*n_open/len(positions):.1f}%) pass openness gate (late >= {OPEN_GATE_LATE_THRESH:.2f})")

is_open_window = target_pred_late >= OPEN_GATE_LATE_THRESH
runs = []
cur_run = []
for idx, op in enumerate(is_open_window):
    if op:
        cur_run.append(idx)
    elif cur_run:
        runs.append(cur_run)
        cur_run = []
if cur_run:
    runs.append(cur_run)

raw_cands = []
for run in runs:
    run_arr = np.array(run)
    best_run_idx = run_arr[np.argmax(target_pred_abs[run_arr])]
    pk_abs = float(target_pred_abs[best_run_idx])
    pk_late = float(target_pred_late[best_run_idx])
    pk_rel_gated = float(target_pred_rel_gated[best_run_idx])
    c_pos = int(positions[best_run_idx])
    e_st = max(locus_start, int(positions[run[0]] - window_bp // 2))
    e_en = min(locus_end, int(positions[run[-1]] + window_bp // 2))
    d_min = float("inf")
    near_gene = tg_coords[0][0]
    for gname, _, gst, gen in tg_coords:
        dist = 0 if (gst <= c_pos <= gen) else min(abs(c_pos - gst), abs(c_pos - gen))
        if dist < d_min:
            d_min = dist
            near_gene = gname
    raw_cands.append({
        "chrom": locus_chr, "start": e_st, "end": e_en, "center": c_pos,
        "peak_y_abs": round(pk_abs, 4), "peak_late": round(pk_late, 4),
        "peak_y_rel_gated": round(pk_rel_gated, 4), "nearest_gene": near_gene,
        "distance_to_nearest_gene": int(d_min), "window_count": len(run)
    })

raw_cands.sort(key=lambda x: x["peak_y_abs"], reverse=True)
kept_candidates = []
for c in raw_cands:
    if not any(abs(c["center"] - kc["center"]) < 1500 for kc in kept_candidates):
        c["candidate_rank"] = len(kept_candidates) + 1
        c["is_candidate"] = True
        c["is_enhancer"] = True
        c["pred_abs"] = c["peak_y_abs"]
        c["pred_rel"] = c["peak_y_rel_gated"]
        c["pred_late"] = c["peak_late"]
        kept_candidates.append(c)

print(f"[+] Identified {len(kept_candidates)} non-redundant open candidate elements (ranked strictly by peak y_abs).")

# Cross-species zebrafish ortholog scan
zf_locus_info = None
zf_pred_abs = None
zf_genes_dict = {}
try:
    with (gzip.open(ZEBRAFISH_GTF, "rt") if str(ZEBRAFISH_GTF).endswith(".gz") else open(ZEBRAFISH_GTF, "rt")) as f:
        for line in f:
            if line.startswith("#"):
                continue
            p = line.strip().split("\t")
            if len(p) >= 9 and p[2] == "gene":
                for tg in TARGET_GENES:
                    tg_l = tg.lower()
                    if f'gene_name "{tg_l}"' in p[8].lower() or f'"{tg_l}"' in p[8].lower():
                        zf_genes_dict[tg_l] = (p[0], int(p[3]), int(p[4]))
    if zf_genes_dict:
        primary_tg = TARGET_GENES[0].lower()
        zf_chr = zf_genes_dict[primary_tg][0] if primary_tg in zf_genes_dict else list(zf_genes_dict.values())[0][0]
        chr_genes = {k: v for k, v in zf_genes_dict.items() if v[0] == zf_chr}
        zf_st = max(0, min(v[1] for v in chr_genes.values()) - LOCUS_FLANK_BP)
        zf_en = max(v[2] for v in chr_genes.values()) + LOCUS_FLANK_BP
        print(f"[+] Scanning orthologous Zebrafish locus: {zf_chr}:{zf_st:,}-{zf_en:,}")
        zf_positions, zf_pred_abs, zf_pred_rel, zf_pred_late = scan_locus_sliding_window(
            grafted_model=grafted_zf_model,
            genome_fa_path=ZEBRAFISH_GENOME_FA,
            chrom=zf_chr, start_bp=zf_st, end_bp=zf_en,
            window_bp=window_bp, stride_bp=stride_bp,
            batch_size=64, device=device, return_late=True
        )
        zf_pred_rel_gated = np.where(zf_pred_late >= OPEN_GATE_LATE_THRESH, zf_pred_rel, np.nan)
        zf_locus_info = (zf_chr, zf_st, zf_en, zf_positions, zf_pred_abs, zf_pred_rel_gated, zf_pred_late, zf_pred_rel)
        zf_max_abs = float(np.max(zf_pred_abs))
        for cand in kept_candidates:
            cand["zf_ortholog_max_abs"] = round(zf_max_abs, 4)
            cand["lineage_gain_delta"] = round(cand["peak_y_abs"] - zf_max_abs, 4)
            cand["lacks_zf_ortholog_peak"] = bool(cand["peak_y_abs"] > zf_max_abs)
        print(f"[+] Evaluated lineage-gain delta vs Zebrafish ortholog max ({zf_max_abs:.4f}).")
except Exception as e:
    print(f"[-] Notice scanning Zebrafish orthologous locus: {e}")

cand_tsv_path = res_dir / "candidate_elements.tsv"
cand_bed_path = res_dir / "candidate_elements.bed"
legacy_tsv_path = res_dir / f"{TARGET_LOCUS_NAME}_candidate_enhancers.tsv"
legacy_bed_path = res_dir / f"{TARGET_LOCUS_NAME}_candidate_enhancers.bed"
bg_abs_path = res_dir / f"{TARGET_LOCUS_NAME}_pred_y_abs.bedgraph"
bg_late_path = res_dir / f"{TARGET_LOCUS_NAME}_pred_late.bedgraph"
bg_rel_gated_path = res_dir / f"{TARGET_LOCUS_NAME}_pred_y_rel_gated.bedgraph"

df_candidates = pd.DataFrame(kept_candidates)
df_candidates.to_csv(cand_tsv_path, sep="\t", index=False)
df_candidates.to_csv(legacy_tsv_path, sep="\t", index=False)

with open(cand_bed_path, "w") as f_bed, open(legacy_bed_path, "w") as f_lbed:
    for _, c in df_candidates.iterrows():
        b_line = f"{c['chrom']}\t{c['start']}\t{c['end']}\t{TARGET_LOCUS_NAME}_cand_{c['candidate_rank']}\t{int(c['peak_y_abs'] * 100)}\t.\t{c['center']}\t{c['center']}\n"
        f_bed.write(b_line)
        f_lbed.write(b_line)

half_stride = stride_bp // 2
with open(bg_abs_path, "w") as f_a, open(bg_late_path, "w") as f_l, open(bg_rel_gated_path, "w") as f_r:
    for pos, sa, sl, sr_g in zip(positions, target_pred_abs, target_pred_late, target_pred_rel_gated):
        step_st = max(locus_start, pos - half_stride)
        step_en = max(step_st + 1, min(locus_end, pos + half_stride))
        f_a.write(f"{locus_chr}\t{step_st}\t{step_en}\t{sa:.4f}\n")
        f_l.write(f"{locus_chr}\t{step_st}\t{step_en}\t{sl:.4f}\n")
        if np.isfinite(sr_g):
            f_r.write(f"{locus_chr}\t{step_st}\t{step_en}\t{sr_g:.4f}\n")

print(f"[+] Exported candidate elements ({len(df_candidates)}) to {cand_tsv_path.name}")
cols_to_print = [c for c in ["candidate_rank", "chrom", "start", "end", "center", "peak_y_abs", "peak_late", "peak_y_rel_gated", "nearest_gene", "distance_to_nearest_gene", "lineage_gain_delta", "lacks_zf_ortholog_peak"] if c in df_candidates.columns]
print(df_candidates[cols_to_print].to_string())

def _draw_gtf_locus_genes(ax_g, gtf_genes_dict, chrom_q, reg_st, reg_en, tg_list):
    seen_ids = set()
    rows = []
    for g_rec in gtf_genes_dict.values():
        if g_rec.gene_id in seen_ids:
            continue
        seen_ids.add(g_rec.gene_id)
        gn = g_rec.gene_name if g_rec.gene_name else g_rec.gene_id
        if (gn.startswith("ENS") or gn.startswith("LOC")) and gn.lower() not in [t.lower() for t in tg_list]:
            continue
        if str(g_rec.chrom).replace("chr", "") == str(chrom_q).replace("chr", "") and g_rec.end >= reg_st and g_rec.start <= reg_en:
            rows.append({
                "name": gn, "st": max(g_rec.start, reg_st), "en": min(g_rec.end, reg_en),
                "strand": g_rec.strand, "is_tg": gn.lower() in [t.lower() for t in tg_list]
            })
    rows.sort(key=lambda r: r["st"])
    x_span_mb = max(0.02, (reg_en - reg_st) / 1e6)
    track_ends = []
    track_ys = [0.16, 0.48, 0.78]
    for g in rows:
        x0, x1 = g["st"] / 1e6, g["en"] / 1e6
        mid = 0.5 * (x0 + x1)
        lbl = f"{g['name']} {'→' if g['strand'] == '+' else '←'}"
        half_w = max(0.0045, 0.52 * len(lbl) * (x_span_mb / 145.0))
        occ0, occ1 = min(x0, mid - half_w), max(x1, mid + half_w)
        tr_i = next((i for i, e in enumerate(track_ends) if occ0 > e + 0.002), None)
        if tr_i is None:
            tr_i = len(track_ends)
            track_ends.append(occ1)
        else:
            track_ends[tr_i] = occ1
        y_g = track_ys[tr_i % len(track_ys)]
        col = "#d73027" if g["is_tg"] else "#2b5c8f"
        ax_g.plot([x0, x1], [y_g, y_g], lw=4.0 if g["is_tg"] else 2.4, color=col, solid_capstyle="round", zorder=3)
        ax_g.text(mid, y_g + 0.04, lbl, ha="center", va="bottom", fontsize=7.8, fontweight="bold" if g["is_tg"] else "semibold",
                  color=col, bbox=dict(boxstyle="round,pad=0.14", fc="white", ec=col, lw=0.7, alpha=0.90), zorder=5)
    ax_g.set_ylim(0, 1.15)
    ax_g.set_yticks([])
    ax_g.grid(True, linestyle=":", alpha=0.35, axis="x")

# Synchronized 5-Row x 2-Column Multi-Track Locus Profile (Figure 3)
ctx_3 = get_inference_context(work_dir=WORK_DIR, device=device)
gtf_t = ctx_3.get_gtf_data(TARGET_SPECIES_NAME)["genes"]
gtf_z = ctx_3.get_gtf_data("Danio_rerio")["genes"]

fig, axes = plt.subplots(5, 2, figsize=(19, 14.5), sharex='col', gridspec_kw={"height_ratios": [0.95, 1.0, 1.1, 1.0, 1.0], "hspace": 0.28, "wspace": 0.16})

# Row 0: Annotated GTF Gene Models (Target Left, Zebrafish Right)
_draw_gtf_locus_genes(axes[0, 0], gtf_t, locus_chr, locus_start, locus_end, TARGET_GENES)
axes[0, 0].set_ylabel("Genes", fontsize=10, fontweight="bold")
axes[0, 0].set_title(f"Target Species: {TARGET_SPECIES_NAME} (Chr {locus_chr}:{locus_start:,}-{locus_end:,})", fontweight="bold", fontsize=11.5)

# Row 1: Predicted Late Dome Openness (Openness Gate = 5.70)
axes[1, 0].plot(positions / 1e6, target_pred_late, color="#756bb1", lw=1.6, label="Predicted late openness (dome)")
axes[1, 0].axhline(OPEN_GATE_LATE_THRESH, color="black", linestyle="--", lw=1.3, label=f"Openness Gate ({OPEN_GATE_LATE_THRESH:.2f})")
axes[1, 0].fill_between(positions / 1e6, OPEN_GATE_LATE_THRESH, target_pred_late, where=(target_pred_late >= OPEN_GATE_LATE_THRESH), color="#756bb1", alpha=0.22)
axes[1, 0].set_ylabel("pred_late\n(Openness)", fontsize=10, fontweight="bold")
axes[1, 0].legend(loc="upper right", fontsize=8.3, frameon=True)
axes[1, 0].grid(True, linestyle=":", alpha=0.4)

# Row 2: Predicted Pre-MZT y_abs + Candidate Rank Peaks
axes[2, 0].plot(positions / 1e6, target_pred_abs, color="#d7191c", lw=1.8, label=r"Predicted $y_{abs}$ (256-cell)")
for _, c in df_candidates.iterrows():
    axes[2, 0].axvline(c["center"] / 1e6, color="#d7191c", linestyle=":", alpha=0.45, lw=1.0)
    axes[2, 0].annotate(f"C{c['candidate_rank']}", (c["center"] / 1e6, c["peak_y_abs"]), xytext=(0, 3), textcoords="offset points", fontsize=8.0, fontweight="bold", color="#990000", ha="center")
axes[2, 0].set_ylabel(r"pred_$y_{abs}$" + "\n(Pre-MZT)", fontsize=10, fontweight="bold")
axes[2, 0].legend(loc="upper right", fontsize=8.3, frameon=True)
axes[2, 0].grid(True, linestyle=":", alpha=0.4)

# Row 3: Un-gated y_rel context + Filled Gated y_rel_gated
x_mb_t = positions / 1e6
open_mask_t = target_pred_late >= OPEN_GATE_LATE_THRESH
axes[3, 0].plot(x_mb_t, target_pred_rel, color="#bdbdbd", lw=0.9, alpha=0.75, label=r"Un-gated $y_{rel}$ (closed background)")
axes[3, 0].fill_between(x_mb_t, 0.0, np.where(open_mask_t & (target_pred_rel >= 0), target_pred_rel, 0.0), color="#2b83ba", alpha=0.82, step="mid", label=r"$y_{resid\_gated} \geq 0$ (Early-biased, late≥5.7)")
axes[3, 0].fill_between(x_mb_t, 0.0, np.where(open_mask_t & (target_pred_rel < 0), target_pred_rel, 0.0), color="#e66101", alpha=0.78, step="mid", label=r"$y_{resid\_gated} < 0$ (Late-biased, late≥5.7)")
axes[3, 0].axhline(0, color="black", linestyle=":", lw=1.0, alpha=0.7)
axes[3, 0].set_ylabel(r"$y_{resid\_gated}$" + "\n(Early vs Late)", fontsize=10, fontweight="bold")
axes[3, 0].legend(loc="upper left", ncol=3, fontsize=7.4, frameon=True)
axes[3, 0].grid(True, linestyle=":", alpha=0.4)

# Row 4 Left: Candidate Peak Intervals
ax_cand = axes[4, 0]
first_gain, first_cons = True, True
for _, c in df_candidates.iterrows():
    is_g = bool(c.get("lacks_zf_ortholog_peak", False))
    col_c = "#d73027" if is_g else "#2b5c8f"
    lbl_c = None
    if is_g and first_gain:
        lbl_c = "Peak > ZF Ortholog Max (Activity Gain)"
        first_gain = False
    elif (not is_g) and first_cons:
        lbl_c = "Open Candidate (Within ZF Activity Range)"
        first_cons = False
    rect = patches.Rectangle((c["start"] / 1e6, 0.0), (c["end"] - c["start"]) / 1e6, c["peak_y_abs"], facecolor=col_c, alpha=0.55, edgecolor="black", lw=0.8, label=lbl_c)
    ax_cand.add_patch(rect)
    ax_cand.text(c["center"] / 1e6, c["peak_y_abs"] + 0.06, f"C{int(c['candidate_rank'])}", ha="center", va="bottom", fontsize=7.5, fontweight="bold", color=col_c)
if zf_locus_info is not None:
    ax_cand.axhline(float(np.max(zf_pred_abs)), color="#636363", linestyle="--", lw=1.2, label=f"ZF Locus Max $y_{{abs}}$ ({float(np.max(zf_pred_abs)):.2f})")
ax_cand.set_ylim(0, max(3.1, float(df_candidates["peak_y_abs"].max()) * 1.24))
ax_cand.set_ylabel("Candidate Spans\n(Zero-Shot)", fontsize=10, fontweight="bold")
ax_cand.set_xlabel(f"Genomic Coordinate on {TARGET_SPECIES_NAME} Chr {locus_chr} (Mb)", fontsize=10.5, fontweight="bold")
ax_cand.legend(loc="upper left", fontsize=7.8, frameon=True)
ax_cand.grid(True, linestyle=":", alpha=0.4)

if zf_locus_info is not None:
    z_chr, z_st, z_en, z_pos, z_abs, z_rel_g, z_late, z_rel = zf_locus_info
    _draw_gtf_locus_genes(axes[0, 1], gtf_z, z_chr, z_st, z_en, TARGET_GENES)
    axes[0, 1].set_title(f"Held-Out Orthologous Zebrafish Locus (Chr {z_chr}:{z_st:,}-{z_en:,})", fontweight="bold", fontsize=11.5)

    axes[1, 1].plot(z_pos / 1e6, z_late, color="#756bb1", lw=1.6, label="ZF Pred late (dome)")
    axes[1, 1].axhline(OPEN_GATE_LATE_THRESH, color="black", linestyle="--", lw=1.3, label=f"Openness Gate ({OPEN_GATE_LATE_THRESH:.2f})")
    axes[1, 1].fill_between(z_pos / 1e6, OPEN_GATE_LATE_THRESH, z_late, where=(z_late >= OPEN_GATE_LATE_THRESH), color="#756bb1", alpha=0.22)
    axes[1, 1].set_ylim(axes[1, 0].get_ylim())
    axes[1, 1].legend(loc="upper right", fontsize=8.3, frameon=True)
    axes[1, 1].grid(True, linestyle=":", alpha=0.4)

    axes[2, 1].plot(z_pos / 1e6, z_abs, color="#d7191c", lw=1.8, label=r"ZF Pred $y_{abs}$ (256-cell)")
    axes[2, 1].set_ylim(axes[2, 0].get_ylim())
    axes[2, 1].legend(loc="upper right", fontsize=8.3, frameon=True)
    axes[2, 1].grid(True, linestyle=":", alpha=0.4)

    x_mb_z = z_pos / 1e6
    open_mask_z = z_late >= OPEN_GATE_LATE_THRESH
    axes[3, 1].plot(x_mb_z, z_rel, color="#bdbdbd", lw=0.9, alpha=0.75, label=r"ZF Un-gated $y_{rel}$ (closed background)")
    axes[3, 1].fill_between(x_mb_z, 0.0, np.where(open_mask_z & (z_rel >= 0), z_rel, 0.0), color="#2b83ba", alpha=0.82, step="mid", label=r"ZF $y_{resid\_gated} \geq 0$")
    axes[3, 1].fill_between(x_mb_z, 0.0, np.where(open_mask_z & (z_rel < 0), z_rel, 0.0), color="#e66101", alpha=0.78, step="mid", label=r"ZF $y_{resid\_gated} < 0$")
    axes[3, 1].axhline(0, color="black", linestyle=":", lw=1.0, alpha=0.7)
    axes[3, 1].set_ylim(axes[3, 0].get_ylim())
    axes[3, 1].legend(loc="upper left", ncol=3, fontsize=7.4, frameon=True)
    axes[3, 1].grid(True, linestyle=":", alpha=0.4)

    # Row 4 Right: Observed ZF ATAC CPM + MACS2/Poisson Dome Reference Peaks
    ax_obs = axes[4, 1]
    zf_early_bw = Path(WORK_DIR) / "atac" / "Danio_rerio" / "Danio_rerio_early_cpm.bw"
    zf_late_bw = Path(WORK_DIR) / "atac" / "Danio_rerio" / "Danio_rerio_late_cpm.bw"
    if zf_early_bw.exists():
        try:
            with pyBigWig.open(str(zf_early_bw)) as bw_e:
                cl = bw_e.chroms()
                ch_key = z_chr if z_chr in cl else f"chr{z_chr}"
                z_en_c = min(z_en, cl.get(ch_key, z_en))
                z_st_c = min(z_st, max(0, z_en_c - 1))
                vals_e = np.nan_to_num(bw_e.values(ch_key, z_st_c, z_en_c))
            ax_obs.plot(np.linspace(z_st_c / 1e6, z_en_c / 1e6, len(vals_e)), np.log2(vals_e + 1.0), color="#2ca25f", lw=1.1, label="Observed 256-cell ATAC log2(CPM+1)")
        except Exception as e_bw:
            print(f"[-] Notice reading ZF early BigWig: {e_bw}")
    if zf_late_bw.exists():
        try:
            with pyBigWig.open(str(zf_late_bw)) as bw_l:
                cl = bw_l.chroms()
                ch_key = z_chr if z_chr in cl else f"chr{z_chr}"
                z_en_c = min(z_en, cl.get(ch_key, z_en))
                z_st_c = min(z_st, max(0, z_en_c - 1))
                vals_l = np.nan_to_num(bw_l.values(ch_key, z_st_c, z_en_c))
            ax_obs.plot(np.linspace(z_st_c / 1e6, z_en_c / 1e6, len(vals_l)), np.log2(vals_l + 1.0), color="#8856a7", lw=1.0, ls="--", alpha=0.85, label="Observed Dome ATAC log2(CPM+1)")
        except Exception as e_bwl:
            print(f"[-] Notice reading ZF late BigWig: {e_bwl}")

    zf_peaks_file = Path(WORK_DIR) / "atac" / "Danio_rerio" / "peaks" / "Danio_rerio_late_peaks.narrowPeak"
    if not zf_peaks_file.exists():
        zf_peaks_file = Path(WORK_DIR) / "atac" / "Danio_rerio" / "Danio_rerio_late_peaks.primary.narrowPeak"
    if zf_peaks_file.exists():
        try:
            df_zf_pks = pd.read_csv(zf_peaks_file, sep="\t", header=None, dtype={0: str})
            p_ch = z_chr.replace("chr", "")
            in_locus = df_zf_pks[((df_zf_pks[0] == z_chr) | (df_zf_pks[0] == p_ch) | (df_zf_pks[0] == f"chr{p_ch}")) & (df_zf_pks[1] < z_en) & (df_zf_pks[2] > z_st)]
            first_pk = True
            for _, pk in in_locus.iterrows():
                pk_s, pk_e = max(z_st, int(pk[1])), min(z_en, int(pk[2]))
                lbl = "MACS2/Poisson Dome Reference Peaks" if first_pk else None
                ax_obs.axvspan(pk_s / 1e6, pk_e / 1e6, color="orange", alpha=0.32, label=lbl)
                first_pk = False
        except Exception as e_pk:
            print(f"[-] Notice overlaying ZF Poisson peaks: {e_pk}")

    ax_obs.set_ylabel("Observed ZF ATAC\nlog2(CPM+1)", fontsize=10, fontweight="bold")
    ax_obs.set_xlabel(f"Genomic Coordinate on Held-Out Zebrafish Chr {z_chr} (Mb)", fontsize=10.5, fontweight="bold")
    ax_obs.legend(loc="upper left", fontsize=7.5, frameon=True)
    ax_obs.grid(True, linestyle=":", alpha=0.4)

fig.subplots_adjust(top=0.95, bottom=0.05, left=0.06, right=0.98)
fig3_path = res_dir / "fig3_locus_scan_profile.png"
plt.savefig(fig3_path, dpi=300, bbox_inches="tight")
plt.savefig(res_dir / "fig3_locus_scan_profile.svg", bbox_inches="tight")
plt.show()
print(f"[+] Figure 3 exported to {fig3_path}")


---
## Block 6: Base-Resolution Attribution, Teleost-Normalized Pre-MZT TF Filtering & Evolutionary Swap

Dissects causal nucleotide sequence motifs inside every candidate regulatory element discovered in Block 5:
1. **Gradient Saliency & In Silico Mutagenesis (ISM):** Computes base-pair contribution logos for predicted pre-MZT accessibility (`y_abs`) and timing (`y_rel`), extracting high-contribution TF binding *seqlets*.
2. **Pre-MZT TF Filtering & TOMTOM Matching:** Matches seqlets against both an embryo-expressed maternal/pre-MZT TF database (`premzt_tfs.meme` via teleost-to-vertebrate paralog normalization) and `JASPAR 2024 CORE vertebrates` (`q`-value = FDR-adjusted match probability).
3. **Reciprocal Evolutionary Swap & Genome-Wide Enrichment:** Aligns homologous locus elements between target and reference genomes, runs *in silico* motif ablation ($\Delta y_{\text{abs}}$) and cross-species sequence swaps, and tests motif & CpG-island (`observed/expected CpG`) enrichment across genome-wide minor-wave vs. GC-matched silent promoters.

**Generated Outputs (Block 6):**
* **Figure 4a (`fig4a_locus_cis_regulatory_landscape.png` & `.svg`):** (`A`) Locus accessibility & timing tracks with lineage-gained/conserved shading, top causal motif callouts, and GTF gene models below $y=0$; (`B`) top 15 locus motifs ranked by *in silico* ablation $\Delta y_{\text{abs}}$ colored by maternal TF match; (`C`) ablation vs. evolutionary swap scatter with visual legends separating feature class, primary vs. secondary motif dot size, and maternal TF rings, labeling every point without overlap.
* **Figure 4b (`fig4b_single_nt_attribution_tracks.png` & `.svg`):** Single-nucleotide gradient attribution logos across top candidate elements highlighting seqlet boxes, outside-window peak motif notes when summits sit beyond $\pm 80$ bp, and homologous target-vs-reference alignments.
* **Figure 4c (`fig4c_genomewide_minor_wave_motifs.png` & `.svg`):** Genome-wide promoter TF incidence comparison and minor-wave vs. silent-control CpG observed/expected violin distributions.
* **Motif Tables (`locus_cis_regulatory_motifs.tsv`, `evolutionary_swap_summary.tsv`, `minor_wave_genomewide_top_motifs.tsv`, `candidate_1_causal_motif.meme`):** Full seqlet coordinates, PWMs, TOMTOM $q$-values, and ablation/swap $\Delta$ scores.

In [ ]:
import os, sys, json
from pathlib import Path
import torch
import pandas as pd
from IPython.display import Image, display
device = 'cuda' if torch.cuda.is_available() else 'cpu'
window_bp = 2114

with open(Path(WORK_DIR) / 'config.json') as f:
    config = json.load(f)

_og_path = Path(WORK_DIR) / 'models' / 'zebrafish' / 'openness_gate.json'
OPEN_GATE_LATE_THRESH = 5.70
if _og_path.exists():
    try:
        with open(_og_path) as _f_og:
            OPEN_GATE_LATE_THRESH = float(json.load(_f_og).get('OPEN_GATE_LATE_THRESH', 5.70))
    except Exception:
        OPEN_GATE_LATE_THRESH = 5.70

MEDAKA_PASSED_QC = False

from scripts.infer_and_attribute import dissect_cis_regulatory_landscape, screen_genomewide_minor_wave_motifs

landscape_results = dissect_cis_regulatory_landscape(
    work_dir=WORK_DIR,
    locus_name=TARGET_LOCUS_NAME,
    tf_list_path=PREMZT_TF_LIST_PATH,
    open_gate_late_thresh=OPEN_GATE_LATE_THRESH,
    device=device,
    window_bp=window_bp,
    central_bp=1000
)

df_motifs = landscape_results['df_motifs']
df_elements = landscape_results['df_elements']

print(f'[+] Successfully dissected all {len(df_elements)} candidate elements ({len(df_motifs)} total causal motifs).')
print(f'[+] Displaying Publication Figure 4a (Whole Locus Cis-Regulatory Landscape):')
display(Image(filename=landscape_results['fig4a_png'], width=900))

print(f'[+] Displaying Publication Figure 4b (Single-Nucleotide Attribution Tracks for Key Elements):')
display(Image(filename=landscape_results['fig4b_png'], width=900))

print('' + '=' * 80)
print('[*] Running Genome-Wide High-Confidence Cichlid Minor-Wave Motif Screen...')
print('=' * 80)

screen_results = screen_genomewide_minor_wave_motifs(
    work_dir=WORK_DIR,
    locus_name=TARGET_LOCUS_NAME,
    device=device,
    tf_list_path=PREMZT_TF_LIST_PATH
)

print(f'[+] Displaying Publication Figure 4c (Genome-Wide Minor-Wave Motifs & CpG Priming):')
display(Image(filename=screen_results['fig4c_png'], width=900))


---
## Block 7: Multi-Task `NTv3` Fine-Tuning & Publication Baselines Benchmark

Trains a multi-task 65,536 bp DNA foundation transformer (**`NTv3`**, central 24,576 bp prediction crop) using activation checkpointing + **LoRA (`r=8, alpha=16`)** strictly on the reference training chromosomes (`scripts/train_ntv3.py`), reusing Block 1b's processed nascent SLAM-seq BigWigs alongside blastula ATAC-seq BigWigs (`10` total stage/strand tracks optimized under `TrackBalancedLoss` weighting early `256-cell` & `0.75/2.5 hpf` tracks `3×` to prevent sparse-signal collapse).

When `NTV3_RUN_BASELINES = True`, benchmarks three transformer training regimes against grafted ChromBPNet on the **exact same held-out test chromosomes** (`chr_fold.json`):
1. **`NTv3_raw_pretrained`:** Unadapted pretrained tracks out of the box.
2. **`NTv3_frozen_head`:** Frozen transformer backbone with a trained 10-track linear readout head—quantifying how much generic sequence representation the pretrained trunk already provides before LoRA adaptation.
3. **`NTv3_multitask_lora`:** Joint LoRA + head adaptation, saving exact hyperparameters in `WORK_DIR/models/ntv3/ntv3_train_config.json` (`ntv3_baselines_benchmark.tsv`).

In [ ]:
# Launch Multi-Task NTv3 Epigenomics/SLAM-seq Fine-Tuning Task
zf_atac_dir = Path(WORK_DIR) / "atac" / "Danio_rerio"
chrom_sizes_zf = zf_atac_dir / "chrom.sizes"
chr_fold_json_zf = (zf_atac_dir / "chr_folds.json") if len(FOLDS_TO_TRAIN) > 1 else (zf_atac_dir / "chr_fold.json")

for f_idx in FOLDS_TO_TRAIN:
    fold_arg = ["--fold", str(f_idx)] if len(FOLDS_TO_TRAIN) > 1 else []
    task_name = f"train_ntv3_zf_fold_{f_idx}" if len(FOLDS_TO_TRAIN) > 1 else "train_ntv3_zf"

    cmd_base = [
        "torchrun",
        "--nproc_per_node", str(NUM_GPUS)
    ] if (COMPUTE_PROFILE == "hpc_1mb" or NUM_GPUS > 1) else ["python", "-u"]

    ntv3_cmd = cmd_base + [
        "scripts/train_ntv3.py",
        "--work-dir", WORK_DIR,
        "--genome-fa", str(ZEBRAFISH_GENOME_FA),
        "--chrom-sizes", str(chrom_sizes_zf),
        "--chr-fold-json", str(chr_fold_json_zf),
        "--model-id", NTV3_MODEL_ID,
        "--compute-profile", COMPUTE_PROFILE,
        "--context-len", str(NTV3_CONTEXT_LEN),
        "--train-mode", NTV3_TRAIN_MODE,
        "--batch-size", str(NTV3_BATCH_SIZE),
        "--grad-accum", str(NTV3_GRAD_ACCUM),
        "--task-name", task_name,
        "--epochs", "3"
    ]
    if NTV3_GRAD_CHECKPOINTING:
        ntv3_cmd.append("--grad-checkpointing")
    else:
        ntv3_cmd.append("--no-grad-checkpointing")

    ckpt_best = Path(WORK_DIR) / "models" / "ntv3" / "ntv3_best_checkpoint.pt"
    if ckpt_best.exists() and not FORCE_RERUN:
        ntv3_cmd.append("--eval-only")
    if NTV3_RUN_BASELINES:
        ntv3_cmd.append("--run-baselines")
    if ENABLE_OPTION2_TARGET_RNA_ABLATION:
        ntv3_cmd.append("--enable-option2-target-rna")
    ntv3_cmd.extend(fold_arg)

    env_vars = {}
    if HF_TOKEN:
        env_vars["HF_TOKEN"] = HF_TOKEN

    tm.submit(task_name, ntv3_cmd, env_vars=env_vars, depends_on=["prep_atac_danio_rerio", "prep_slamseq_zf"], force_rerun=FORCE_RERUN, requires_gpu=True)

display(tm.status())

In [ ]:
# ==============================================================================
# Monitor Multi-Task NTv3 Training & Inspect Publication Baselines Benchmark
# ==============================================================================
display(tm.status())

bench_tsv = Path(WORK_DIR) / "models" / "ntv3" / "ntv3_baselines_benchmark.tsv"
if not bench_tsv.exists():
    bench_tsv_f0 = Path(WORK_DIR) / "models" / "ntv3" / "ntv3_baselines_benchmark_fold_0.tsv"
    if bench_tsv_f0.exists():
        bench_tsv = bench_tsv_f0

if bench_tsv.exists():
    df_baselines = pd.read_csv(bench_tsv, sep="\t")
    print("=" * 80)
    print("[*] Publication Cross-Architecture & Baselines Benchmark:")
    print(df_baselines.to_string())
    print("=" * 80)
    display(df_baselines)
else:
    print("[*] NTv3 benchmark table not yet generated (requires completed model training).")
    print("    Task 'train_ntv3_zf' is currently queued or in progress.")

---
## Block 8: `NTv3` Inference, Distal Perturbation & Cross-Architecture Comparison

Runs downstream inference (`scripts/ntv3_infer.py`) using the fine-tuned multi-task **`NTv3`** checkpoint and benchmarks it directly against **`ChromBPNet`**:
1. **Zero-Shot Cohort Scoring:** Scores target-species genes across matched ATAC windows (`promoter_1kb`, `promoter_intron1_capped`, `proximal_5kb`, full crop) and strand-matched **QuantSeq 3'-end TES-centered** (`1,000 bp` integrated signal per kb) + **exon-normalized** nascent SLAM-seq tracks. *(Note: To refresh cached `1kb` vs. `intron1` NTv3 cohort columns after updating scripts, run `--subtask score --force-rescore`.)*
2. **Locus Scan, Distal Coupling Screen & Full-Locus Saliency:** Tiles target and reference loci across overlapping 65 kb windows, tests top candidate enhancers under 500 bp dinucleotide shuffle (which preserves overall GC/CpG density while scrambling base order), CpG ablation (`CG -> TG`), and evolutionary motif swaps, and computes base-resolution gradient saliency ($|\nabla|$). **Total-locus SLAM saliency** sums gradients across every base/gene in the window (highlighting shared chromatin/regulatory bases that overlap ATAC peaks), whereas **gene-exon-specific SLAM saliency** isolates gradients driving transcription of each target gene's exons specifically. Because reference pre-MZT nascent SLAM BigWigs are extremely sparse ($<0.04\%$ non-zero bases genome-wide) and `F.softplus` + bias terms create a positive baseline floor, predicted `+` and `-` strand SLAM tracks track shared promoter openness closely with a small strand bias offset.

**Generated Outputs (Block 8a/8b):**
* **Figure 5 (`fig5_cross_architecture_locus_overlay.png` & `.svg`):** Five-track locus alignment across signed distance from the primary target TSS: (`A`) ChromBPNet vs. NTv3 ATAC accessibility with TF badges, (`B`) strand-resolved NTv3 pre-MZT nascent SLAM prediction (with co-variation & strand-offset note), (`C`) independent **dual-axis** locus base saliency (`ATAC |\nabla|` left axis vs. total-locus & target-exon-specific `SLAM |\nabla|` right `twinx` axis), (`D`) GTF gene models, and (`E`) orthologous reference-species NTv3 profile.
* **Figure 6 (`fig6_knockout_and_motif_saliency.png` & `.svg`):** (`A`) `% change` in local pre-MZT ATAC predicted by NTv3 under 500 bp dinucleotide shuffle, `CG -> TG` CpG ablation (testing regional CpG-island dependence), and causal motif swap across top candidate elements; (`B`) single-nucleotide NTv3 ATAC saliency (`±150 bp`) centered on the primary target TSS.
* **Figure 7 (`fig7_cross_architecture_benchmark.png` & `.svg`):** Head-to-head ROC (`A`) and Precision-Recall (`B`) curves evaluated on the **identical intersected gene cohort** (single shared chance baseline, complementing Block 4's genome-wide benchmark) plus progression bars (`C`) comparing `ChromBPNet_grafted`, `NTv3_raw`, `NTv3_frozen`, and `NTv3_multitask`.
* **Tables & Tracks (`{TARGET_LOCUS_NAME}_cross_architecture_summary.tsv`, `ntv3_locus_scan.tsv`, `ntv3_zf_locus_scan.tsv`, `ntv3_distal_coupling_screen.tsv`, `ntv3_knockout_saliency_summary.tsv`, `ntv3_saliency_tracks.npz`, `*_ntv3_*.bedgraph`).**

In [ ]:
# ==============================================================================
# Block 8a: Launch Standalone NTv3 Inference Worker via TaskManager
# ==============================================================================
block8_cmd = [
    "python", "-u", "scripts/ntv3_infer.py",
    "--work-dir", WORK_DIR,
    "--config", f"{WORK_DIR}/config.json",
    "--context-len", str(NTV3_CONTEXT_LEN),
    "--flank-bp", str(LOCUS_FLANK_BP)
]

tm.submit(
    "ntv3_inference",
    block8_cmd,
    depends_on=["train_ntv3_zf"],
    force_rerun=FORCE_RERUN,
    requires_gpu=True
)

display(tm.status())

In [ ]:
# ==============================================================================
# Block 8b: Cross-Architecture Publication Comparisons (Figures 5, 6, 7 & Summary)
# ==============================================================================
from pathlib import Path
import json
import gzip
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as patches
import matplotlib
import logomaker
from scipy import stats
from sklearn.metrics import roc_curve, precision_recall_curve, auc
from IPython.display import Image, display
from scripts.infer_and_attribute import benchmark_transfer_metrics

# Ensure paths and config are resolved
if "WORK_DIR" not in globals():
    WORK_DIR = Path("work_astatotilapia_premzt").resolve()
else:
    WORK_DIR = Path(WORK_DIR).resolve()

if "TARGET_LOCUS_NAME" not in globals():
    with open(WORK_DIR / "config.json") as f_cfg:
        _cfg = json.load(f_cfg)
        TARGET_LOCUS_NAME = _cfg.get("TARGET_LOCUS_NAME", "notch2_slc35a3a")
        TARGET_GENES = _cfg.get("TARGET_GENES", ["notch2", "slc35a3a"])

if "res_dir" not in globals():
    res_dir = WORK_DIR / "results" / TARGET_LOCUS_NAME

# Load summary metrics and results
ntv3_scored_tsv = res_dir / "ntv3_target_genes_scored.tsv"
chrombpnet_scored_tsv = res_dir / "target_genes_scored_metrics.tsv"
locus_scan_tsv = res_dir / "ntv3_locus_scan.tsv"
zf_scan_tsv = res_dir / "ntv3_zf_locus_scan.tsv"
ko_summary_tsv = res_dir / "ntv3_knockout_saliency_summary.tsv"
coupling_tsv = res_dir / "ntv3_distal_coupling_screen.tsv"
saliency_npz_path = res_dir / "ntv3_saliency_tracks.npz"
bench_tsv = Path(WORK_DIR) / "models" / "ntv3" / "ntv3_baselines_benchmark.tsv"
cross_summary_path = res_dir / f"{TARGET_LOCUS_NAME}_cross_architecture_summary.tsv"

# Parse gene spans and exons for TARGET_GENES and flanking genes from TARGET_GTF
target_gene_spans = {}
target_gene_exons = {}
flanking_gene_spans = {}

if "TARGET_GTF" not in globals() or not Path(str(TARGET_GTF)).exists():
    with open(Path(WORK_DIR) / "config.json") as f_cfg:
        TARGET_GTF = json.load(f_cfg).get("TARGET_GTF")

from scripts.premzt_inference import get_inference_context
inf_ctx = get_inference_context(work_dir=WORK_DIR)
gtf_dict = inf_ctx.get_gtf_data(TARGET_SPECIES_NAME)

primary_gene = TARGET_GENES[0]
p_rec = gtf_dict["genes"].get(primary_gene.lower())
if p_rec is None:
    for k, v in gtf_dict["genes"].items():
        if k.lower() == primary_gene.lower() or (v.gene_name and v.gene_name.lower() == primary_gene.lower()):
            p_rec = v
            break

if p_rec is not None:
    p_tss = p_rec.tss
    p_strand = p_rec.strand
    primary_chrom = p_rec.chrom
else:
    p_tss = 0
    p_strand = "+"
    primary_chrom = "1"

for tg in TARGET_GENES:
    rec = gtf_dict["genes"].get(tg.lower())
    if rec:
        target_gene_spans[tg] = (rec.chrom, rec.start, rec.end, rec.strand)
        target_gene_exons[tg] = rec.exons

def to_signed_kb(pos_bp):
    return ((pos_bp - p_tss) if p_strand == "+" else (p_tss - pos_bp)) / 1000.0

# ------------------------------------------------------------------------------
# ------------------------------------------------------------------------------
# FIGURE 5: Synchronized Cross-Architecture Locus Overlay on TARGET_LOCUS_NAME
# ------------------------------------------------------------------------------
print("[*] Generating Figure 5: Cross-Architecture Locus Overlay (Signed Distance)...")
df_scan = pd.read_csv(locus_scan_tsv, sep="\t") if locus_scan_tsv.exists() else pd.DataFrame()
df_zf = pd.read_csv(zf_scan_tsv, sep="\t") if zf_scan_tsv.exists() else pd.DataFrame()
cand_path = res_dir / "candidate_elements.tsv"
motif_path = res_dir / "locus_cis_regulatory_motifs.tsv"
df_cands = pd.read_csv(cand_path, sep="\t") if cand_path.exists() else pd.DataFrame()
df_motifs = pd.read_csv(motif_path, sep="\t") if motif_path.exists() else pd.DataFrame()

bg_abs_path = res_dir / f"{TARGET_LOCUS_NAME}_pred_y_abs.bedgraph"
pred_min_bp = None
pred_max_bp = None
if bg_abs_path.exists():
    try:
        df_cb = pd.read_csv(bg_abs_path, sep="\t", header=None, names=["chr", "st", "en", "val"])
        pred_min_bp = float(df_cb["st"].min())
        pred_max_bp = float(df_cb["en"].max())
    except Exception:
        pass

if pred_min_bp is not None and pred_max_bp is not None:
    x_min_kb = min(to_signed_kb(pred_min_bp), to_signed_kb(pred_max_bp))
    x_max_kb = max(to_signed_kb(pred_min_bp), to_signed_kb(pred_max_bp))
else:
    x_min_kb = -49.07
    x_max_kb = 99.43

fig5 = plt.figure(figsize=(16, 13.5))
gs5 = gridspec.GridSpec(5, 1, height_ratios=[1.4, 1.2, 1.1, 1.1, 1.2], hspace=0.36)

# 1. Panel 1: Target Species Pre-MZT ATAC (ChromBPNet vs NTv3) + Element Spans & Callouts
ax_atac = fig5.add_subplot(gs5[0])
if bg_abs_path.exists() and "df_cb" in locals():
    try:
        cb_mids_kb = [to_signed_kb(0.5 * (s + e)) for s, e in zip(df_cb["st"], df_cb["en"])]
        ax_atac.plot(cb_mids_kb, df_cb["val"], color="#2b5c8f", lw=1.8, label="ChromBPNet Pre-MZT ($y_{abs}$)")
        ax_atac.fill_between(cb_mids_kb, df_cb["val"], color="#2b5c8f", alpha=0.15)
    except Exception:
        pass

if not df_scan.empty:
    nt_mids_kb = [to_signed_kb(c) for c in df_scan["center"]]
    ax_atac.plot(nt_mids_kb, df_scan["atac_256cell"], color="#d73027", lw=1.6, label="NTv3 ATAC (256-cell)")

ax_atac.axvline(0, color="#d73027", linestyle="--", lw=1.4, alpha=0.8, label=f"{primary_gene} TSS (0 kb)")
ax_atac.set_xlim(x_min_kb, x_max_kb)
y_min_at, y_max_at = ax_atac.get_ylim()
ax_atac.set_ylim(y_min_at, max(0.6, y_max_at * 1.25))

# Overlay candidate elements and annotate all elements with matched TFs cleanly below top legend
top_motifs_by_elem = df_motifs.sort_values(by="target_ablation_delta", ascending=True).groupby("element_rank").first().to_dict(orient="index") if not df_motifs.empty else {}
annot_y_levels = [0.96, 0.80, 0.64, 0.50]
annot_idx = 0

if not df_cands.empty:
    for _, c_row in df_cands.sort_values(by="center").iterrows():
        rk = int(c_row["candidate_rank"])
        c_st_kb = to_signed_kb(float(c_row["start"]))
        c_en_kb = to_signed_kb(float(c_row["end"]))
        c_mid_kb = to_signed_kb(float(c_row["center"]))
        x_st = min(c_st_kb, c_en_kb)
        x_en = max(c_st_kb, c_en_kb)
        if x_en < x_min_kb or x_st > x_max_kb:
            continue

        m_info = top_motifs_by_elem.get(rk, {})
        is_lineage = m_info.get("is_lineage_gain", bool(c_row.get("lacks_zf_ortholog_peak", False)))
        feat_cls = m_info.get("gtf_feature_class", "enhancer")

        if is_lineage:
            col = "#d73027"
            alpha = 0.35
            lbl_col = "#990000"
        elif feat_cls == "promoter_tss_2kb":
            col = "#1b7837"
            alpha = 0.25
            lbl_col = "#005a32"
        else:
            col = "#4575b4"
            alpha = 0.18
            lbl_col = "#2166ac"

        ax_atac.axvspan(max(x_st, x_min_kb), min(x_en, x_max_kb), color=col, alpha=alpha, zorder=1)

        tf_call = m_info.get("display_name", "")
        if tf_call and tf_call != "None" and tf_call != "Unmatched" and (x_min_kb <= c_mid_kb <= x_max_kb):
            ov_gene = m_info.get("overlapping_gene", "")
            badge = f"R{rk} ({ov_gene})\n{tf_call}"
            y_annot = y_max_at * annot_y_levels[annot_idx % len(annot_y_levels)]
            annot_idx += 1
            ax_atac.annotate(
                badge,
                xy=(c_mid_kb, float(c_row.get("pred_abs", 2.0))),
                xytext=(c_mid_kb, y_annot),
                arrowprops=dict(arrowstyle="->", color=lbl_col, lw=1.0),
                fontsize=7.8,
                fontweight="bold",
                ha="center",
                color=lbl_col,
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=lbl_col, alpha=0.92, lw=0.9),
                zorder=6
            )

legend_handles_5a = [
    matplotlib.lines.Line2D([0], [0], color="#2b5c8f", lw=1.8, label="ChromBPNet Pre-MZT ($y_{abs}$)"),
    matplotlib.lines.Line2D([0], [0], color="#d73027", lw=1.6, label="NTv3 ATAC (256-cell)"),
    matplotlib.lines.Line2D([0], [0], color="#d73027", lw=1.4, ls="--", label=f"{primary_gene} TSS (0 kb)"),
    patches.Patch(facecolor="#d73027", alpha=0.35, edgecolor="none", label="Lineage-Gained Element (Cichlid-Specific)"),
    patches.Patch(facecolor="#1b7837", alpha=0.25, edgecolor="none", label="Conserved Syntenic Promoter (TSS ±2 kb)"),
    patches.Patch(facecolor="#4575b4", alpha=0.18, edgecolor="none", label="Conserved Syntenic Enhancer")
]
ax_atac.legend(handles=legend_handles_5a, loc="lower center", bbox_to_anchor=(0.5, 1.01), ncol=3, frameon=True, fontsize=8.0, framealpha=0.94)
ax_atac.set_ylabel("Predicted ATAC", fontsize=10, fontweight="bold")
ax_atac.set_title(f"A. Target Species Pre-MZT Chromatin Accessibility ({TARGET_LOCUS_NAME})", fontsize=11.5, fontweight="bold", pad=44)
ax_atac.grid(True, linestyle=":", alpha=0.4)
plt.setp(ax_atac.get_xticklabels(), visible=False)

# 2. Panel 2: Target Species Nascent SLAM-seq Transcription (NTv3 2.5 hpf)
ax_slam = fig5.add_subplot(gs5[1], sharex=ax_atac)
if not df_scan.empty:
    ax_slam.plot(nt_mids_kb, df_scan["slam_2.5hpf_fwd"], color="#1b7837", lw=1.5, label="NTv3 Nascent SLAM (+) Strand")
    ax_slam.plot(nt_mids_kb, df_scan["slam_2.5hpf_rev"], color="#7570b3", lw=1.5, ls="--", label="NTv3 Nascent SLAM (-) Strand")
ax_slam.axvline(0, color="#d73027", linestyle="--", lw=1.2, alpha=0.7)
ax_slam.text(0.012, 0.94, "Note: (+) and (-) strands co-vary (r=0.99; +0.021 offset) — sparse 2.5 hpf head tracks shared promoter openness + coding periodicity.",
             transform=ax_slam.transAxes, fontsize=7.7, va="top", color="#333333",
             bbox=dict(boxstyle="round,pad=0.18", fc="white", ec="#cccccc", alpha=0.90))
ax_slam.set_ylabel("NTv3 Nascent (log1p)", fontsize=10, fontweight="bold")
ax_slam.set_title("B. Target Species Predicted Nascent Transcription (2.5 hpf)", fontsize=11.5, fontweight="bold")
ax_slam.legend(loc="upper right", frameon=True, fontsize=8.5)
ax_slam.grid(True, linestyle=":", alpha=0.4)
plt.setp(ax_slam.get_xticklabels(), visible=False)

# 3. Panel 3: Continuous Saliency Attribution Tracks (ATAC left y-axis, SLAM right twinx y-axis)
ax_sal = fig5.add_subplot(gs5[2], sharex=ax_atac)
ax_sal_r = ax_sal.twinx()
sal_handles, sal_labels = [], []
if saliency_npz_path.exists():
    sal_dict = np.load(saliency_npz_path)
    step = 50
    if "full_locus_sal_atac_smooth" in sal_dict:
        s_arr = sal_dict["full_locus_sal_atac_smooth"]
        w_st = int(sal_dict["full_locus_st"][0])
        s_mids_kb = np.array([to_signed_kb(w_st + i) for i in range(len(s_arr))])
        m_sub = (s_mids_kb >= x_min_kb) & (s_mids_kb <= x_max_kb)
        h1, = ax_sal.plot(s_mids_kb[m_sub][::step], s_arr[m_sub][::step], color="#e7298a", lw=1.4, label="NTv3 ATAC Saliency (|∇|, left)")
        ax_sal.fill_between(s_mids_kb[m_sub][::step], s_arr[m_sub][::step], color="#e7298a", alpha=0.14)
        sal_handles.append(h1)
        sal_labels.append(h1.get_label())

    # Full-locus aligned SLAM saliency on right y-axis (twinx)
    if "full_locus_sal_slam_smooth" in sal_dict:
        sl_arr = sal_dict["full_locus_sal_slam_smooth"]
        w_st = int(sal_dict["full_locus_st"][0])
        sl_mids_kb = np.array([to_signed_kb(w_st + i) for i in range(len(sl_arr))])
        m_sub_sl = (sl_mids_kb >= x_min_kb) & (sl_mids_kb <= x_max_kb)
        h2, = ax_sal_r.plot(sl_mids_kb[m_sub_sl][::step], sl_arr[m_sub_sl][::step], color="#1b7837", lw=1.4, ls="--", label="Total Locus 2.5 hpf SLAM Saliency (|∇|, right)")
        sal_handles.append(h2)
        sal_labels.append(h2.get_label())

    # Optional target-gene-specific exon-masked SLAM saliency across full locus
    k_tg_locus_slam = f"{primary_gene}_locus_sal_slam_smooth"
    if k_tg_locus_slam not in sal_dict:
        k_tg_locus_slam = f"{primary_gene.lower()}_locus_sal_slam_smooth"
    if k_tg_locus_slam in sal_dict and "full_locus_st" in sal_dict:
        tgl_arr = sal_dict[k_tg_locus_slam]
        w_st = int(sal_dict["full_locus_st"][0])
        tgl_kb = np.array([to_signed_kb(w_st + i) for i in range(len(tgl_arr))])
        m_sub_t = (tgl_kb >= x_min_kb) & (tgl_kb <= x_max_kb)
        h3, = ax_sal_r.plot(tgl_kb[m_sub_t][::step], tgl_arr[m_sub_t][::step], color="#00441b", lw=1.2, ls=":", label=f"{primary_gene} Exon-Specific SLAM Saliency (|∇|, right)")
        sal_handles.append(h3)
        sal_labels.append(h3.get_label())

ax_sal.axvline(0, color="#d73027", linestyle="--", lw=1.2, alpha=0.7)
y0_l, y1_l = ax_sal.get_ylim()
ax_sal.set_ylim(0.0, max(1.0, y1_l * 1.18))
y0_r, y1_r = ax_sal_r.get_ylim()
ax_sal_r.set_ylim(0.0, max(1.0, y1_r * 1.25))
ax_sal.set_ylabel("ATAC Saliency (|∇|)", fontsize=10, fontweight="bold", color="#e7298a")
ax_sal.tick_params(axis="y", colors="#e7298a")
ax_sal_r.set_ylabel("SLAM Saliency (|∇|)", fontsize=10, fontweight="bold", color="#1b7837")
ax_sal_r.tick_params(axis="y", colors="#1b7837")
ax_sal.set_title(f"C. Continuous Base-Resolution Saliency Attribution across Entire Locus ({TARGET_LOCUS_NAME})", fontsize=11.5, fontweight="bold")
if sal_handles:
    ax_sal_r.legend(sal_handles, sal_labels, loc="upper right", frameon=True, fontsize=8.0, framealpha=0.92)
ax_sal.grid(True, linestyle=":", alpha=0.4)
plt.setp(ax_sal.get_xticklabels(), visible=False)

# 4. Panel 4: Human-Readable Canonical Gene Models (Width-Aware Multi-Track Layout)
ax_genes = fig5.add_subplot(gs5[3], sharex=ax_atac)
ax_genes.set_yticks([])

locus_genes = []
seen_ids = set()
for g_rec in gtf_dict["genes"].values():
    if g_rec.gene_id in seen_ids:
        continue
    seen_ids.add(g_rec.gene_id)
    if str(g_rec.chrom) == str(primary_chrom) and g_rec.end >= (pred_min_bp or 0) and g_rec.start <= (pred_max_bp or 1e9):
        locus_genes.append({
            "name": g_rec.gene_name if g_rec.gene_name else g_rec.gene_id,
            "st": g_rec.start,
            "en": g_rec.end,
            "strand": g_rec.strand,
            "is_tg": (g_rec.gene_name.lower() in [tg.lower() for tg in TARGET_GENES] or g_rec.gene_id.lower() in [tg.lower() for tg in TARGET_GENES])
        })
locus_genes.sort(key=lambda x: x["st"])

# Compute kb intervals accounting for rendered gene-name pill width
x_span_kb_5d = max(10.0, x_max_kb - x_min_kb)
tracks_end_kb = []
track_y_positions = [0.12, 0.34, 0.56, 0.78]

for g in locus_genes:
    c_st = max(g["st"], pred_min_bp if pred_min_bp else g["st"])
    c_en = min(g["en"], pred_max_bp if pred_max_bp else g["en"])
    gx_st = min(to_signed_kb(c_st), to_signed_kb(c_en))
    gx_en = max(to_signed_kb(c_st), to_signed_kb(c_en))
    arrow = "→" if g["strand"] == "+" else "←"
    lbl_text = f"{g['name']} {arrow}"
    mid_gx = 0.5 * (gx_st + gx_en)
    half_pill_kb = max(4.8, 0.58 * len(lbl_text) * (x_span_kb_5d / 140.0))
    occ_st = min(gx_st, mid_gx - half_pill_kb)
    occ_en = max(gx_en, mid_gx + half_pill_kb)
    g["gx_st"], g["gx_en"], g["mid_gx"], g["lbl"] = gx_st, gx_en, mid_gx, lbl_text

    assigned = False
    for t_idx, last_en_kb in enumerate(tracks_end_kb):
        if occ_st > last_en_kb + 1.8:
            tracks_end_kb[t_idx] = occ_en
            g["track"] = t_idx % len(track_y_positions)
            assigned = True
            break
    if not assigned:
        tracks_end_kb.append(occ_en)
        g["track"] = (len(tracks_end_kb) - 1) % len(track_y_positions)

for g in locus_genes:
    is_tg = g["is_tg"]
    g_col = "#d73027" if is_tg else "#2b5c8f"
    g_lw = 3.6 if is_tg else 2.2
    y_pos = track_y_positions[g["track"]]

    ax_genes.plot([g["gx_st"], g["gx_en"]], [y_pos, y_pos], color=g_col, lw=g_lw, solid_capstyle="round", zorder=3)
    if x_min_kb <= g["mid_gx"] <= x_max_kb:
        ax_genes.text(
            g["mid_gx"], y_pos + 0.045, g["lbl"],
            ha="center", va="bottom", fontsize=8.0,
            fontweight="bold" if is_tg else "semibold",
            color=g_col,
            bbox=dict(boxstyle="round,pad=0.16", fc="white", ec=g_col, lw=0.85, alpha=0.92),
            zorder=5
        )

ax_genes.set_ylim(0, 1.05)
ax_genes.set_xlabel(f"Signed Distance from {primary_gene} TSS (kb)", fontsize=10.5, fontweight="bold")
ax_genes.set_title(f"D. Annotated Syntenic Gene Models across '{TARGET_LOCUS_NAME}' Locus", fontsize=11.5, fontweight="bold")
ax_genes.grid(True, linestyle=":", alpha=0.4, axis="x")

# 5. Panel 5: Syntenic Zebrafish Locus Profile (Cross-Species Alignment)
ax_zf = fig5.add_subplot(gs5[4])
if not df_zf.empty:
    zf_mb = df_zf["center"] / 1e6
    ax_zf.plot(zf_mb, df_zf["atac_256cell"], color="#2b5c8f", lw=1.5, label="ZF NTv3 ATAC (256-cell)")
    zf_slam_max = np.maximum(df_zf["slam_2.5hpf_fwd"], df_zf["slam_2.5hpf_rev"])
    ax_zf.plot(zf_mb, zf_slam_max, color="#1b7837", lw=1.3, ls="--", label="ZF NTv3 SLAM (2.5 hpf)")
    zf_chr_lbl = str(df_zf["chrom"].iloc[0])
    ax_zf.set_xlabel(f"Zebrafish Chromosome {zf_chr_lbl} Coordinate (Mb)", fontsize=9.5)
else:
    ax_zf.text(0.5, 0.5, "Syntenic Zebrafish Locus Profile", ha="center", va="center")
ax_zf.set_ylabel("ZF NTv3 Signal", fontsize=10, fontweight="bold")
ax_zf.set_title("E. Syntenic Zebrafish Orthologous Locus Profile", fontsize=11.5, fontweight="bold")
ax_zf.legend(loc="upper right", frameon=True, fontsize=8.5)
ax_zf.grid(True, linestyle=":", alpha=0.4)

fig5.subplots_adjust(top=0.90, bottom=0.06, left=0.08, right=0.91, hspace=0.46)
fig5_path = res_dir / "fig5_cross_architecture_locus_overlay.png"
fig5_svg = res_dir / "fig5_cross_architecture_locus_overlay.svg"
plt.savefig(fig5_path, dpi=300)
plt.savefig(fig5_svg)
display(Image(filename=str(fig5_path), width=900))
print(f"[+] Figure 5 exported to {fig5_path} and .svg")

# FIGURE 6: Multi-Candidate Distal Coupling Matrix & Single-nt Zoom
# ------------------------------------------------------------------------------
print("[*] Generating Figure 6: Distal-to-Promoter Coupling Screen & Attribution Zoom...")
fig6 = plt.figure(figsize=(15, 6))
gs6 = gridspec.GridSpec(1, 2, width_ratios=[1.3, 1.0], wspace=0.3)

# Left: Distal Perturbation Impact on Local ATAC and Target Promoters
ax_coup = fig6.add_subplot(gs6[0])
if coupling_tsv.exists():
    df_c = pd.read_csv(coupling_tsv, sep="\t")
    pert_order = ["dinuc_shuffle_500bp", "cpg_ablation_500bp", "motif_swap"]
    pert_labels = {"dinuc_shuffle_500bp": "Dinuc Shuffle", "cpg_ablation_500bp": "CpG Ablation (CG->TG)", "motif_swap": "Motif Swap"}
    ranks_present = sorted(df_c["element_rank"].unique())
    x_pos = np.arange(len(ranks_present))
    w = 0.25

    for p_i, pert in enumerate(pert_order):
        df_p = df_c[df_c["perturbation"] == pert].set_index("element_rank")
        deltas = [df_p.loc[rk, "pct_delta_local_atac"] if rk in df_p.index else 0.0 for rk in ranks_present]
        colors = ["#3182bd", "#de2d26", "#31a354"]
        ax_coup.bar(x_pos + (p_i - 1) * w, deltas, width=w, label=pert_labels[pert], color=colors[p_i], edgecolor="black", lw=0.7)

    x_labels = []
    for rk in ranks_present:
        sub_r = df_c[df_c["element_rank"] == rk].iloc[0]
        x_labels.append(f"Rank {rk}\n({sub_r['overlapping_gene']})")

    ax_coup.set_xticks(x_pos)
    ax_coup.set_xticklabels(x_labels, fontsize=9.5)
    ax_coup.set_ylabel("% Change in Local Pre-MZT ATAC", fontsize=10.5, fontweight="bold")
    ax_coup.set_title("A. NTv3 In Silico Perturbation of Candidate Elements", fontsize=11.5, fontweight="bold")
    ax_coup.axhline(0, color="gray", linestyle="--", lw=1)
    ax_coup.legend(loc="lower left", frameon=True, fontsize=8.5)
    ax_coup.grid(True, linestyle=":", alpha=0.4, axis="y")

# Right: Single-Nucleotide Saliency Profile around Primary Promoter / Enhancer
ax_zoom = fig6.add_subplot(gs6[1])
if saliency_npz_path.exists():
    sal_dict = np.load(saliency_npz_path)
    if f"{primary_gene}_sal_atac" in sal_dict:
        raw_sal = sal_dict[f"{primary_gene}_sal_atac"]
        smooth_sal = sal_dict.get(f"{primary_gene}_sal_atac_smooth", raw_sal)
        mid_s = len(raw_sal) // 2
        zoom_st = max(0, mid_s - 150)
        zoom_en = min(len(raw_sal), mid_s + 150)
        ax_zoom.plot(np.arange(-150, 150), raw_sal[zoom_st:zoom_en], color="#bdbdbd", lw=1.0, label="Raw |grad|")
        ax_zoom.plot(np.arange(-150, 150), smooth_sal[zoom_st:zoom_en], color="#d73027", lw=1.8, label="Smoothed Saliency")
        ax_zoom.set_xlabel("Relative Offset from Center (bp)", fontsize=10)
        ax_zoom.set_ylabel("NTv3 Saliency Magnitude", fontsize=10, fontweight="bold")
        ax_zoom.set_title(f"B. Single-Nucleotide Attribution ({primary_gene} Promoter)", fontsize=11.5, fontweight="bold")
        ax_zoom.legend(loc="upper right", frameon=True, fontsize=8.5)
        ax_zoom.grid(True, linestyle=":", alpha=0.4)

fig6.subplots_adjust(left=0.08, right=0.96, top=0.92, bottom=0.12, wspace=0.28)
fig6_path = res_dir / "fig6_knockout_and_motif_saliency.png"
fig6_svg = res_dir / "fig6_knockout_and_motif_saliency.svg"
plt.savefig(fig6_path, dpi=300)
plt.savefig(fig6_svg)
display(Image(filename=str(fig6_path), width=900))
print(f"[+] Figure 6 exported to {fig6_path} and .svg")

# ------------------------------------------------------------------------------
# FIGURE 7: Unified Cross-Architecture Benchmark (Primary control_silent)
# ------------------------------------------------------------------------------
print("[*] Generating Figure 7: Unified Cross-Architecture Benchmark...")
fig7 = plt.figure(figsize=(16, 5))
gs7 = gridspec.GridSpec(1, 3, width_ratios=[1.0, 1.0, 1.2], wspace=0.3)

df_cbp_sc = pd.read_csv(chrombpnet_scored_tsv, sep="\t", low_memory=False) if chrombpnet_scored_tsv.exists() else pd.DataFrame()
df_nt_sc = pd.read_csv(ntv3_scored_tsv, sep="\t", low_memory=False) if ntv3_scored_tsv.exists() else pd.DataFrame()

# Intersect ChromBPNet & NTv3 onto identical non-NaN genes so ROC and PR curves share the exact same cohort & prevalence
nt_keep_cols = [
    "gene_id",
    "ntv3_pred_atac_promoter_1kb_max",
    "ntv3_pred_atac_promoter_intron1_capped_max",
    "ntv3_pred_slam_2.5hpf_3p",
    "ntv3_pred_slam_2.5hpf_exon_mean_per_kb",
    "slam_3p_tes_centered",
]
avail_nt_k = [c for c in nt_keep_cols if c in df_nt_sc.columns]
if not df_cbp_sc.empty and not df_nt_sc.empty:
    df_common = pd.merge(df_cbp_sc, df_nt_sc[avail_nt_k], on="gene_id", how="inner")
    # Exclude stale un-TES-centered long-gene 0.0 SLAM artifacts when evaluating across shared finite cohort
    if "ntv3_pred_slam_2.5hpf_3p" in df_common.columns and "slam_3p_tes_centered" in df_common.columns:
        span_c = (df_common["end"] - df_common["start"]).abs() if ("start" in df_common.columns and "end" in df_common.columns) else pd.Series(0, index=df_common.index)
        bad_sl = (df_common["ntv3_pred_slam_2.5hpf_3p"] == 0.0) & (~df_common["slam_3p_tes_centered"].fillna(False).astype(bool)) & (span_c > 10500)
        med_p = float(df_common.loc[df_common["ntv3_pred_slam_2.5hpf_3p"] > 1.0, "ntv3_pred_slam_2.5hpf_3p"].median()) if (df_common["ntv3_pred_slam_2.5hpf_3p"] > 1.0).any() else 0.0
        if med_p > 10.0:
            bad_sl = bad_sl | ((df_common["ntv3_pred_slam_2.5hpf_3p"] > 0.0) & (df_common["ntv3_pred_slam_2.5hpf_3p"] < 1.0))
        df_common.loc[bad_sl, "ntv3_pred_slam_2.5hpf_3p"] = np.nan

    req_eval_cols = [
        "pred_promoter_1kb_y_abs",
        "pred_promoter_intron1_capped_y_abs",
        "ntv3_pred_atac_promoter_1kb_max",
        "ntv3_pred_atac_promoter_intron1_capped_max",
        "ntv3_pred_slam_2.5hpf_3p",
        "ntv3_pred_slam_2.5hpf_exon_mean_per_kb",
    ]
    req_eval_cols = [c for c in req_eval_cols if c in df_common.columns]
    chrom_c = df_common["chrom"].astype(str).str.strip().str.lower()
    valid_c = ~chrom_c.isin({"mt", "chrm", "m", "chrmt", "unknown", "nan", ""})
    if "start" in df_common.columns and "end" in df_common.columns:
        valid_c = valid_c & ~((df_common["start"].fillna(0) == 0) & (df_common["end"].fillna(0) == 0))
    df_common = df_common.loc[valid_c].dropna(subset=req_eval_cols).copy()
    is_mw_comm = (df_common.get("is_strictly_zygotic_minor_wave", pd.Series(False, index=df_common.index)) == True) | (df_common["category"].isin(["strictly_zygotic_minor_wave", "strictly_zygotic_highconf"]))
    is_sil_comm = (df_common.get("is_control_silent", pd.Series(False, index=df_common.index)) == True) | (df_common["category"] == "control_silent")
    df_common_eval = df_common[is_mw_comm | is_sil_comm].copy()
    df_common_eval["_y_true"] = is_mw_comm.loc[df_common_eval.index].astype(int).values
    if "ntv3_pred_atac_promoter_1kb_max" in df_common_eval.columns and "ntv3_pred_atac_promoter_intron1_capped_max" in df_common_eval.columns:
        eq_f = float((df_common_eval["ntv3_pred_atac_promoter_1kb_max"] == df_common_eval["ntv3_pred_atac_promoter_intron1_capped_max"]).mean())
        if eq_f > 0.98:
            print(f"[!] Notice: Cached ntv3_target_genes_scored.tsv has identical 1kb & intron1 values ({eq_f:.1%}); run `python scripts/ntv3_infer.py --subtask score --force-rescore` to separate NTv3 ATAC windows.")
else:
    df_common_eval = pd.DataFrame()

n_pos_eval = int((df_common_eval["_y_true"] == 1).sum()) if not df_common_eval.empty else 0
n_ctrl_eval = int((df_common_eval["_y_true"] == 0).sum()) if not df_common_eval.empty else 0

models_to_eval = [
    ("ChromBPNet (Promoter ±1 kb)", df_common_eval, "pred_promoter_1kb_y_abs", "#2b5c8f", "-", 2.2),
    ("ChromBPNet (Promoter + Intron 1)", df_common_eval, "pred_promoter_intron1_capped_y_abs", "#6baed6", "--", 1.8),
    ("NTv3 ATAC (Promoter ±1 kb)", df_common_eval, "ntv3_pred_atac_promoter_1kb_max", "#d94801", "-", 2.4),
    ("NTv3 ATAC (Promoter + Intron 1)", df_common_eval, "ntv3_pred_atac_promoter_intron1_capped_max", "#fdae6b", ":", 1.8),
    ("NTv3 SLAM (2.5 hpf TES 3'-End)", df_common_eval, "ntv3_pred_slam_2.5hpf_3p", "#1b7837", "-", 2.0),
    ("NTv3 SLAM (2.5 hpf Exon Mean/kb)", df_common_eval, "ntv3_pred_slam_2.5hpf_exon_mean_per_kb", "#74c476", "--", 1.8),
]

# ROC Curves on Matched Shared Cohort
ax_roc = fig7.add_subplot(gs7[0])
for m_label, df_m, sc_col, m_col, m_ls, m_lw in models_to_eval:
    if not df_m.empty and sc_col in df_m.columns:
        y_t = df_m["_y_true"].values
        y_s = df_m[sc_col].values
        fpr, tpr, _ = roc_curve(y_t, y_s)
        roc_auc = auc(fpr, tpr)
        ax_roc.plot(fpr, tpr, label=f"{m_label} (AUC={roc_auc:.3f})", color=m_col, lw=m_lw, ls=m_ls, alpha=0.92)

ax_roc.plot([0, 1], [0, 1], "k--", alpha=0.4)
ax_roc.set_xlabel("False Positive Rate", fontsize=10)
ax_roc.set_ylabel("True Positive Rate", fontsize=10)
ax_roc.set_title(f"A. ROC: Minor-Wave vs. Silent\n(Matched Cohort: n={n_pos_eval} vs {n_ctrl_eval})", fontsize=10.5, fontweight="bold")
ax_roc.legend(loc="lower right", frameon=True, fontsize=7.8)
ax_roc.grid(True, linestyle=":", alpha=0.4)

# PR Curves on Identical Matched Cohort (same baseline prevalence across all models)
ax_pr = fig7.add_subplot(gs7[1])
if n_pos_eval + n_ctrl_eval > 0:
    base_prev = n_pos_eval / float(n_pos_eval + n_ctrl_eval)
    ax_pr.axhline(base_prev, color="#636363", linestyle="--", lw=1.2, alpha=0.75, label=f"Chance Baseline ({base_prev:.3f})")

for m_label, df_m, sc_col, m_col, m_ls, m_lw in models_to_eval:
    if not df_m.empty and sc_col in df_m.columns:
        y_t = df_m["_y_true"].values
        y_s = df_m[sc_col].values
        prec, rec, _ = precision_recall_curve(y_t, y_s)
        pr_auc = auc(rec, prec)
        ax_pr.plot(rec, prec, label=f"{m_label} (AUPRC={pr_auc:.3f})", color=m_col, lw=m_lw, ls=m_ls, alpha=0.92)

ax_pr.set_xlim(0.0, 1.0)
ax_pr.set_ylim(0.0, 1.02)
ax_pr.set_xlabel("Recall", fontsize=10)
ax_pr.set_ylabel("Precision", fontsize=10)
ax_pr.set_title(f"B. Precision-Recall: Matched Cohort\n(n={n_pos_eval} MW vs {n_ctrl_eval} Silent)", fontsize=10.5, fontweight="bold")
ax_pr.legend(loc="upper right", frameon=True, fontsize=7.6)
ax_pr.grid(True, linestyle=":", alpha=0.4)

# Model Comparison Bar Chart from cross_architecture_summary.tsv
ax_bar = fig7.add_subplot(gs7[2])
if cross_summary_path.exists():
    df_cs = pd.read_csv(cross_summary_path, sep="\t")
    eval_models = df_cs.copy()
    name_map = {
        "ChromBPNet_grafted": "ChromBPNet\ngrafted (2.1kb)",
        "NTv3_raw_pretrained": "NTv3_raw\n(no blastula head)",
        "NTv3_frozen_head": "NTv3_frozen\n(trained head)",
        "NTv3_multitask_lora": "NTv3_multitask\n(LoRA + head)"
    }
    m_names = [name_map.get(m, m.replace("_", "\n")) for m in eval_models["model"]]
    x_pos = np.arange(len(m_names))
    w = 0.25

    atac_aucs = pd.to_numeric(eval_models["atac_auroc_vs_silent"], errors="coerce").fillna(0).values
    zf_spear = pd.to_numeric(eval_models["atac_256cell_spearman_r"], errors="coerce").fillna(0).values
    slam_aucs = pd.to_numeric(eval_models["slam_auroc_vs_silent"], errors="coerce").fillna(0).values

    b1 = ax_bar.bar(x_pos - w, atac_aucs, width=w, label="Target ATAC AUROC (vs Silent)", color="#d73027", edgecolor="black", lw=0.7)
    b2 = ax_bar.bar(x_pos, zf_spear, width=w, label="ZF Test Spearman r", color="#2b5c8f", edgecolor="black", lw=0.7)
    b3 = ax_bar.bar(x_pos + w, slam_aucs, width=w, label="Target SLAM AUROC", color="#1b7837", edgecolor="black", lw=0.7)

    for grp_bars, raw_col in [(b1, "atac_auroc_vs_silent"), (b2, "atac_256cell_spearman_r"), (b3, "slam_auroc_vs_silent")]:
        for idx_b, (bar, raw_v) in enumerate(zip(grp_bars, eval_models[raw_col])):
            num_v = pd.to_numeric(raw_v, errors="coerce")
            if pd.notna(num_v):
                ax_bar.text(bar.get_x() + bar.get_width() / 2.0, float(num_v) + 0.015, f"{float(num_v):.2f}", ha="center", va="bottom", fontsize=7.0, fontweight="bold")
            elif idx_b == 0 and raw_col == "slam_auroc_vs_silent":
                ax_bar.text(bar.get_x() + bar.get_width() / 2.0, 0.04, "ATAC\nonly", ha="center", va="bottom", fontsize=6.5, color="#525252")

    ax_bar.axhline(0.50, color="gray", linestyle="--", lw=0.9, alpha=0.55)
    ax_bar.set_ylim(0.0, 1.18)
    ax_bar.set_xticks(x_pos)
    ax_bar.set_xticklabels(m_names, fontsize=7.8)
    ax_bar.set_ylabel("Metric Score", fontsize=10)
    ax_bar.set_title("C. Training Progression & Cross-Architecture Summary", fontsize=10.5, fontweight="bold")
    ax_bar.legend(loc="upper left", frameon=True, fontsize=7.6)
    ax_bar.grid(True, linestyle=":", alpha=0.4, axis="y")

fig7.subplots_adjust(left=0.06, right=0.96, top=0.90, bottom=0.14, wspace=0.30)
fig7_path = res_dir / "fig7_cross_architecture_benchmark.png"
fig7_svg = res_dir / "fig7_cross_architecture_benchmark.svg"
plt.savefig(fig7_path, dpi=300)
plt.savefig(fig7_svg)
display(Image(filename=str(fig7_path), width=900))
print(f"[+] Figure 7 exported to {fig7_path} and .svg")

# Display final summary table
if cross_summary_path.exists():
    df_final_summary = pd.read_csv(cross_summary_path, sep="\t")
    print(f"\n[+] Final Publication Cross-Architecture Summary Table ({cross_summary_path}):")
    display(df_final_summary)


---
## Block 8c: Target Gene Statistical Ranking & Minor-Wave Parity Test

Executes leave-one-out statistical testing (`scripts/target_gene_benchmark.py`, excluding mitochondrial `MT`/`chrM` and unplaced zero-span contigs) comparing every gene in `TARGET_GENES` and its reference orthologs against genome-wide controls (`control_silent`, `control_non_activated` maternal housekeeping controls, GC/span-matched silent controls) and *bona fide* minor-wave cohorts (`strictly_zygotic_minor_wave` and SLAM-confirmed `high_conf_minor_wave`):
* Reports empirical right-tail $p$-values, Benjamini–Hochberg FDR $q$-values, median/MAD robust $z$-scores, minor-wave percentile ranks, and parity tiers (`Above Q75`, `Within IQR Q25–Q75`, `Q10–Q25`, `Below Q10`).

**Generated Outputs (Block 8c):**
* **Figure 8 (`fig8_target_gene_parity_benchmark.png` & `.svg`):** Four-panel statistical parity benchmark overlaying target genes and canonical ZGA controls onto cohort violin/box distributions: (`A`) ChromBPNet core promoter $y_{\text{abs}}$, (`B`) NTv3 promoter + intron 1 ATAC (noting why constitutive maternal housekeeping genes mirror minor-wave openness), (`C`) NTv3 nascent SLAM 3'-end TES signal (`log(1 + Signal/kb)`, where smooth `F.softplus` + sparse track up-weighting lift the baseline floor above zero), and (`D`) **Silent-Calibrated Parity Curves** plotting `Control-Silent Percentile` (`x`, `0–100%`) vs. `Minor-Wave Percentile` (`y`, `0–100%`, shaded `Q25–Q75` IQR & `>Q75` top quartile; dashed diagonal `y = x` marks no enrichment `AUROC = 0.50`, so **curves bowing farther down and right beneath `y = x`** indicate stronger enrichment over silent background) with target genes plotted directly on all three modality curves.
* **Ranking Statistics Table (`{TARGET_LOCUS_NAME}_target_gene_ranking_stats.tsv`):** Complete metric × target gene table with empirical $p$-values, FDR $q$-values, cohort percentiles, MAD $z$-scores, and orthologous reference counter-scores.

In [ ]:
# ==============================================================================
# Block 8c: Target Gene Statistical Ranking & Minor-Wave Parity Test
# ==============================================================================
from pathlib import Path
import pandas as pd
from IPython.display import Image, display
from scripts.target_gene_benchmark import run_target_gene_benchmark

print("[*] Executing Block 8c Target Gene Statistical Enrichment & Minor-Wave Parity Benchmark...")

benchmark_results = run_target_gene_benchmark(
    work_dir=WORK_DIR,
    locus_name=TARGET_LOCUS_NAME,
    target_genes=TARGET_GENES,
    canonical_markers=CANONICAL_ZGA_MARKERS,
    target_species_name=TARGET_SPECIES_NAME,
    target_genome_fa=TARGET_GENOME_FA
)

df_benchmark_stats = benchmark_results["df_stats"]
df_benchmark_summary = benchmark_results["df_summary"]

print("\n[+] Executive Summary Table: Target Gene Statistical Ranking & Minor-Wave Parity:")
display(df_benchmark_summary)

print("\n[+] Displaying Publication Figure 8 (Target Gene Parity Benchmark - 300 DPI):")
display(Image(filename=benchmark_results["fig8_png"], width=950))


---
## Block 8d: Unified Inference API (`infer_premzt` & `infer_premzt_batch`)

Demonstrates the cached, GPU-batched (`bfloat16` + unique-window deduplication) inference API (`scripts/premzt_inference.py`):
* Supports `"chrombpnet"`, `"ntv3atac"`, `"ntv3slam"`, or `"all"` across configured species names/aliases.
* Supports length-matched `"auto"` presets (`promoter_1kb`, `promoter_intron1_capped`, `proximal_5kb`, `bounded_25kb`, `tes_1kb`) alongside custom strand-aware locus DSL queries (`parse_locus_range`).

In [ ]:
# ==============================================================================
# Block 8d: Unified Pre-MZT Inference API Usage Demo
# ==============================================================================
import pandas as pd
from scripts.premzt_inference import infer_premzt

print("[*] 1. Cross-Species Benchmark: Target Species vs. Zebrafish Ortholog Counter-Scoring ('auto')")
df_orthologs = infer_premzt(
    species=TARGET_SPECIES_NAME,
    gene=TARGET_GENES,
    model="all",
    locus_range="auto",
    work_dir=WORK_DIR
)
df_zf_orthologs = infer_premzt(
    species="Danio_rerio",
    gene=TARGET_GENES,
    model="all",
    locus_range="auto",
    work_dir=WORK_DIR
)
if isinstance(df_orthologs, pd.DataFrame) and isinstance(df_zf_orthologs, pd.DataFrame):
    display(pd.concat([df_orthologs, df_zf_orthologs], ignore_index=True))


---
### Block 8d Example: Flexible Locus DSL & Raw DNA Sequence Scoring
Queries composite feature intervals (e.g., `TSS+200-100,intron1`, `TSS+200,intron1,5utr,3utr`) and direct FASTA/synthetic DNA sequences bypassing GTF annotations (`sequence=...`).

In [ ]:
print("\n[*] 2. Flexible Locus Range DSL Syntax Demonstration:")
dsl_examples = [
    ("TSS+-200", "Symmetric ±200 bp around TSS"),
    ("TSS+200-100,intron1", "TSS upstream 200 to downstream 100 bp unioned with canonical intron 1"),
    ("+2000-1000", "Gene body extended 2 kb upstream of TSS to 1 kb downstream of TES"),
    ("TSS+200,intron1,5utr,3utr", "Multi-feature composite: promoter + intron 1 + 5' UTR + 3' UTR")
]
for dsl_str, desc in dsl_examples:
    res_dsl = infer_premzt(
        species=TARGET_SPECIES_NAME,
        gene=TARGET_GENES[0],
        model="chrombpnet",
        locus_range=dsl_str,
        work_dir=WORK_DIR
    )
    print(f"  -> DSL '{dsl_str}' ({desc}):")
    print(f"     Resolved Intervals: {res_dsl.get('resolved_intervals')}")
    if "chrombpnet" in res_dsl:
        print(f"     ChromBPNet y_abs={res_dsl['chrombpnet'].get('pred_y_abs', 0.0):.3f}, y_rel={res_dsl['chrombpnet'].get('pred_y_rel', 0.0):.3f}")

print("\n[*] 3. Direct Raw Sequence Mode (bypasses genome annotation):")
sample_seq = "ACGT" * 500
res_seq = infer_premzt(
    model="all",
    sequence=sample_seq,
    work_dir=WORK_DIR
)
print("  -> Direct sequence scoring result:")
for m_k, m_v in res_seq.items():
    print(f"     {m_k}: {m_v}")